# Objective

The goal is to detect cells in 3D microscopy images, associate detections across time, and reconstruct cell lineages, including division events.

# Setup

In [1]:
%pip install -q zarr

ERROR: Could not find a version that satisfies the requirement zarr (from versions: none)
ERROR: No matching distribution found for zarr
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import zarr

sns.set_theme(style="whitegrid")

ModuleNotFoundError: No module named 'zarr'

# Data Structure 

## Inspect the File Structure

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
COMPETITION_NAME = "biohub-cell-tracking-during-development"
DATA_ROOT = INPUT_ROOT / "competitions" / COMPETITION_NAME

print("Data root exists:", DATA_ROOT.exists())
print("Data root:", DATA_ROOT)

if DATA_ROOT.exists():
    for path in sorted(DATA_ROOT.iterdir()):
        print(path.name)
else:
    print("\nAvailable input directories:")
    for path in sorted(INPUT_ROOT.iterdir()):
        print(path)

The competition data contains three main components:

- `train/`: training microscopy movies and their sparse lineage annotations
- `test/`: the four microscopy movies for which predictions are required
- `sample_submission.csv`: an example of the required node-and-edge submission format

Unlike a typical dataset where each row is one observation, each sample here is a sequence of 3D microscopy images over time. Instead of predicting a single value, the goal is to detect cells and build a graph that connects them across different time points.

In [ ]:
train_root = DATA_ROOT / "train"
test_root = DATA_ROOT / "test"

train_entries = sorted(train_root.iterdir())
test_entries = sorted(test_root.iterdir())

print("Number of train entries:", len(train_entries))
print("Number of test entries:", len(test_entries))

print("\nFirst 10 train entries:")
for path in train_entries[:10]:
    print(path.name)

print("\nFirst 10 test entries:")
for path in test_entries[:10]:
    print(path.name)

## Build Dataset Inventory 

In [ ]:
train_zarr = sorted(train_root.glob("*.zarr"))
train_geff = sorted(train_root.glob("*.geff"))
test_zarr = sorted(test_root.glob("*.zarr"))

train_zarr_ids = {path.stem for path in train_zarr}
train_geff_ids = {path.stem for path in train_geff}
test_ids = {path.stem for path in test_zarr}

print("Training Zarr movies:", len(train_zarr_ids))
print("Training GEFF graphs:", len(train_geff_ids))
print("Test Zarr movies:", len(test_ids))

print("\nZarr movies without matching GEFF:")
print(sorted(train_zarr_ids - train_geff_ids))

print("\nGEFF graphs without matching Zarr:")
print(sorted(train_geff_ids - train_zarr_ids))

overlapping_ids = train_zarr_ids & test_ids

print("\nTrain/test filename overlap:")
print(sorted(overlapping_ids))

### Number of movies and annotation files

The training directory contains 398 entries, consisting of:

- 199 Zarr microscopy movies
- 199 corresponding GEFF lineage graphs

Every training movie has a matching annotation graph, and every annotation Each Zarr movie has one matching Zarr movie, so there are really 199 training samples, not 398 separate ones.

The test folder is much smaller and only contains 4 Zarr microscopy movies, which are the ones we'll be making predictions for.

In [ ]:
def get_group(sample_id):
    return sample_id.split("_")[0]

inventory = pd.DataFrame(
    [
        {
            "sample_id": sample_id,
            "group": get_group(sample_id),
            "split": "train"
        }
        for sample_id in sorted(train_zarr_ids)
    ]
    +
    [
        {
            "sample_id": sample_id,
            "group": get_group(sample_id),
            "split": "test"
        }
        for sample_id in sorted(test_ids)
    ]
)

group_counts = (
    inventory
    .groupby(["split", "group"])
    .size()
    .rename("n_movies")
    .reset_index()
)

group_counts

### Train–test identifier overlap

I noticed that all four test movie IDs also appear in the training folder, and each one has a matching GEFF annotation file.

This seems intentional. I guess that the training files only include sparse annotations, while the test set is evaluated using additional hidden annotations from those same movies. That said, just because the filenames match doesn't necessarily mean the image data is the same, so I'll compare the actual image slices later to check.

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 4))

ax = sns.barplot(
    data=group_counts,
    x="group",
    y="n_movies",
    hue="split"
)

ax.set(
    title="Number of movies by group and split",
    xlabel="Filename prefix",
    ylabel="Number of movies"
)

plt.tight_layout()
plt.show()

### Apparent dataset groups

I noticed that every movie ID starts with either 44b6 or 6bba.

- 44b6: 71 training movies and 2 test movies
- 6bba: 128 training movies and 2 test movies

For now, I'm treating these prefixes as two separate dataset groups since I don't know what they represent biologically or experimentally.

The groups also aren't evenly distributed. About 64.3% of the training movies belong to the 6bba group, while the rest are in 44b6. If these groups have different image characteristics, cell density, movement patterns, or annotation coverage, that could affect model performance.

Because of that, it would probably make more sense to evaluate performance within each group instead of only looking at one overall score. 

## Inspect Submission Structure

In [ ]:
sample_submission = pd.read_csv(
    DATA_ROOT / "sample_submission.csv"
)

print("Shape:", sample_submission.shape)
print("\nColumns:")
print(sample_submission.columns.tolist())

print("\nFirst 10 rows:")
display(sample_submission.head(10))

print("\nDataset values:")
print(sample_submission["dataset"].value_counts(dropna=False))

print("\nRow types:")
print(sample_submission["row_type"].value_counts(dropna=False))

In [ ]:
display(
    sample_submission.groupby(["dataset", "row_type"])
    .size()
    .rename("n_rows")
    .reset_index()
)

### Submission structure

The sample submission contains two row types:

- `node`: a predicted cell detection with a node ID, timepoint, and 3D voxel coordinates
- `edge`: a directed link from a source node to a target node

The sample submission includes three nodes and two edges for each test movie. These aren't the actual predictions; they're just example values that show what the final submission should look like.

When submitting predictions, all four test movies need to be included. The node IDs only have to be unique within each movie, while the id column should be numbered consecutively across the entire submission.

# Inspect One Zarr Movie

## Array Structure and Physical Scale

In [ ]:
sample_id = "44b6_0113de3b"

train_zarr_path = train_root / f"{sample_id}.zarr"
test_zarr_path = test_root / f"{sample_id}.zarr"
geff_path = train_root / f"{sample_id}.geff"

print("Zarr version:", zarr.__version__)
print("Training path:", train_zarr_path)
print("Test path:", test_zarr_path)
print("GEFF path:", geff_path)

In [ ]:
train_store = zarr.open(str(train_zarr_path), mode="r")

print("Object type:", type(train_store))
print(train_store)

In [ ]:
if hasattr(train_store, "tree"):
    print(train_store.tree())
else:
    print("This object has no tree method.")

In [ ]:
if hasattr(train_store, "attrs"):
    print("Root attributes:")
    
    for key, value in train_store.attrs.items():
        print(f"{key}: {value}")

### Structure of an example microscopy movie

The movie `44b6_0113de3b` contains one 4D image array with a shape of `(100, 64, 256, 256)`. Based on the metadata, the dimensions represent time (T), depth (Z), height (Y), and width (X).

This means there are 100 timepoints, and each timepoint has 64 depth slices. Each slice is 256 × 256 pixels. The image intensities are stored as 16-bit unsigned integers.

The physical spacing is 1 second between timepoints, 1.625 µm per voxel in Z, and 0.40625 µm per voxel in X and Y. This gives a field of view of about 104 × 104 × 104 µm. Even though the overall field of view is roughly cubic, the voxels themselves aren't. A step in Z is about 4 times larger than a step in X or Y, so I need to use the physical spacing for distance calculations instead of just relying on the voxel coordinates.

The full image array contains around 419 million intensity values and would take about 800 MiB if I loaded everything at once. Since the data is stored in Zarr, I can load only the frames or slices I actually need instead of loading the entire movie.

Looking at the intensity values, most of the voxels have relatively low brightness, while only a small number are much brighter. I still need to inspect the images before assuming those bright regions are actually cells.

## Compare Train and Test Versions

In [ ]:
test_store = zarr.open(str(test_zarr_path), mode="r")

print("Training store:")
print(train_store)

print("\nTest store:")
print(test_store)

print("\nTraining hierarchy:")
if hasattr(train_store, "tree"):
    print(train_store.tree())

print("\nTest hierarchy:")
if hasattr(test_store, "tree"):
    print(test_store.tree())

In [ ]:
train_attrs = dict(train_store.attrs)
test_attrs = dict(test_store.attrs)

print("Same root attributes:", train_attrs == test_attrs)

print("\nTraining attributes:")
print(train_attrs)

print("\nTest attributes:")
print(test_attrs)

### Comparison of the train and test versions

The train and test versions of `44b6_0113de3b` have the same array structure, shape, data type, physical scale, and intensity statistics.

Based on this, they most likely represent the same microscopy movie. It doesn't look like the competition is using completely different movies for the test set. Instead, it seems like four movies from the training collection are also being used for testing.

However, matching metadata doesn't necessarily mean that every pixel is identical. I'll compare a few image slices side by side next to confirm whether the actual image data matches.

# Inspect the Sparse GEFF Annotations

##  GEFF Storage Hierarchy

In [ ]:
geff_store = zarr.open(str(geff_path), mode="r")

print("GEFF object type:", type(geff_store))
print(geff_store)

if hasattr(geff_store, "tree"):
    print("\nGEFF hierarchy:")
    print(geff_store.tree())

print("\nGEFF root attributes:")
for key, value in geff_store.attrs.items():
    print(f"{key}: {value}")

### Sparse GEFF lineage annotations

The GEFF file for `44b6_0113de3b` contains 52 annotated nodes and 50 directed edges. Each node has a timepoint and its `(z, y, x)` coordinates, while each edge connects one node to another to show how a cell is linked over time.

The annotations only cover timepoints 0–75, even though the actual movie has 100 timepoints, from 0–99. The coordinates are just showing where the annotated cells are located, so they shouldn't be confused with the dimensions of the full image.

The metadata estimates that the full movie contains around 25,755 cells, but only 52 of them are actually annotated. That's only about 0.2% of the cells, so the training data is extremely sparse.

Because of this, I can't assume that an unlabeled cell is a negative example. The annotations are only showing a small subset of the actual cells and their lineage connections, rather than labeling every cell in the movie.


## Supplied and Derived Graph Information

Each training GEFF file has a sparse annotation graph made up of a node table and a directed edge list.

Each node contains:

* a unique node ID
* a timepoint
* `(z, y, x)` centroid coordinates

Each edge connects a source node to a target node, showing how a cell is connected over time.

The GEFF files only provide basic node and edge information. They don't directly include things like node degrees, roots, leaves, connected components, or whether the graph is acyclic. I calculate those separately after loading the data into a NetworkX directed graph.

Because the annotations are sparse, the NetworkX graph represents only the annotated cells and their connections. It doesn't represent the complete cell lineage across the entire microscopy movie.


## Extract Nodes and Edges

In [ ]:
node_ids = geff_store["nodes/ids"][:]

nodes = pd.DataFrame({
    "node_id": node_ids,
    "t": geff_store["nodes/props/t/values"][:],
    "z": geff_store["nodes/props/z/values"][:],
    "y": geff_store["nodes/props/y/values"][:],
    "x": geff_store["nodes/props/x/values"][:]
})

edge_ids = geff_store["edges/ids"][:]

edges = pd.DataFrame({
    "source_id": edge_ids[:, 0],
    "target_id": edge_ids[:, 1]
})

print("Nodes shape:", nodes.shape)
display(nodes.head(10))

print("Edges shape:", edges.shape)
display(edges.head(10))

## Graph Integrity Checks

In [ ]:
print("Duplicate node IDs:", nodes["node_id"].duplicated().sum())
print("Missing node values:", nodes.isna().sum().sum())
print("Duplicate edges:", edges.duplicated().sum())

known_node_ids = set(nodes["node_id"])

unknown_sources = set(edges["source_id"]) - known_node_ids
unknown_targets = set(edges["target_id"]) - known_node_ids

print("Unknown source IDs:", unknown_sources)
print("Unknown target IDs:", unknown_targets)

### Graph extraction and integrity checks

I converted the GEFF graph for `44b6_0113de3b` into two tables:

* a node table with the node ID, timepoint, and `(z, y, x)` coordinates
* an edge table with the source and target node IDs

Before doing any analysis, I checked that the data was clean. The node IDs were all unique, there were no missing values, there weren't any duplicate edges, and every edge pointed to a valid node.

The large node ID numbers are just identifiers. They don't have any biological meaning. The actual information about the cell, like its timepoint and location, is stored separately in the node attributes.


# Explore the Annotation Graph

## Graph-Theory Definitions

These are the main graph-theory terms I'll be using throughout the graph EDA:

| **Term**                     | **Meaning in this notebook**                                      |
| ---------------------------- | ----------------------------------------------------------------- |
| Node or vertex               | An annotated cell centroid at one timepoint                       |
| Directed edge                | A connection from a source node to a later target node            |
| In-degree                    | The number of edges entering a node                               |
| Out-degree                   | The number of edges leaving a node                                |
| Root                         | A node with an in-degree of 0                                     |
| Leaf                         | A node with an out-degree of 0                                    |
| Division node                | A node with an out-degree of 2 or more                            |
| Merge node                   | A node with an in-degree of 2 or more                             |
| Directed path                | A sequence of nodes connected by directed edges                   |
| Weak component               | A group of nodes connected when edge direction is ignored         |
| Directed cycle               | A path that eventually returns to its starting node               |
| Directed acyclic graph (DAG) | A directed graph with no directed cycles                          |
| Arborescence                 | A rooted directed tree where each node has one path from the root |
| Directed forest              | A collection of disconnected directed trees                       |

Since the annotations are sparse, the root and leaf nodes only represent the beginning and end of the **annotated** part of a lineage. They don't necessarily mean the actual birth or death of a cell because a lot of cells in the movie aren't labeled.


## Connected Components and Acyclicity

In [ ]:
G = nx.DiGraph()

for row in nodes.itertuples(index=False):
    G.add_node(
        int(row.node_id),
        t=int(row.t),
        z=int(row.z),
        y=int(row.y),
        x=int(row.x)
    )

G.add_edges_from(
    (int(row.source_id), int(row.target_id))
    for row in edges.itertuples(index=False)
)

print("Number of vertices:", G.number_of_nodes())
print("Number of directed edges:", G.number_of_edges())
print("Directed acyclic graph:", nx.is_directed_acyclic_graph(G))
print("Number of self-loops:", nx.number_of_selfloops(G))

weak_components = list(nx.weakly_connected_components(G))
strong_components = list(nx.strongly_connected_components(G))

print("Weakly connected components:", len(weak_components))
print("Strongly connected components:", len(strong_components))

print(
    "Weak component sizes:",
    sorted([len(component) for component in weak_components], reverse=True)
)

## Node Degrees and Graph Roles

In [ ]:
degree_summary = pd.DataFrame({
    "node_id": list(G.nodes),
    "in_degree": [G.in_degree(node) for node in G.nodes],
    "out_degree": [G.out_degree(node) for node in G.nodes]
})

degree_summary = degree_summary.merge(
    nodes,
    on="node_id",
    how="left"
)

print("In-degree counts:")
display(
    degree_summary["in_degree"]
    .value_counts()
    .sort_index()
    .rename_axis("in_degree")
    .reset_index(name="n_nodes")
)

print("Out-degree counts:")
display(
    degree_summary["out_degree"]
    .value_counts()
    .sort_index()
    .rename_axis("out_degree")
    .reset_index(name="n_nodes")
)

In [ ]:
root_nodes = degree_summary[
    degree_summary["in_degree"] == 0
]

leaf_nodes = degree_summary[
    degree_summary["out_degree"] == 0
]

division_nodes = degree_summary[
    degree_summary["out_degree"] >= 2
]

merge_nodes = degree_summary[
    degree_summary["in_degree"] >= 2
]

print("Root nodes:", len(root_nodes))
print("Leaf nodes:", len(leaf_nodes))
print("Division nodes:", len(division_nodes))
print("Merge nodes:", len(merge_nodes))

print("\nDivision-node details:")
display(division_nodes.sort_values("t"))

print("\nMerge-node details:")
display(merge_nodes.sort_values("t"))

### Graph topology for `44b6_0113de3b`

The annotation graph is a directed acyclic graph (DAG) with two weakly connected components, one with 49 nodes and the other with 3 nodes. Every node except the roots has one incoming edge, and every node except the leaves has one outgoing edge.

Because of this, the graph is basically two separate directed paths:

$$P_{49} \sqcup P_3$$

The expected number of edges is:

$$(49-1)+(3-1)=50,$$

This matches the 50 edges that were actually observed, so the graph structure is consistent.

I also didn't find any annotated cell divisions, merges, self-loops, or cycles in this graph. But this only applies to the cells that were actually annotated. It doesn't mean that cell divisions or other events don't happen in the full microscopy movie.

Since only a small part of the movie is annotated, I should treat the root and leaf nodes as the start and end of the observed trajectories, not as the actual birth or death of a cell.


# Explore Movement Along the Annotated Trajectories

## Temporal Gaps

For each directed edge, I calculated the difference between the target and source timepoints:

$$ \Delta t = t_{\text{target}} - t_{\text{source}} $$

I used this to check whether the edges are actually moving forward in time and whether the annotations connect consecutive frames or skip over any timepoints. This is also important for calculating cell movement speed, since I need to know how much time passes between connected points.


In [ ]:
source_nodes = nodes.rename(columns={
    "node_id": "source_id",
    "t": "source_t",
    "z": "source_z",
    "y": "source_y",
    "x": "source_x"
})

target_nodes = nodes.rename(columns={
    "node_id": "target_id",
    "t": "target_t",
    "z": "target_z",
    "y": "target_y",
    "x": "target_x"
})

edge_data = (
    edges
    .merge(source_nodes, on="source_id", how="left")
    .merge(target_nodes, on="target_id", how="left")
)

edge_data["delta_t"] = (
    edge_data["target_t"] - edge_data["source_t"]
)

edge_data["delta_z_voxels"] = (
    edge_data["target_z"] - edge_data["source_z"]
)

edge_data["delta_y_voxels"] = (
    edge_data["target_y"] - edge_data["source_y"]
)

edge_data["delta_x_voxels"] = (
    edge_data["target_x"] - edge_data["source_x"]
)

display(edge_data.head())

In [ ]:
print("Edges moving forward in time:",
      (edge_data["delta_t"] > 0).all())

print("\nTime-gap distribution:")
display(
    edge_data["delta_t"]
    .value_counts()
    .sort_index()
    .rename_axis("delta_t")
    .reset_index(name="n_edges")
)

All 50 edges move forward in time, and every edge has a $\Delta t = 1$. This means each annotated connection links two consecutive frames, so there aren't any skipped timepoints in this graph.

This result only applies to `44b6_0113de3b`. I'll check the rest of the dataset later to see if any of the other annotation graphs have skipped frames.


## Physical Displacement

In [ ]:
Z_SCALE = 1.625
Y_SCALE = 0.40625
X_SCALE = 0.40625
TIME_SCALE = 1.0

edge_data["delta_z_um"] = (
    edge_data["delta_z_voxels"] * Z_SCALE
)

edge_data["delta_y_um"] = (
    edge_data["delta_y_voxels"] * Y_SCALE
)

edge_data["delta_x_um"] = (
    edge_data["delta_x_voxels"] * X_SCALE
)

edge_data["distance_um"] = np.sqrt(
    edge_data["delta_z_um"] ** 2
    + edge_data["delta_y_um"] ** 2
    + edge_data["delta_x_um"] ** 2
)

edge_data["elapsed_seconds"] = (
    edge_data["delta_t"] * TIME_SCALE
)

edge_data["speed_um_per_second"] = (
    edge_data["distance_um"] / edge_data["elapsed_seconds"]
)

In [ ]:
display(
    edge_data[
        [
            "delta_z_um",
            "delta_y_um",
            "delta_x_um",
            "distance_um",
            "speed_um_per_second"
        ]
    ].describe()
)

### Physical movement

The median physical displacement is 2.88 µm per frame, while the mean is 3.12 µm. The middle 50% of movements fall between about 2.10 and 3.85 µm, with the largest observed displacement being 7.81 µm.

The mean signed displacement is positive across all three spatial axes, especially along the Z-axis. This could suggest some directional movement through the volume, but I wouldn't generalize this yet. 48 of the 50 edges come from one trajectory, so the movements within that trajectory aren't statistically independent.

The 7 µm tolerance in the evaluation metric is used for matching predicted and ground-truth node positions. It shouldn't be treated as a maximum distance that an edge is allowed to have.


## Visualise the Trajectories

In [ ]:
components_by_size = sorted(
    weak_components,
    key=len,
    reverse=True
)

component_map = {
    int(node_id): component_number
    for component_number, component in enumerate(
        components_by_size,
        start=1
    )
    for node_id in component
}

nodes_plot = nodes.copy()

nodes_plot["component"] = (
    nodes_plot["node_id"]
    .map(component_map)
    .astype(str)
)

nodes_plot["z_um"] = nodes_plot["z"] * Z_SCALE
nodes_plot["y_um"] = nodes_plot["y"] * Y_SCALE
nodes_plot["x_um"] = nodes_plot["x"] * X_SCALE

nodes_plot = nodes_plot.sort_values(
    ["component", "t"]
)

display(
    nodes_plot.groupby("component").agg(
        n_nodes=("node_id", "size"),
        first_t=("t", "min"),
        last_t=("t", "max")
    )
)

In [ ]:
component_palette = {
    "1": "tab:blue",
    "2": "tab:orange"
}

component_order = ["1", "2"]

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(11, 9),
    sharex=True
)

coordinates = [
    ("z_um", "Z position (µm)"),
    ("y_um", "Y position (µm)"),
    ("x_um", "X position (µm)")
]

for ax, (column, ylabel) in zip(axes, coordinates):
    sns.lineplot(
        data=nodes_plot,
        x="t",
        y=column,
        hue="component",
        hue_order=component_order,
        palette=component_palette,
        marker="o",
        ax=ax,
        legend=(ax is axes[0])
    )
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")

axes[0].set_title(
    "Annotated cell coordinates through time"
)

axes[-1].set_xlabel("Time (seconds)")

plt.tight_layout()
plt.show()

In [ ]:
edge_data["component"] = (
    edge_data["source_id"]
    .map(component_map)
    .astype(str)
)

plt.figure(figsize=(11, 4))

sns.lineplot(
    data=edge_data,
    x="source_t",
    y="distance_um",
    hue="component",
    hue_order=component_order,
    palette=component_palette,
    marker="o"
)

plt.axhline(
    edge_data["distance_um"].median(),
    color="black",
    linestyle="--",
    linewidth=1,
    label="Overall median"
)

plt.title("Physical displacement between consecutive annotated nodes")
plt.xlabel("Source timepoint (seconds)")
plt.ylabel("Displacement (µm)")
plt.legend()
plt.tight_layout()
plt.show()

### Visual characteristics of the annotated trajectories

The larger component has 49 consecutive nodes from timepoints 27–75, while the smaller component has three consecutive nodes from timepoints 0–2.

The longer trajectory shows a pretty clear increase in Z over time, which suggests the cell is moving in the positive axial direction. The Y-coordinate increases at first and then becomes relatively stable, while the X-coordinate also shows a clear upward trend.

Even though the overall coordinate trends look pretty smooth, the frame-to-frame displacement varies a lot. It ranges from about 0.41 µm to 7.81 µm, with a median of 2.88 µm. This isn't necessarily contradictory because displacement is based on the first differences between positions, so it picks up the smaller changes in movement from frame to frame. The integer voxel coordinates can also add some quantisation to the estimated movement.

Most of these observations are based on Component 1 since it contains 48 of the graph's 50 edges. Component 2 only has two edges, so it's too short to really establish a movement pattern.

# Dataset-Level EDA

The initial investigation examined one selected movie in detail. The analysis now expands to all 199 training movies to determine how image dimensions, annotation coverage and graph sizes vary across the dataset.

Only Zarr and GEFF metadata will initially be read. The complete image arrays will not be loaded into memory.

## Build a movie-level metadata table

In [ ]:
def extract_movie_metadata(zarr_path):
    sample_id = zarr_path.stem
    group = sample_id.split("_")[0]

    # Open the microscopy movie without loading the full array
    movie_store = zarr.open(str(zarr_path), mode="r")
    image = movie_store["0"]

    t_size, z_size, y_size, x_size = image.shape

    scale = (
        movie_store.attrs["multiscales"][0]
        ["datasets"][0]
        ["coordinateTransformations"][0]
        ["scale"]
    )

    time_scale, z_scale, y_scale, x_scale = scale

    # Open the corresponding sparse annotation graph
    graph_path = train_root / f"{sample_id}.geff"
    graph_store = zarr.open(str(graph_path), mode="r")

    n_annotated_nodes = graph_store["nodes/ids"].shape[0]
    n_annotated_edges = graph_store["edges/ids"].shape[0]

    graph_metadata = graph_store.attrs["geff"]

    estimated_nodes = (
        graph_metadata
        .get("extra", {})
        .get("estimated_number_of_nodes", np.nan)
    )

    return {
        "sample_id": sample_id,
        "group": group,
        "is_test_movie": sample_id in test_ids,
        "n_timepoints": t_size,
        "z_size": z_size,
        "y_size": y_size,
        "x_size": x_size,
        "dtype": str(image.dtype),
        "time_scale": time_scale,
        "z_scale": z_scale,
        "y_scale": y_scale,
        "x_scale": x_scale,
        "n_annotated_nodes": n_annotated_nodes,
        "n_annotated_edges": n_annotated_edges,
        "estimated_nodes": estimated_nodes
    }

In [ ]:
movie_metadata = pd.DataFrame(
    extract_movie_metadata(path)
    for path in train_zarr
)

movie_metadata["annotation_fraction"] = (
    movie_metadata["n_annotated_nodes"]
    / movie_metadata["estimated_nodes"]
)

print("Shape:", movie_metadata.shape)

display(movie_metadata.head())

In [ ]:
print("Missing values:")
display(movie_metadata.isna().sum())

print("\nData types:")
display(movie_metadata.dtypes)

In [ ]:
display(
    movie_metadata[
        [
            "n_timepoints",
            "z_size",
            "y_size",
            "x_size",
            "n_annotated_nodes",
            "n_annotated_edges",
            "estimated_nodes",
            "annotation_fraction"
        ]
    ].describe()
)

### Initial Movie-Level Findings

All 199 training movies have the same dimensions: 100 timepoints, 64 Z-slices, and 256 × 256 pixel images. Since every movie has the same shape, differences between the datasets aren't coming from differences in image size.

The number of annotations varies quite a bit across the dataset. The supplied node count ranges from 50 to 1,950, while the estimated total number of cells ranges from 3,783 to 78,644.

The median annotation coverage is around 3.56%, with the middle 50% of movies falling between 1.04% and 11.80%. The average is a little higher at 6.13%, which suggests that a smaller number of movies have much higher annotation coverage and are pulling the average up. I'll look at the two dataset groups separately to see if that explains some of the variation.

For the example movie `44b6_0113de3b`, the estimated annotation coverage is only about 0.20%, which makes it one of the more sparsely annotated movies in the training set.

It's also important to remember that this coverage is only an estimate because it's based on the estimated total number of cells rather than an exact cell count.


## Check structural consistency

In [ ]:
structural_columns = [
    "n_timepoints",
    "z_size",
    "y_size",
    "x_size",
    "dtype",
    "time_scale",
    "z_scale",
    "y_scale",
    "x_scale"
]

structural_summary = pd.DataFrame({
    "n_unique_values": movie_metadata[
        structural_columns
    ].nunique(),
    
    "values": [
        movie_metadata[column].unique().tolist()
        for column in structural_columns
    ]
})

display(structural_summary)

## Examine Test-Movie Annotation Coverage

In [ ]:
test_movie_metadata = movie_metadata.loc[
    movie_metadata["is_test_movie"],
    [
        "sample_id",
        "group",
        "n_annotated_nodes",
        "n_annotated_edges",
        "estimated_nodes",
        "annotation_fraction"
    ]
].copy()

test_movie_metadata["annotation_percentage"] = (
    100 * test_movie_metadata["annotation_fraction"]
)

display(test_movie_metadata)

In [ ]:
movie_metadata["coverage_percentile_overall"] = (
    100 * movie_metadata["annotation_fraction"].rank(pct=True)
)

movie_metadata["coverage_percentile_within_group"] = (
    100
    * movie_metadata.groupby("group")["annotation_fraction"]
    .rank(pct=True)
)

display(
    movie_metadata.loc[
        movie_metadata["is_test_movie"],
        [
            "sample_id",
            "group",
            "annotation_fraction",
            "coverage_percentile_overall",
            "coverage_percentile_within_group"
        ]
    ].sort_values("annotation_fraction")
)

### Structural Consistency and Test-Movie Coverage

All 199 training movies have the same dimensions, data type, time intervals, and physical voxel spacing. So, any differences between the movies aren't coming from changes in image structure or resolution.

The four test movies have pretty different levels of annotation coverage. The two `44b6` test movies are among the most sparsely annotated, with estimated coverage of about 0.16% and 0.20%.

The two `6bba` test movies have estimated coverage of about 1.76% and 13.53%. One is still relatively sparse, while the other has much higher coverage compared to the rest of its group.

Since the overall percentiles and the group-specific percentiles don't really line up, it looks like the `44b6` and `6bba` groups may have different annotation coverage distributions. Because of this, it probably makes more sense to analyze the two groups separately instead of combining them into one overall distribution.

## Compare annotation coverage between groups

Estimated Coverage = annotated nodes / estimated total nodes


In [ ]:
movie_metadata["annotation_percentage"] = (
    100 * movie_metadata["annotation_fraction"]
)

coverage_by_group = (
    movie_metadata
    .groupby("group")
    .agg(
        n_movies=("sample_id", "size"),
        minimum=("annotation_percentage", "min"),
        first_quartile=(
            "annotation_percentage",
            lambda values: values.quantile(0.25)
        ),
        median=("annotation_percentage", "median"),
        mean=("annotation_percentage", "mean"),
        third_quartile=(
            "annotation_percentage",
            lambda values: values.quantile(0.75)
        ),
        maximum=("annotation_percentage", "max")
    )
)

display(coverage_by_group.round(3))

In [ ]:
plt.figure(figsize=(9, 6))

ax = sns.boxplot(
    data=movie_metadata,
    x="group",
    y="annotation_percentage",
    hue="group",
    palette={
        "44b6": "tab:blue",
        "6bba": "tab:orange"
    },
    showfliers=False,
    legend=False
)

sns.stripplot(
    data=movie_metadata,
    x="group",
    y="annotation_percentage",
    color="black",
    alpha=0.45,
    jitter=0.22,
    size=4,
    ax=ax
)

test_plot_data = movie_metadata[
    movie_metadata["is_test_movie"]
]

sns.scatterplot(
    data=test_plot_data,
    x="group",
    y="annotation_percentage",
    color="red",
    edgecolor="black",
    marker="D",
    s=110,
    label="Test movie",
    ax=ax
)

ax.set_yscale("log")

ax.set(
    title="Estimated annotation coverage by dataset group",
    xlabel="Dataset group",
    ylabel="Estimated annotation coverage (%)"
)

ax.legend()
plt.tight_layout()
plt.show()

### Annotation Coverage Differs Strongly Between Groups

The estimated annotation coverage is pretty different between the `44b6` and `6bba` groups. The median coverage is about 0.77% for `44b6` and 9.72% for `6bba`, so a typical `6bba` movie has a lot more of its estimated cells annotated.

The middle 50% of the two groups don't overlap either. For `44b6`, the interquartile range is about 0.39% to 1.24%, while for `6bba` it's about 3.94% to 13.40%.

The two `44b6` test movies are both near the lower end of their group's distribution. The two `6bba` test movies are more different from each other, with one having relatively low coverage and the other having much higher coverage.

Overall, this shows that combining the two groups into one summary can hide some important differences. Because of this, I'll keep the `44b6` and `6bba` groups separate for the rest of the analysis and when evaluating model performance.


## Decompose the Annotation-Coverage Difference

In [ ]:
node_summary_by_group = (
    movie_metadata
    .groupby("group")
    .agg(
        median_annotated_nodes=(
            "n_annotated_nodes",
            "median"
        ),
        mean_annotated_nodes=(
            "n_annotated_nodes",
            "mean"
        ),
        median_estimated_nodes=(
            "estimated_nodes",
            "median"
        ),
        mean_estimated_nodes=(
            "estimated_nodes",
            "mean"
        )
    )
)

display(node_summary_by_group.round(1))

In [ ]:
group_palette = {
    "44b6": "tab:blue",
    "6bba": "tab:orange"
}

fig, ax = plt.subplots(figsize=(10, 7))

sns.scatterplot(
    data=movie_metadata,
    x="estimated_nodes",
    y="n_annotated_nodes",
    hue="group",
    hue_order=["44b6", "6bba"],
    palette=group_palette,
    alpha=0.7,
    s=55,
    ax=ax
)

sns.scatterplot(
    data=movie_metadata[movie_metadata["is_test_movie"]],
    x="estimated_nodes",
    y="n_annotated_nodes",
    color="red",
    edgecolor="black",
     marker="D",
    s=120,
    label="Test movie",
    ax=ax
)

x_values = np.geomspace(
    movie_metadata["estimated_nodes"].min(),
    movie_metadata["estimated_nodes"].max(),
    200
)

for percentage in [0.2, 1, 5, 10, 20]:
    ax.plot(
        x_values,
        x_values * percentage / 100,
        linestyle="--",
        linewidth=1,
        alpha=0.7,
        label=f"{percentage}% coverage"
    )

ax.set_xscale("log")
ax.set_yscale("log")

ax.set(
    title="Annotated nodes versus estimated total nodes",
    xlabel="Estimated total nodes",
    ylabel="Annotated nodes"
)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

### Annotated and Estimated Node Counts

The two dataset groups are pretty clearly separated in the plot. Most `6bba` movies have fewer estimated total cells but more annotated nodes, so they show up toward the upper-left. Most `44b6` movies have more estimated cells but fewer annotations, putting them toward the lower-right.

There is some overlap between the groups, but the overall difference is still pretty noticeable. This also helps explain why the `6bba` movies have higher annotation coverage overall. They generally have more annotated cells while also having fewer estimated total cells.

Because the two groups are so different, looking at all the movies together could give a misleading idea of the relationship between estimated and annotated node counts. I'll keep the `44b6` and `6bba` groups separate for the rest of the EDA.


## Simpson’s paradox

In [ ]:
overall_correlation = movie_metadata[
    ["estimated_nodes", "n_annotated_nodes"]
].corr(method="spearman").iloc[0, 1]

group_correlations = (
    movie_metadata
    .groupby("group")
    .apply(
        lambda group_data: group_data[
            ["estimated_nodes", "n_annotated_nodes"]
        ].corr(method="spearman").iloc[0, 1],
        include_groups=False
    )
    .rename("spearman_correlation")
)

print(
    "Overall Spearman correlation:",
    round(overall_correlation, 3)
)

display(group_correlations.round(3))

### Pooled and Within-Group Associations

Across all of the movies, the estimated total nodes and annotated nodes have a Spearman correlation of −0.278. But when I look at the two groups separately, the relationship is different. The `44b6` group has a positive correlation of 0.259, while the `6bba` group has a positive correlation of 0.160.

So, the overall negative correlation seems to be coming from the differences between the two groups rather than what's happening within each group. `44b6` movies generally have more estimated cells but fewer annotations, while `6bba` movies generally have fewer estimated cells but more annotations.

Within each group, movies with larger estimated cell populations tend to have slightly more annotated nodes, but both relationships are weak. This is another example of why combining the two groups can be misleading, since the differences between the groups are affecting the overall pattern.


## Section Summary

* All movies have the same array dimensions, data type, time intervals, and voxel spacing, so these aren't what explain the differences between the datasets.
* Annotation coverage is very different between the two groups, with `6bba` generally having much higher coverage than `44b6`.
* The four test movies have different levels of annotation coverage within their respective groups.
* Overall, `44b6` movies tend to have more estimated cells but fewer annotations, while `6bba` movies tend to have fewer estimated cells but more annotations.
* The relationship between estimated and annotated node counts changes depending on whether I look at all the movies together or each group separately. Because of this, the dataset group needs to be taken into account in the rest of the analysis and validation.


# Looking at the Microscopy Images

So far, I've mainly looked at the image metadata and the sparse lineage annotations. Now I want to actually look at the microscopy images to get a better idea of what the cells look like, how packed they are, and how they're distributed throughout the image.

To keep things manageable, I'm starting with just one timepoint from the example movie instead of loading the entire 4D dataset. This lets me inspect the image without using all the memory it would take to load the full movie.


## What I'm Looking for in the Images

Since the goal of this competition is to detect cells and track them over time, I first want to understand what the cells actually look like in the images and how they stand out from the background before choosing a detection method.

While looking through the images, I'll focus on a few things:

* Do the cells show up as distinct bright structures?
* Are the cell boundaries sharp or blurry?
* Can I easily tell nearby cells apart?
* Does the cell density change across different parts of the image?
* Is the background intensity fairly consistent?
* Are there any bright artifacts or effects around the edges?
* Would a simple local-maximum detector be a reasonable starting point?
* How much information am I losing by looking at individual 2D slices instead of the full 3D volume?

Answering these questions should give me a better idea of what preprocessing and detection methods make sense next, including intensity normalization, smoothing, detection thresholds, minimum separation distances, and whether I should detect cells in 2D or 3D.


## Load One 3D Timepoint

In [ ]:
image_array = train_store["0"]

print("Full shape:", image_array.shape)
print("Chunk shape:", image_array.chunks)
print("Data type:", image_array.dtype)

selected_t = 50
volume = np.asarray(image_array[selected_t])

print("\nSelected timepoint:", selected_t)
print("Volume shape:", volume.shape)
print("Memory used (MiB):", volume.nbytes / 1024**2)

display(
    pd.Series(volume.ravel())
    .describe(
        percentiles=[0.01, 0.10, 0.50, 0.90, 0.99, 0.999]
    )
)

### Volume and Intensity Summary

The selected timepoint contains **4,194,304 voxels** and uses about **8 MiB** of memory. Since the data is stored in **Zarr**, I can load one 3D timepoint at a time instead of loading the entire movie, which makes exploring the data much easier and more efficient.

The median voxel intensity is **202**, while the mean is about **269**. Around **90%** of the voxels have intensity values below **474**, and only the brightest **1%** are above **1,436**. These numbers tell me about the brightness of the fluorescence signal, not the number of cells in the image.

It's also important to remember that neighboring voxels aren't independent. One cell can take up many voxels, so I shouldn't treat the total number of voxels as the number of cells or as separate observations.

## Visualise Depth Slices


In [ ]:
z_slices = [0, 8, 16, 24, 32, 40, 48, 56]

vmin = np.quantile(volume, 0.01)
vmax = np.quantile(volume, 0.999)

fig, axes = plt.subplots(
    2,
    4,
    figsize=(14, 7)
)

for ax, z_index in zip(axes.flat, z_slices):
    ax.imshow(
        volume[z_index],
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )

    ax.set_title(
        f"Z = {z_index} "
        f"({z_index * Z_SCALE:.1f} µm)"
    )

    ax.axis("off")

fig.suptitle(
    f"Depth slices from {sample_id} at t = {selected_t}",
    fontsize=14
)

plt.tight_layout()
plt.show()

### Visual Characteristics Across Depth

The cells show up as bright structures against a darker background, and a lot of them have clear bright centers. Some cells are very close together or overlap, and the blurry boundaries make them harder to separate when looking at individual 2D slices.

The cell density looks fairly consistent across most of the slices from what I can see, but this is just a visual observation and not an actual measurement. Since cells can extend across multiple depth slices, the same cell can also appear in more than one slice.

At higher Z-values, especially around **Z = 48** and **Z = 56**, most of the bright structures shift toward the right side of the image. This means fewer cells are visible within the field of view in those slices.

Overall, it looks like the cell centers could be detected using local intensity peaks. However, blurry boundaries, overlapping cells, and cells near the image edges will probably make detection more difficult.

This makes local intensity maxima a reasonable starting point for detecting cell centers, but I'll need to account for close neighboring cells, blurred boundaries, and cells that are partially clipped by the image edges.

## Maximum-Intensity Projections

A maximum-intensity projection takes the brightest voxel along each line through the 3D volume. This gives me a quick overview of where the strongest fluorescence signals are without having to go through all **64** depth slices individually.

I'll look at three different projections:

* **XY:** viewing through the Z-axis
* **XZ:** viewing through the Y-axis
* **YZ:** viewing through the X-axis

These projections make it easier to see the overall shape of the fluorescent region, where the bright structures are located, and how they're distributed throughout the volume. The main downside is that they remove the depth information, so cells at different depths can appear on top of each other.


In [ ]:
xy_projection = volume.max(axis=0)
xz_projection = volume.max(axis=1)
yz_projection = volume.max(axis=2)

x_extent = volume.shape[2] * X_SCALE
y_extent = volume.shape[1] * Y_SCALE
z_extent = volume.shape[0] * Z_SCALE

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
    constrained_layout=True
)

projection_data = [
    (
        xy_projection,
        [0, x_extent, 0, y_extent],
        "XY projection",
        "X position (µm)",
        "Y position (µm)"
    ),
    (
                xz_projection,
        [0, x_extent, 0, z_extent],
        "XZ projection",
        "X position (µm)",
        "Z position (µm)"
    ),
    (
        yz_projection,
        [0, y_extent, 0, z_extent],
        "YZ projection",
        "Y position (µm)",
        "Z position (µm)"
    )
]

for ax, (
    projection,
    extent,
    title,
    xlabel,
    ylabel
) in zip(axes, projection_data):

    image = ax.imshow(
        projection,
        cmap="gray",
        origin="lower",
        extent=extent,
        vmin=vmin,
        vmax=vmax,
        aspect="equal"
    )

    ax.set(
        title=title,
        xlabel=xlabel,
        ylabel=ylabel
    )

fig.colorbar(
    image,
    ax=axes,
    shrink=0.8,
    label="Fluorescence intensity"
)

fig.suptitle(
    f"Maximum-intensity projections for {sample_id} at t = {selected_t}",
    fontsize=14
)

plt.show() 

### Spatial Structure in the Maximum-Intensity Projections

In the **XY projection**, the bright structures are spread across most of the image, but there are noticeably fewer cells in the lower-left corner. Some of the bright structures also reach the top, right, and bottom edges, so they look partially cut off by the image boundaries.

The **XZ projection** shows a pretty clear diagonal band going from the lower-left toward the upper-right. This suggests that as **Z** increases, the main fluorescent region shifts toward larger **X** values. The middle of the band also looks more crowded, while there are fewer distinct structures toward the far right.

The **YZ projection** looks a little different. Bright structures are spread across most of the image, and there isn't the same strong diagonal pattern that I see in the XZ projection. From looking at these projections, the fluorescent region seems to change more along the **X-axis** than the **Y-axis** as depth increases.

These projections are useful for getting a general idea of the 3D volume, but they don't show the actual 3D arrangement of the cells. Since one dimension is collapsed into a single image, cells at different depths can overlap and make some areas look more crowded than they really are.

## Section Summary

* The cells appear as bright structures, so a 3D local-maximum detector seems like a reasonable starting point.
* Blurry boundaries, overlapping cells, and cells near the image edges will probably make detection harder.
* The fluorescent region shifts toward larger **X** values as **Z** increases instead of being evenly distributed.
* Individual slices preserve the depth information, while maximum-intensity projections give me a better overall view but can cause cells at different depths to overlap.

## Overlaying the Annotations on the Microscopy Image

Next, I overlaid the GEFF annotations on the microscopy image to check whether the coordinates line up correctly and whether the annotated cell centers actually match the bright fluorescent structures.

Instead of using maximum-intensity projections, I looked at the exact image slices where each annotation is located. This makes it easier to see the specific cell without bright structures from other depths overlapping with it.

## Select an Annotated Node


In [ ]:
annotations_at_t = (
    nodes[nodes["t"] == selected_t]
    .copy()
    .sort_values("node_id")
)

print(
    f"Annotated nodes at t = {selected_t}:",
    len(annotations_at_t)
)

display(annotations_at_t)

In [ ]:
if annotations_at_t.empty:
    raise ValueError(
        f"No annotated nodes found at t = {selected_t}"
    )

selected_node = annotations_at_t.iloc[0]

node_id = int(selected_node["node_id"])
node_z = int(selected_node["z"])
node_y = int(selected_node["y"])
node_x = int(selected_node["x"])

node_z_um = node_z * Z_SCALE
node_y_um = node_y * Y_SCALE
node_x_um = node_x * X_SCALE

print("Node ID:", node_id)
print("Voxel coordinates:", (node_z, node_y, node_x))
print(
    "Physical coordinates (µm):",
    (
        round(node_z_um, 2),
        round(node_y_um, 2),
        round(node_x_um, 2)
     )
)

## Inspect the Annotated Cell from Three Directions

In [ ]:
xy_slice = volume[node_z, :, :]
xz_slice = volume[:, node_y, :]
yz_slice = volume[:, :, node_x]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
    constrained_layout=True
)

slice_data = [
    (
        xy_slice,
        [0, x_extent, 0, y_extent],
        node_x_um,
        node_y_um,
        f"XY slice at Z = {node_z}",
        "X position (µm)",
        "Y position (µm)"
    ),
    (
        xz_slice,
        [0, x_extent, 0, z_extent],
         node_x_um,
        node_z_um,
        f"XZ slice at Y = {node_y}",
        "X position (µm)",
        "Z position (µm)"
    ),
    (
        yz_slice,
        [0, y_extent, 0, z_extent],
        node_y_um,
        node_z_um,
        f"YZ slice at X = {node_x}",
        "Y position (µm)",
        "Z position (µm)"
    )
]

for ax, (
    image_slice,
    extent,
    point_horizontal,
    point_vertical,
    title,
    xlabel,
    ylabel
 ) in zip(axes, slice_data):

    image = ax.imshow(
        image_slice,
        cmap="gray",
        origin="lower",
        extent=extent,
        vmin=vmin,
        vmax=vmax,
        aspect="equal"
    )

    ax.scatter(
        point_horizontal,
        point_vertical,
        s=180,
        facecolors="none",
        edgecolors="red",
        linewidths=2,
        label="Annotated centroid"
    )

    ax.set(
        title=title,
        xlabel=xlabel,
        ylabel=ylabel
    )

axes[0].legend(loc="upper right")

fig.colorbar(
    image,
    ax=axes,
    shrink=0.8,
    label="Fluorescence intensity"
)

fig.suptitle(
    f"Orthogonal views through annotated node {node_id} at t = {selected_t}",
    fontsize=14
)

plt.show()

### Annotation Alignment and Apparent Shape

The annotated centroid lines up pretty well with a bright fluorescent structure in all three views.

The red circle is just showing the annotated center. It doesn't represent the actual size or boundary of the cell.

The cell looks more compact in the **XY** view but more stretched out in the **XZ** and **YZ** views. This is probably because the resolution along the **Z-axis** is lower, although one example isn't enough to say that for sure.


## Checking Annotation Alignment Across Time

Next, I looked at several nodes from the longer annotated trajectory to see whether the annotated centroids stay aligned with the bright cell structures over time.

Each image is an **XY** crop taken at the annotated **Z** slice. I placed the annotated centroid at **(0, 0)** in each crop so I could more easily compare the cell across different time points.


In [ ]:
long_component = (
    nodes_plot[nodes_plot["component"] == "1"]
    .sort_values("t")
)

selected_indices = np.linspace(
    0,
    len(long_component) - 1,
    6
).round().astype(int)

selected_annotations = (
    long_component.iloc[selected_indices]
    .reset_index(drop=True)
)

display(
    selected_annotations[
        ["node_id", "t", "z", "y", "x"]
    ]
)

### Annotation Alignment Across Time

All six sampled annotations line up well with the bright center of their corresponding cells. The alignment stays pretty consistent as the cells move over time and across the image.

Some of the crops have nearby bright cells, but the annotated cell is still easy to identify. This suggests that the GEFF coordinates are doing a good job of marking the cell centers.

All six samples come from the same trajectory, so I shouldn't treat them as six independent cells.

## Section Summary

* The GEFF annotations line up well with the Zarr images in the example movie.
* The annotated centroids match the bright cell centers in all three views and stay aligned throughout the sampled trajectory.
* Cells look more stretched in views that include the **Z-axis**, which is probably because the resolution is lower in that direction.
* Overall, the annotations seem to be good representations of the cell centers. Since I only checked one trajectory, I'll look at more movies later to make sure this alignment is consistent across the dataset.


# Explore Intensity Variation Across Movies and Time

So far, I've mainly focused on the image structure and the annotation data. Next, I want to look at how the fluorescence intensity changes between different movies, dataset groups, and timepoints. The main question is whether one intensity scale works across the whole dataset or if I should normalize the images before training a model.


## Compare Intensity Quantiles Across Movies

Each Zarr movie has precomputed intensity quantiles, so I can compare the fluorescence brightness across the dataset without having to load every image volume.

By comparing these values, I can see how similar or different the intensity ranges are between movies. If the differences are large, it would probably make more sense to normalize the images instead of using one fixed intensity threshold across every movie.


In [ ]:
intensity_records = []

for zarr_path in train_zarr:
    sample_id_current = zarr_path.stem

    store = zarr.open(
        str(zarr_path),
        mode="r"
    )

    quantiles = (
        store.attrs["image_statistics"]["quantiles"]
    )

    intensity_records.append({
        "sample_id": sample_id_current,
        "intensity_min": float(quantiles["0.0"]),
        "intensity_p001": float(quantiles["0.001"]),
        "intensity_p01": float(quantiles["0.01"]),
        "intensity_p10": float(quantiles["0.1"]),
        "intensity_p90": float(quantiles["0.9"]),
        "intensity_p99": float(quantiles["0.99"]),
        "intensity_p999": float(quantiles["0.999"]),
        "intensity_max": float(quantiles["1.0"])
    })

intensity_metadata = pd.DataFrame(
    intensity_records
)

movie_metadata = movie_metadata.merge(
    intensity_metadata,
    on="sample_id",
    how="left",
    validate="one_to_one"
)

display(intensity_metadata.head())

In [ ]:
intensity_columns = [
    "intensity_min",
    "intensity_p001",
    "intensity_p01",
    "intensity_p10",
    "intensity_p90",
    "intensity_p99",
    "intensity_p999",
    "intensity_max"
]

display(
    movie_metadata[intensity_columns]
    .describe()
)

In [ ]:
display(
    movie_metadata
    .groupby("group")[intensity_columns]
    .median()
    .round(1)
)

### Initial Findings from Stored Intensity Quantiles

The raw fluorescence intensity varies quite a bit across the **199 movies**, so using one fixed intensity threshold for the whole dataset probably won't work very well.

I also noticed that the **44b6** movies generally have higher intensity values than the **6bba** movies. For example, the median **P90** intensity is about **1,294** for **44b6**, compared to **452** for **6bba**.

The maximum intensity also varies a lot, ranging from **1,010** to **18,827**. Since the maximum can be affected by just a few unusually bright voxels, the higher intensity percentiles will probably be more useful for normalization than the minimum or maximum values.

I also don't want to assume that these differences mean one movie is actually brighter than another. They could partly be caused by how much of the image is occupied by fluorescent cells.



### Upper-Intensity Quantiles by Dataset Group

To make these differences easier to see, I'm comparing the upper intensity quantiles on a **logarithmic scale** because the values vary so much between movies. I'm mainly using this to compare the intensity ranges between the two dataset groups rather than trying to analyze the exact shape of each distribution.


In [ ]:
upper_quantile_names = {
    "intensity_p90": "P90",
    "intensity_p99": "P99",
    "intensity_p999": "P99.9"
}

intensity_long = (
    movie_metadata
    .melt(
        id_vars=["sample_id", "group", "is_test_movie"],
        value_vars=list(upper_quantile_names),
        var_name="quantile",
        value_name="intensity"
    )
)

intensity_long["quantile"] = (
    intensity_long["quantile"]
    .map(upper_quantile_names)
)

quantile_order = ["P90", "P99", "P99.9"]

plt.figure(figsize=(10, 6))

ax = sns.boxplot(
    data=intensity_long,
    x="quantile",
    y="intensity",
    hue="group",
    order=quantile_order,
    hue_order=["44b6", "6bba"],
    palette=group_palette,
    showfliers=False
)

sns.stripplot(
    data=intensity_long,
    x="quantile",
    y="intensity",
    hue="group",
    order=quantile_order,
    hue_order=["44b6", "6bba"],
    dodge=True,
    palette=group_palette,
    alpha=0.3,
    size=3,
    legend=False,
    ax=ax
)

ax.set_yscale("log")

ax.set(
    title="Upper-intensity quantiles by dataset group",
    xlabel="Movie-level intensity quantile",
    ylabel="Fluorescence intensity (log scale)"
)

ax.legend(title="Dataset group")

plt.tight_layout()
plt.show()

### Differences in Upper-Intensity Levels

The **44b6** movies generally have higher intensity values than the **6bba** movies across **P90, P99, and P99.9**. The middle 50% of the two groups don't overlap at any of these quantiles, so there is a pretty clear difference between the groups.

There is still some overlap between individual movies, though, so the groups aren't completely separated based on intensity alone.

The difference also gets smaller at the highest intensity levels. The median ratio drops from about **2.86** at **P90** to **1.85** at **P99.9**, which means the two groups become more similar toward the brightest part of the intensity distribution.



### Test-Movie Intensity Percentiles

I compared the upper-intensity values of each test movie to the other movies in the same dataset group. This helps me see whether the test movies have typical intensity levels for their group or if they are unusually high compared to the rest.


In [ ]:
test_intensity_columns = [
    "intensity_p90",
    "intensity_p99",
    "intensity_p999"
]

for column in test_intensity_columns:
    movie_metadata[f"{column}_group_percentile"] = (
        100
        * movie_metadata.groupby("group")[column]
        .rank(pct=True)
    )

test_intensity_summary = movie_metadata.loc[
    movie_metadata["is_test_movie"],
    [
        "sample_id",
        "group",
        "intensity_p90",
        "intensity_p90_group_percentile",
        "intensity_p99",
        "intensity_p99_group_percentile",
        "intensity_p999",
        "intensity_p999_group_percentile"
    ]
]

display(
    test_intensity_summary.round(1)
)

## Test Movies Span Different Intensity Levels

The four test movies have pretty different intensity patterns compared to the other movies in their respective groups.

`44b6_0113de3b` is relatively dim, especially at **P90**, while `44b6_0b24845f` is much brighter. In the `6bba` group, `6bba_05b6850b` starts below average at **P90** but gets closer to average by **P99.9**. Meanwhile, `6bba_05db0fb1` stays very bright and is above the **93rd percentile** across all three intensity levels.

This shows that being in the same dataset group doesn't necessarily mean the movies will have similar intensity levels. Because of this, I may need to normalize each movie or frame separately when setting up the detection method.

The fact that the rankings change across different intensity levels also makes it seem like this isn't just a simple brightness difference. Things like contrast or differences in how many bright structures are present could also be affecting the intensity distribution.



## Compare Test Movies Before and After Normalisation

I'm comparing the four test movies using maximum-intensity projections before and after normalization. I'm looking at two intensity settings:

* a shared raw intensity scale, which keeps the original brightness differences between the movies
* movie-specific percentile normalization, which adjusts each movie based on its own intensity range so the fluorescent structures are easier to compare

This should help me see which differences are removed by normalization and which differences between the movies are still there.


In [ ]:
test_frame_data = {}

for test_path in test_zarr:
    test_id = test_path.stem

    test_array = zarr.open(
        str(test_path),
        mode="r"
    )["0"]

    test_volume = np.asarray(
        test_array[selected_t]
    )

    test_frame_data[test_id] = {
        "volume": test_volume,
        "xy_projection": test_volume.max(axis=0)
    }

In [ ]:
test_metadata_indexed = (
    movie_metadata[
        movie_metadata["is_test_movie"]
    ]
    .set_index("sample_id")
)

shared_vmin = (
    test_metadata_indexed["intensity_p01"].min()
)

shared_vmax = (
    test_metadata_indexed["intensity_p999"].max()
)

print("Shared minimum:", shared_vmin)
print("Shared maximum:", shared_vmax)

In [ ]:
test_order = [
    "44b6_0113de3b",
    "44b6_0b24845f",
    "6bba_05b6850b",
    "6bba_05db0fb1"
]

fig, axes = plt.subplots(
    2,
    4,
    figsize=(16, 8),
    constrained_layout=False
)

for column, test_id in enumerate(test_order):
    projection = (
        test_frame_data[test_id]["xy_projection"]
    )

    metadata_row = test_metadata_indexed.loc[test_id]

    movie_vmin = metadata_row["intensity_p01"]
    movie_vmax = metadata_row["intensity_p999"]

    normalised_projection = np.clip(
        (projection - movie_vmin)
        / (movie_vmax - movie_vmin),
        0,
        1
    )

    axes[0, column].imshow(
        projection,
        cmap="gray",
        origin="lower",
        vmin=shared_vmin,
        vmax=shared_vmax
    )

    axes[1, column].imshow(
        normalised_projection,
        cmap="gray",
        origin="lower",
        vmin=0,
        vmax=1
    )

    axes[0, column].set_title(test_id)

    for row in range(2):
        axes[row, column].axis("off")

fig.text(
    0.025,
    0.72,
    "Shared raw intensity scale",
    rotation=90,
    va="center",
    ha="center",
    fontsize=12,
    fontweight="bold"
)

fig.text(
    0.025,
    0.29,
    "Movie-specific normalisation",
    rotation=90,
    va="center",
    ha="center",
    fontsize=12,
    fontweight="bold"
)

fig.suptitle(
    f"XY projections of test movies at t = {selected_t}",
    fontsize=14
)

plt.tight_layout(
    rect=[0.02, 0.02, 1, 0.94]
)

plt.show()

### Effect of Movie-Specific Normalisation

Movie-specific normalization makes the bright structures stand out more in `44b6_0113de3b` and `6bba_05b6850b` because these movies had lower intensity values compared to the others in their groups.

For `44b6_0b24845f` and `6bba_05db0fb1`, the raw and normalized images look more similar because these movies were already relatively bright.

Normalization doesn't add any new information to the images. It just changes the intensity scale so the existing structures are easier to see. At the same time, it can also make background noise more noticeable. Because of this, I'll need to actually test whether normalization helps with cell detection instead of assuming it will always improve it.


### Examine Intensity Stability Through Time

Next, I'm looking at how the intensity changes over time using every fifth timepoint from each test movie. I'm also using a smaller sample of voxels to make the calculation faster while still getting a general idea of the intensity pattern.

I'm mainly checking whether the intensity stays relatively consistent, increases, or decreases throughout the movie. Any changes could come from imaging conditions, movement of the fluorescent structures, or differences in how much of the volume is occupied by those structures.


In [ ]:
sampled_timepoints = range(0, 100, 5)

temporal_intensity_records = []

for test_path in test_zarr:
    test_id = test_path.stem

    image_data = zarr.open(
        str(test_path),
        mode="r"
    )["0"]

    for t in sampled_timepoints:
        frame = np.asarray(image_data[t])

        # Use a regular spatial subsample
        sampled_voxels = frame[
            ::2,
            ::4,
            ::4
        ].ravel()

        p10, p50, p90, p99 = np.quantile(
            sampled_voxels,
            [0.10, 0.50, 0.90, 0.99]
        )

        temporal_intensity_records.append({
            "sample_id": test_id,
            "group": test_id.split("_")[0],
            "t": t,
            "intensity_p10": p10,
            "intensity_p50": p50,
            "intensity_p90": p90,
            "intensity_p99": p99
        })

temporal_intensity = pd.DataFrame(
    temporal_intensity_records
)

print("Shape:", temporal_intensity.shape)
display(temporal_intensity.head())

In [ ]:
temporal_summary = (
    temporal_intensity
    .groupby("sample_id")
    .agg(
        p90_min=("intensity_p90", "min"),
        p90_max=("intensity_p90", "max"),
        p90_mean=("intensity_p90", "mean"),
        p90_std=("intensity_p90", "std"),
        p99_min=("intensity_p99", "min"),
        p99_max=("intensity_p99", "max"),
        p99_mean=("intensity_p99", "mean"),
        p99_std=("intensity_p99", "std")
    )
)

temporal_summary["p90_cv"] = (
    temporal_summary["p90_std"]
    / temporal_summary["p90_mean"]
)

temporal_summary["p99_cv"] = (
    temporal_summary["p99_std"]
    / temporal_summary["p99_mean"]
)

display(
    temporal_summary[
        [
            "p90_min",
            "p90_max",
            "p90_cv",
            "p99_min",
            "p99_max",
            "p99_cv"
        ]
    ].round(3)
)

### Initial Temporal-Variability Findings

The four test movies show pretty different levels of intensity stability over time. `6bba_05db0fb1` is the most stable, with coefficients of variation below 2% for both **P90** and **P99**.

On the other hand, `44b6_0b24845f` has the most variation. Its **P90** intensity goes from 1,548 to 2,537, with a coefficient of variation of about 13.7%.

The other two movies have more moderate variation, with coefficients of variation around 5%–7%. Across all four movies, **P99** is also slightly more stable than **P90**.

So, the intensity does change over time, but this only tells me how much it changes. It doesn't tell me yet whether the intensity is gradually increasing or decreasing or if it's mostly just fluctuating from one timepoint to another.


### Relative Intensity Through Time

For each movie, I normalized the **P90** and **P99** intensity values by dividing them by that movie's temporal median. A value of **1** represents the movie's usual intensity, while values above or below 1 show how much the intensity is changing compared to its normal level.

This gets rid of the overall brightness differences between the movies and makes it easier to compare how much the intensity changes over time.

In [ ]:
for column in ["intensity_p90", "intensity_p99"]:
    temporal_intensity[f"{column}_relative"] = (
        temporal_intensity[column]
        / temporal_intensity.groupby("sample_id")[column]
        .transform("median")
    )

test_movie_palette = {
    "44b6_0113de3b": "navy",
    "44b6_0b24845f": "cornflowerblue",
    "6bba_05b6850b": "darkorange",
    "6bba_05db0fb1": "goldenrod"
}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 5),
    sharey=True
)

plot_settings = [
    (
        "intensity_p90_relative",
        "P90 relative to temporal median"
    ),
    (
        "intensity_p99_relative",
        "P99 relative to temporal median"
    )
]

for ax, (column, title) in zip(
    axes,
    plot_settings
):
    sns.lineplot(
        data=temporal_intensity,
        x="t",
        y=column,
        hue="sample_id",
        hue_order=test_order,
        palette=test_movie_palette,
        marker="o",
        ax=ax,
        legend=(ax is axes[0])
    )

    ax.axhline(
        1,
        color="black",
        linestyle="--",
        linewidth=1
    )

    ax.set(
        title=title,
        xlabel="Timepoint",
        ylabel="Relative intensity"
    )

axes[0].legend(
    title="Test movie",
    bbox_to_anchor=(1.03, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()
        

### Temporal Patterns in Upper Intensities

The **P90** and **P99** values show pretty similar patterns over time within each test movie. This suggests that the changes aren't only coming from a few extremely bright voxels, but are affecting a larger part of the brighter intensity range.

`44b6_0b24845f` shows a steady increase in intensity throughout the movie. `44b6_0113de3b` increases at first but starts to decrease around timepoint 65. `6bba_05b6850b` has a temporary increase around timepoints 60–65, while `6bba_05db0fb1` stays relatively stable over time.

So, there are definitely changes in intensity over time, but I can't say yet what is causing them. They could be related to imaging conditions, movement of the fluorescent structures, or changes in how much of the volume is occupied by brighter structures.


### Lower and Central Intensities Through Time

I'm looking at **P10** and **P50** to see whether the intensity changes also affect the darker and middle parts of the image.

If **P10, P50, P90, and P99** all change in similar ways over time, that would suggest the whole image is going through a general intensity shift. But if the changes are mostly happening in the higher quantiles, the variation is more likely coming from changes in the fluorescent structures or how many bright regions are present in the image.


In [ ]:
for column in ["intensity_p10", "intensity_p50"]:
    temporal_median = (
        temporal_intensity
        .groupby("sample_id")[column]
        .transform("median")
    )

    temporal_intensity[f"{column}_relative"] = np.where(
        temporal_median > 0,
        temporal_intensity[column] / temporal_median,
        np.nan
    )

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 5),
    sharey=False
)

lower_plot_settings = [
    (
        "intensity_p10_relative",
        "P10 relative to temporal median"
    ),
    (
        "intensity_p50_relative",
        "P50 relative to temporal median"
    )
]

for ax, (column, title) in zip(
    axes,
    lower_plot_settings
):
    sns.lineplot(
        data=temporal_intensity,
        x="t",
        y=column,
        hue="sample_id",
        hue_order=test_order,
        palette=test_movie_palette,
        marker="o",
        ax=ax,
        legend=(ax is axes[0])
    )

    ax.axhline(
        1,
        color="black",
        linestyle="--",
        linewidth=1
    )

    ax.set(
        title=title,
        xlabel="Timepoint",
        ylabel="Relative intensity"
    )

axes[0].legend(
    title="Test movie",
    bbox_to_anchor=(1.03, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show() 

### Temporal Changes Differ Across the Intensity Distribution

The intensity doesn't change the same way across **P10, P50, P90, and P99** over time. This suggests that the whole image isn't just getting brighter or darker by the same amount.

`44b6_0b24845f` shows the biggest changes over time. Its **P50, P90, and P99** increase, while **P10** has a much stronger increase later in the movie. This could mean that the amount of background or darker regions is changing instead of all the voxels increasing in intensity equally. Since P10 represents the darkest 10% of voxels, it can increase if there are fewer background voxels and brighter structures start making up more of that range.

`44b6_0113de3b` shows an increase followed by a decrease in its middle and higher intensity values. However, its **P10** continues to decrease later in the movie.

The `6bba` movies have more stable **P10** values, but their **P50, P90, and P99** still change over time. So even when the darker parts of the image stay relatively consistent, the overall intensity distribution is still changing.

Frame-specific percentile normalization could help reduce some of these intensity changes, but it could also remove meaningful changes in tissue occupancy or contrast. Because of that, I'll need to test whether it actually helps detection performance rather than assuming it does.


### Investigate the Temporal Change in `44b6_0b24845f`

The large change in **P10** for `44b6_0b24845f` is worth looking at more closely. I'm comparing the raw intensity values and fixed-scale projections to see whether the whole image is getting brighter or if the change is mainly coming from the amount of low-intensity background in the image.


In [ ]:
focus_id = "44b6_0b24845f"
focus_timepoints = [0, 20, 40, 60, 80, 95]

display(
    temporal_intensity.loc[
        (temporal_intensity["sample_id"] == focus_id)
        & temporal_intensity["t"].isin(focus_timepoints),
        [
            "t",
            "intensity_p10",
            "intensity_p50",
            "intensity_p90",
            "intensity_p99"
        ]
    ].reset_index(drop=True)
)

### Fixed-Scale Mean Projections Through Time

I'm showing the mean-intensity projections using the same intensity scale across all of the selected time points. Unlike using a separate scale for each frame, this keeps the actual brightness differences visible over time and makes it easier to see changes in the background and how much of the image is occupied by brighter structures.


In [ ]:
focus_array = zarr.open(
    str(test_root / f"{focus_id}.zarr"),
    mode="r"
)["0"]

mean_projections = []

for t in focus_timepoints:
    frame = np.asarray(focus_array[t])
    mean_projections.append(
        frame.mean(axis=0)
    )

mean_projection_stack = np.stack(
    mean_projections
)

projection_vmin = np.quantile(
    mean_projection_stack,
    0.01
)

projection_vmax = np.quantile(
    mean_projection_stack,
    0.99
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(12, 8),
    constrained_layout=True
)

for ax, t, projection in zip(
    axes.flat,
    focus_timepoints,
    mean_projections
):
    image = ax.imshow(
        projection,
        cmap="gray",
        origin="lower",
        vmin=projection_vmin,
        vmax=projection_vmax
    )

    ax.set_title(f"t = {t}")
    ax.axis("off")

fig.colorbar(
    image,
    ax=axes,
    shrink=0.8,
    label="Mean fluorescence intensity"
)

fig.suptitle(
    f"Fixed-scale mean projections for {focus_id}",
    fontsize=14
)

plt.show()

### Interpretation of the P10 Increase

The fixed-scale mean projections show that the dark regions become smaller at the later timepoints. By timepoint 95, fluorescent tissue takes up most of the projected image, with only a smaller dark region left in the lower-right corner.

This supports the idea that the increase in **P10** is partly because the low-intensity background is making up a smaller part of the volume. Once the background becomes less than around 10% of the image, P10 starts including brighter tissue voxels, which causes the value to increase.

At the same time, the change doesn't seem to be only from tissue occupancy. **P50, P90, and P99** also increase after around timepoint 20, and the fluorescent regions look brighter on the shared scale. So the temporal changes are probably coming from both changes in how much fluorescent tissue is present and changes in the overall intensity.

The mean projections are useful for seeing these changes, but they don't directly tell me how much tissue is actually present in the full 3D volume.

### Section Summary

* Raw fluorescence intensity varies between movies, and the two dataset groups also have different upper-intensity distributions. Because of this, one fixed raw detection threshold probably won't work well across all movies.
* There is still a lot of variation within each group, and the four test movies have very different intensity rankings compared to the other movies in their groups. This suggests that group-specific scaling alone may not be enough.
* Movie-specific normalization makes structures easier to see in dimmer movies, but it can also make background noise more noticeable and doesn't fix intensity changes that happen over time within a movie.
* The changes over time are different across **P10, P50, P90, and P99**, showing that the intensity variation can't always be explained by one simple brightness shift.
* In `44b6_0b24845f`, the large increase in **P10** seems to be caused by both a smaller amount of low-intensity background and an increase in tissue intensity.
* Overall, movie-level and frame-level normalization could help reduce some of the intensity variation, but I'll need to test how much they actually help with detection and tracking.

### Dataset-Level Graph EDA

So far, the graph analysis has mainly focused on one sparse annotation graph with two simple directed paths. Next, I'm extending this analysis to all **199 training graphs**.

The main things I want to look at are:

* Are all of the annotation graphs directed acyclic graphs?
* How many connected components, roots, and leaves are there?
* How often do annotated divisions and merges occur?
* How long are the annotated trajectories?
* Are temporal gaps common between annotations?
* How do the graph structure and movement differ between `44b6` and `6bba`?
* Are the four test movies representative of their respective groups?

Looking at these properties should give me a better idea of how the tracking data is structured and what a baseline tracking method would need to handle.

### Build a Graph-Level Summary Table

I created a summary table for each training graph to look at its overall structure. The table includes information about the graph size, connectivity, and different node roles.

I count a division node when a node has two or more outgoing edges and a merge node when it has two or more incoming edges. Roots and leaves represent the beginning and end of the annotated graph components, but they don't necessarily mean that a cell was actually born or died there since the annotations are sparse.


In [ ]:
test_sample_ids = {path.stem for path in test_zarr}

graph_records = []

for geff_path in train_geff:
    sample_id = geff_path.stem
    store = zarr.open(str(geff_path), mode="r")

    node_ids = store["nodes/ids"][:]
    edge_ids = store["edges/ids"][:]

    graph = nx.DiGraph()
    graph.add_nodes_from(node_ids.tolist())
    graph.add_edges_from(map(tuple, edge_ids.tolist()))

    in_degrees = np.array([degree for _, degree in graph.in_degree()])
    out_degrees = np.array([degree for _, degree in graph.out_degree()])

    component_sizes = sorted(
        (len(component) for component in nx.weakly_connected_components(graph)),
        reverse=True
    )

    graph_records.append({
        "sample_id": sample_id,
        "group": get_group(sample_id),
        "is_test_movie": sample_id in test_sample_ids,
        "n_nodes": graph.number_of_nodes(),
        "n_edges": graph.number_of_edges(),
        "is_dag": nx.is_directed_acyclic_graph(graph),
        "n_self_loops": nx.number_of_selfloops(graph),
        "n_weak_components": len(component_sizes),
        "largest_component_nodes": component_sizes[0],
        "largest_component_fraction": (
            component_sizes[0] / graph.number_of_nodes()
        ),
        "n_roots": int((in_degrees == 0).sum()),
        "n_leaves": int((out_degrees == 0).sum()),
        "n_isolated_nodes": int(
            ((in_degrees == 0) & (out_degrees == 0)).sum()
        ),
        "n_division_nodes": int((out_degrees >= 2).sum()),
        "n_merge_nodes": int((in_degrees >= 2).sum()),
        "maximum_in_degree": int(in_degrees.max()),
        "maximum_out_degree": int(out_degrees.max())
    })

graph_summary = pd.DataFrame(graph_records)

print("Shape:", graph_summary.shape)
display(graph_summary.head())

print("\nMissing values:")
display(graph_summary.isna().sum())

### Initial Graph-Level Check

The graph-level summary table has one row for each of the **199 training movies**, and there aren't any missing values.

The results for `44b6_0113de3b` match what I found earlier, with **52 nodes, 50 edges, two weakly connected components, and no divisions or merges**. This gives me some confidence that the graph extraction is working correctly.

The first few rows already show that the graph structure can be pretty different between movies. In `44b6_0113de3b`, the largest component contains about **94%** of all the nodes, while in `44b6_12dfb391`, the largest component only contains about **14%**. So `44b6_12dfb391` is much more fragmented, with **15 separate annotated components**.

`44b6_12dfb391` is also the first example here with an annotated division. It has **15 roots, 16 leaves, and one node with an out-degree of 2**.

These are just the first few graphs, so they don't tell me how common these patterns are across the full dataset. I'll need to look across all of the training graphs to see how often divisions, merges, and fragmented trajectories actually occur.



### Check Graph Integrity Across All Movies

Before looking at the graph summaries in more detail, I'm checking the raw GEFF arrays to make sure the graph data itself is valid. This includes checking for duplicate node IDs, duplicate edges, and edges that point to nodes that don't exist.

These checks are important because NetworkX can automatically combine duplicate information, which could hide problems in the original GEFF data.

I'm also checking the full dataset for things like directed cycles, self-loops, isolated nodes, and unusually high node degrees. This should give me a better idea of whether there are any structural issues in the annotation graphs before I continue with the analysis.


In [ ]:
integrity_records = []

for geff_path in train_geff:
    sample_id = geff_path.stem
    store = zarr.open(str(geff_path), mode="r")

    node_ids = store["nodes/ids"][:]
    edge_ids = store["edges/ids"][:]

    node_id_set = set(node_ids.tolist())
    source_ids = edge_ids[:, 0]
    target_ids = edge_ids[:, 1]

    integrity_records.append({
        "sample_id": sample_id,
        "duplicate_node_ids": (
            len(node_ids) - len(np.unique(node_ids))
        ),
        "duplicate_edges": (
            len(edge_ids) - len(np.unique(edge_ids, axis=0))
        ),
        "unknown_source_ids": sum(
            source_id not in node_id_set
            for source_id in source_ids
        ),
        "unknown_target_ids": sum(
            target_id not in node_id_set
            for target_id in target_ids
        )
    })

graph_integrity = pd.DataFrame(integrity_records)

structural_integrity_summary = pd.Series({
    "total_duplicate_node_ids":
        graph_integrity["duplicate_node_ids"].sum(),

    "total_duplicate_edges":
        graph_integrity["duplicate_edges"].sum(),
    
    "total_unknown_source_ids":
        graph_integrity["unknown_source_ids"].sum(),

    "total_unknown_target_ids":
        graph_integrity["unknown_target_ids"].sum(),

    "graphs_that_are_DAGs":
        graph_summary["is_dag"].sum(),

    "graphs_with_self_loops":
        (graph_summary["n_self_loops"] > 0).sum(),

    "graphs_with_isolated_nodes":
        (graph_summary["n_isolated_nodes"] > 0).sum(),

    "graphs_with_divisions":
        (graph_summary["n_division_nodes"] > 0).sum(),

    "graphs_with_merges":
        (graph_summary["n_merge_nodes"] > 0).sum(),

    "graphs_with_maximum_out_degree_above_2":
        (graph_summary["maximum_out_degree"] > 2).sum(),

    "graphs_with_maximum_in_degree_above_1":
        (graph_summary["maximum_in_degree"] > 1).sum(),

    "graphs_satisfying_edges_equals_nodes_minus_components":
        (
            graph_summary["n_edges"]
            == graph_summary["n_nodes"]
            - graph_summary["n_weak_components"]
        ).sum(),

    "graphs_satisfying_roots_equals_components":
        (
            graph_summary["n_roots"]
            == graph_summary["n_weak_components"]
        ).sum()
    }, name="value").to_frame()

display(structural_integrity_summary)

### Graph Integrity and Structural Form

I didn't find any duplicate node IDs, duplicate edges, or edges pointing to unknown nodes across the **199 GEFF files**. This means the graph construction isn't losing information from duplicated entries when I create the NetworkX graphs.

All **199 graphs are directed acyclic graphs (DAGs)**, and there are no self-loops, isolated nodes, or merges. So every annotated node is connected through a temporal edge and has at most one parent.

Each graph follows the relationship:

`number of edges = number of nodes − number of weak components`

This means that each weak component forms a tree, and every component has exactly one root node.

The maximum out-degree across the dataset is **2**, so the annotations only show normal temporal continuation and binary divisions. I didn't find any cases where one node splits into more than two daughter nodes.

Overall, the annotation graphs can be described as **directed forests made up of rooted binary trees**. Each component has one root, with a unique path from that root to every other annotated node.

Out of the **199 graphs**, **87 (about 43.7%)** contain at least one annotated division, while the other **112** don't have any annotated divisions. Since the annotations are sparse, though, this doesn't mean that biological divisions didn't happen in those movies. It only means they weren't included in the annotations.


### Compare Graph Topology Between Dataset Groups

Next, I'm comparing the graph structure of the `44b6` and `6bba` groups by looking at things like graph size and how fragmented the annotations are.

Raw counts aren't enough for this comparison because the two groups have different numbers of annotated nodes. For example, a graph could have more components simply because it has more annotations overall.

To make the comparison more consistent, I'm using a few normalized measurements:

* **mean component size:** the average number of annotated nodes in each weak component;
* **components per 100 nodes:** shows how fragmented the graph is while accounting for its size;
* **divisions per 100 nodes:** shows how often annotated divisions occur relative to the number of annotated nodes;
* **largest-component percentage:** shows what percentage of the annotated nodes belong to the largest component.


In [ ]:
graph_summary["has_division"] = (
    graph_summary["n_division_nodes"] > 0
)

graph_summary["mean_component_nodes"] = (
    graph_summary["n_nodes"]
    / graph_summary["n_weak_components"]
)

graph_summary["components_per_100_nodes"] = (
    100
    * graph_summary["n_weak_components"]
    / graph_summary["n_nodes"]
)

graph_summary["divisions_per_100_nodes"] = (
    100
    * graph_summary["n_division_nodes"]
    / graph_summary["n_nodes"]
)

graph_summary["largest_component_percentage"] = (
    100 * graph_summary["largest_component_fraction"]
)

graph_summary["leaf_identity_holds"] = (
    graph_summary["n_leaves"]
    == graph_summary["n_roots"]
    + graph_summary["n_division_nodes"]
)

print(
    "Graphs satisfying leaves = roots + divisions:",
    graph_summary["leaf_identity_holds"].sum(),
    "out of",
    len(graph_summary)
)

topology_by_group = (
    graph_summary
    .groupby("group")
    .agg(
        n_movies=("sample_id", "size"),
        median_nodes=("n_nodes", "median"),
        median_components=("n_weak_components", "median"),
        median_mean_component_nodes=(
            "mean_component_nodes", "median"
        ),
         median_components_per_100_nodes=(
            "components_per_100_nodes", "median"
        ),
        median_largest_component_percentage=(
            "largest_component_percentage", "median"
        ),
        movies_with_divisions=("has_division", "sum"),
        proportion_with_divisions=("has_division", "mean"),
        median_division_nodes=("n_division_nodes", "median"),
        median_divisions_per_100_nodes=(
            "divisions_per_100_nodes", "median"
        )
    )
)

topology_by_group["movies_with_divisions_percentage"] = (
    100 * topology_by_group["proportion_with_divisions"]
)

topology_by_group = topology_by_group.drop(
    columns="proportion_with_divisions"
).round(3)

display(topology_by_group)

In [ ]:
test_graph_columns = [
    "sample_id",
    "group",
    "n_nodes",
    "n_weak_components",
    "mean_component_nodes",
    "largest_component_percentage",
    "n_division_nodes",
    "divisions_per_100_nodes"
]

display(
    graph_summary.loc[
        graph_summary["is_test_movie"],
        test_graph_columns
    ].round(3)
)

### Initial Dataset-Group Topology Findings

All **199 graphs** follow the relationship:

`number of leaves = number of roots + number of division nodes`

This matches the directed binary-forest structure I found earlier. A node with one child just continues the existing branch, while a node with two children creates one additional branch, which results in one additional leaf.

The `6bba` annotations are also noticeably more fragmented than the `44b6` annotations. A typical `6bba` graph has 29

## Compare Annotation Fragmentation Between Groups

I'm comparing two measures to get a better idea of how fragmented the annotations are:

* **components per 100 nodes:** shows how often separate components occur relative to the size of the graph;
* **largest-component percentage:** shows how much of the annotated graph is contained in one main lineage.

In general, more fragmentation should mean more components per 100 nodes and a smaller percentage of nodes in the largest component.

I'm also showing the individual movie values along with the box plots. The red diamonds mark the four test movies so I can see where they fall compared to the rest of their group.


In [ ]:
group_order = ["44b6", "6bba"]

fragmentation_metrics = [
    (
        "components_per_100_nodes",
        "Components per 100 annotated nodes",
        "Number of separate components"
    ),
    (
        "largest_component_percentage",
        "Largest component (% of annotated nodes)",
        "Size of largest component"
    )
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (column, ylabel, title) in zip(
    axes,
    fragmentation_metrics
):
    sns.boxplot(
        data=graph_summary,
        x="group",
        y=column,
        order=group_order,
        hue="group",
        palette=group_palette,
        legend=False,
        showfliers=False,
        ax=ax
    )

    sns.stripplot(
        data=graph_summary,
        x="group",
        y=column,
        order=group_order,
        color="black",
        alpha=0.35,
        size=4,
        jitter=0.22,
        ax=ax
    )

    for x_position, group in enumerate(group_order):
        test_values = (
            graph_summary.loc[
                (graph_summary["group"] == group)
                & graph_summary["is_test_movie"],
                column
            ]
            .sort_values()
            .to_numpy()
        )

        offsets = np.linspace(
            -0.06,
            0.06,
            len(test_values)
        )

        ax.scatter(
            x_position + offsets,
            test_values,
            marker="D",
            s=90,
            color="red",
            edgecolor="black",
            linewidth=1,
            zorder=5
        )

    ax.set_title(title)
    ax.set_xlabel("Dataset group")
    ax.set_ylabel(ylabel)

axes[0].scatter(
    [],
    [],
    marker="D",
    s=90,
    color="red",
    edgecolor="black",
    label="Test movie"
)
axes[0].legend()

plt.tight_layout()
plt.show()

### Group Differences in Annotation Fragmentation

The `6bba` group generally has more weak components per 100 annotated nodes than `44b6`. There is some overlap between individual movies, but the interquartile ranges have little to no overlap. This suggests that the difference is pretty consistent across the groups rather than being caused by just a few outliers.

The difference is even more noticeable when looking at the size of the largest component. The lower quartile for `44b6` is above the upper whisker for `6bba`, although there are a few `6bba` outliers that overlap with the lower part of the `44b6` distribution. Overall, a typical `44b6` graph has a much larger share of its annotations concentrated in one main component.

Both `44b6` test graphs are toward the higher end of their group for both measures. This makes sense because each one has one long main component and one much smaller component. Since they also have relatively few total nodes, having two components results in almost four components per 100 nodes.

For the `6bba` test graphs, one is below the group's interquartile range for components per 100 nodes, while the other is closer to the middle. Both fall within the group's interquartile range for largest-component percentage.



## Relate the Two Fragmentation Measures

Next, I'm using a movie-level scatter plot to see whether graphs with more weak components per 100 nodes also tend to have a smaller percentage of annotations in their largest component.

I'm calculating Spearman correlations for the full dataset and then separately for each group. The within-group correlations are more useful here because the overall relationship could mostly be coming from the differences between `44b6` and `6bba`.


In [ ]:
def spearman_rank_correlation(data, x, y):
    return (
        data[x]
        .rank()
        .corr(data[y].rank())
    )


fragmentation_correlations = pd.Series({
    "overall": spearman_rank_correlation(
        graph_summary,
        "components_per_100_nodes",
        "largest_component_percentage"
    ),
    "44b6": spearman_rank_correlation(
        graph_summary.loc[graph_summary["group"] == "44b6"],
        "components_per_100_nodes",
        "largest_component_percentage"
    ),
    "6bba": spearman_rank_correlation(
        graph_summary.loc[graph_summary["group"] == "6bba"],
        "components_per_100_nodes",
        "largest_component_percentage"
    )
}, name="spearman_correlation")

display(fragmentation_correlations.round(3).to_frame())

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=graph_summary.loc[~graph_summary["is_test_movie"]],
    x="components_per_100_nodes",
    y="largest_component_percentage",
    hue="group",
    hue_order=group_order,
    palette=group_palette,
    alpha=0.7,
    s=45
)

sns.scatterplot(
    data=graph_summary.loc[graph_summary["is_test_movie"]],
    x="components_per_100_nodes",
    y="largest_component_percentage",
    color="red",
    marker="D",
    edgecolor="black",
    linewidth=1,
    s=120,
    label="Test movie",
    zorder=5
)

plt.title("Relationship Between the Two Fragmentation Measures")
plt.xlabel("Weak components per 100 annotated nodes")
plt.ylabel("Nodes in largest weak component (%)")
plt.tight_layout()
plt.show()

### The Fragmentation Measures Are Not Interchangeable

Across all **199 movies**, components per 100 nodes and largest-component percentage have a moderately strong negative Spearman correlation of **−0.646**. So, movies with more separate components generally have a smaller percentage of their annotations concentrated in one component.

The overall relationship is partly coming from the difference between the two groups. Most `44b6` movies are toward the upper-left of the plot, while most `6bba` movies are toward the lower-right.

Within `6bba`, the correlation is still moderately negative at **−0.462**. For `44b6`, though, it's only **−0.148**, which is a weak negative relationship. This means that even with similar values of components per 100 nodes, `44b6` movies can have either one large dominant component or a more even distribution of nodes across their components.

This isn't a sign reversal like the earlier Simpson's-paradox example because all three correlations are still negative. However, combining the groups makes the relationship look much stronger than it is within `44b6`.

So, the two measures are related, but they aren't interchangeable. Components per 100 nodes tells me how many separate annotation fragments there are relative to graph size, while largest-component percentage tells me how concentrated the annotations are within those components.



## Examine Annotated Divisions Across Movies

I'm looking at division structure separately because division detection is explicitly part of the competition metric.

There are two different things I want to distinguish here:

* **prevalence:** how many movies contain at least one annotated division;
* **frequency:** among the movies that have divisions, how many division nodes are actually annotated?

Raw division counts are affected by things like graph size and annotation coverage, so I'm also looking at divisions per 100 annotated nodes. This makes the comparison more consistent, although it still can't completely account for differences in how the annotations were created.


In [ ]:
division_summary_records = []

for group in group_order:
    group_data = graph_summary.loc[
        graph_summary["group"] == group
    ]

    positive_data = group_data.loc[
        group_data["n_division_nodes"] > 0
    ]

    division_summary_records.append({
        "group": group,
        "n_movies": len(group_data),
        "movies_with_divisions": len(positive_data),
        "movies_without_divisions": (
            len(group_data) - len(positive_data)
        ),
        "movies_with_divisions_percentage": (
            100 * len(positive_data) / len(group_data)
        ),
        "total_annotated_divisions": (
            group_data["n_division_nodes"].sum()
        ),
        "median_divisions_all_movies": (
            group_data["n_division_nodes"].median()
        ),
        "median_divisions_when_present": (
            positive_data["n_division_nodes"].median()
        ),
        "maximum_divisions_in_one_movie": (
            group_data["n_division_nodes"].max()
        ),
        "median_divisions_per_100_nodes_when_present": (
            positive_data["divisions_per_100_nodes"].median()
        )
    })

division_summary_by_group = (
    pd.DataFrame(division_summary_records)
    .set_index("group")
    .round(3)
)

display(division_summary_by_group)
    

### Division Prevalence and Frequency

Annotated divisions are more common in `6bba`. About **51.6%** of `6bba` movies have at least one annotated division, compared to **29.6%** of `44b6` movies.

Among the movies that actually have divisions, the typical `6bba` graph has **two division nodes**, while `44b6` usually has **one**. The highest number I found is **five** in a `6bba` graph and **two** in a `44b6` graph.

The normalized result looks different, though. Among movies that contain at least one division, `44b6` has a median of about **0.47 divisions per 100 annotated nodes**, compared to **0.18** for `6bba`.

This doesn't mean that divisions are generally more common in `44b6`. Divisions show up in fewer `44b6` movies, but those annotation graphs are also generally smaller. Because of that, even one division in a small graph can result in a relatively high divisions-per-100-nodes value.

So, there are really two separate things to look at:

* whether a movie has any annotated divisions at all;
* how many divisions it has if it does.

This makes the division counts look more like a two-part or zero-inflated count problem, so I should look at the occurrence of divisions and the number of divisions separately.

I also don't want to compare the raw totals directly because `6bba` has more movies, more annotated nodes, and much higher annotation coverage overall. These are also sparse annotations, so they shouldn't be treated as complete counts of the biological divisions in each movie.



## Visualise Annotated Division Counts

I'm comparing the percentage of movies with each possible number of annotated division nodes between the two groups.

I'm calculating the percentages separately within each group since there are **71 `44b6` movies** and **128 `6bba` movies**. This keeps the larger `6bba` group from dominating the comparison.

In [ ]:
division_count_percentages = (
    pd.crosstab(
        graph_summary["group"],
        graph_summary["n_division_nodes"],
        normalize="index"
    )
    * 100
)

all_division_counts = range(
    graph_summary["n_division_nodes"].max() + 1
)

division_count_percentages = (
    division_count_percentages
    .reindex(
        index=group_order,
        columns=all_division_counts,
        fill_value=0
    )
)

display(division_count_percentages.round(1))

ax = (
    division_count_percentages
    .T
    .plot(
        kind="bar",
        figsize=(10, 6),
        color=[
            group_palette[group]
            for group in group_order
        ],
        width=0.8
    )
)

ax.set_title("Distribution of Annotated Division Counts")
ax.set_xlabel("Annotated division nodes in a movie")
ax.set_ylabel("Movies within dataset group (%)")
ax.legend(title="Dataset group")
ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=0
)

plt.tight_layout()
plt.show()

### Distribution of Annotated Division Counts

Having zero annotated divisions is the most common outcome in both groups, but it's much more common in `44b6`. About **70.4%** of `44b6` movies have no supplied divisions, compared to **48.4%** of `6bba` movies.

The percentage with exactly one division is almost the same between the groups, at **22.5%** for `44b6` and **23.4%** for `6bba`.

The bigger difference is in movies with two or more divisions:

* **7.0%** of `44b6` movies have at least two divisions;
* **28.1%** of `6bba` movies have at least two divisions.

No `44b6` movie has more than two annotated divisions. In comparison, the `6bba` group has a longer tail that goes up to **five divisions**.

Looking only at movies that have at least one division, most positive `44b6` movies have exactly one. For `6bba`, a little more than half of the movies with divisions have multiple division events.

So the main difference between the groups seems to be repeated annotated divisions rather than just whether a single division occurs. However, `6bba` also has larger graphs and much higher annotation coverage, which gives it more opportunities to have multiple divisions included in the annotations. Because of that, I wouldn't interpret this as a biological difference in division rate yet.




## Relate Division Counts to Annotation Graph Size

Larger annotation graphs have more nodes, which means there are also more opportunities for a division event to be included. I'm looking at the relationship between the number of annotated nodes and the number of annotated divisions, both overall and within each dataset group.

I'm using Spearman correlation because division count is a discrete variable with a lot of tied zero values, and I don't expect the relationship with graph size to be perfectly linear.

This analysis is only about the supplied annotations. It doesn't tell me the actual biological probability of an individual cell dividing.


In [ ]:
division_size_correlations = pd.Series({
    "overall": spearman_rank_correlation(
        graph_summary,
        "n_nodes",
        "n_division_nodes"
    ),
    "44b6": spearman_rank_correlation(
        graph_summary.loc[
            graph_summary["group"] == "44b6"
        ],
        "n_nodes",
        "n_division_nodes"
    ),
    "6bba": spearman_rank_correlation(
        graph_summary.loc[
            graph_summary["group"] == "6bba"
        ],
        "n_nodes",
        "n_division_nodes"
    )
}, name="spearman_correlation")

display(division_size_correlations.round(3).to_frame())

In [ ]:
division_plot_data = graph_summary.copy()

random_generator = np.random.default_rng(42)

division_plot_data["division_count_jittered"] = (
    division_plot_data["n_division_nodes"]
    + random_generator.uniform(
        -0.08,
        0.08,
        size=len(division_plot_data)
    )
)

fig, ax = plt.subplots(figsize=(11, 6))

sns.scatterplot(
    data=division_plot_data.loc[
        ~division_plot_data["is_test_movie"]
    ],
    x="n_nodes",
    y="division_count_jittered",
    hue="group",
    hue_order=group_order,
    palette=group_palette,
    alpha=0.65,
    s=45,
    ax=ax
)

test_label_offsets = {
    "44b6_0113de3b": (18, 30),
    "44b6_0b24845f": (18, 10),
    "6bba_05b6850b": (15, 18),
    "6bba_05db0fb1": (15, 12)
}

for group in group_order:
    group_test_data = division_plot_data.loc[
        division_plot_data["is_test_movie"]
        & (division_plot_data["group"] == group)
    ]

    ax.scatter(
        group_test_data["n_nodes"],
        group_test_data["n_division_nodes"],
        marker="D",
        s=130,
        facecolor=group_palette[group],
        edgecolor="red",
        linewidth=2.5,
        label=f"{group} test movie",
        zorder=5
    )

    for row in group_test_data.itertuples():
        short_id = row.sample_id

        ax.annotate(
            short_id,
            xy=(row.n_nodes, row.n_division_nodes),
            xytext=test_label_offsets[row.sample_id],
            textcoords="offset points",
            fontsize=9,
            arrowprops={
                "arrowstyle": "-",
                "color": "black",
                "linewidth": 0.8
            }
        )

ax.set_title("Annotated Division Count versus Graph Size")
ax.set_xlabel("Annotated nodes in movie")
ax.set_ylabel("Annotated division nodes")
ax.set_yticks(
    range(
        graph_summary["n_division_nodes"].max() + 1
    )
)
ax.set_ylim(-0.25, 5.35)
ax.legend(title="Movie type and group")

plt.tight_layout()
plt.show()

### Graph Size Is Weakly Associated with Division Count

The number of annotated nodes has a positive Spearman correlation with the number of division nodes, but the relationship is still pretty weak. The correlation is **0.305 overall**, **0.226 within `44b6`**, and **0.203 within `6bba`**.

So, larger annotation graphs have a slight tendency to contain more division nodes, but graph size only explains a small amount of the variation. There are still plenty of movies with zero divisions across a wide range of graph sizes, including some of the larger `6bba` graphs.

Since the correlations are also positive within both groups, the overall relationship isn't just being caused by the difference between `44b6` and `6bba`. However, these weak relationships still don't explain why `6bba` has so many more movies with multiple annotated divisions.

The two `44b6` test movies look pretty similar here because both have around **50 annotated nodes** and **zero divisions**. The two `6bba` test movies are more different: one has **861 nodes and no divisions**, while the other has **1,229 nodes and three divisions**.



## Build a Component-Level Summary Table

Each weak component represents a separate supplied trajectory or lineage tree. I'm creating a component-level table to look more closely at how large these annotation fragments are and how long they last.

For each component:

* **component size:** the total number of annotated nodes;
* **duration:** the number of timepoints from the first annotation to the last, including both endpoints;
* **observed timepoints:** the number of distinct timepoints that contain a node from that component;
* **simple path:** a component with no division;
* **branched component:** a component with at least one division.

For branched components, the node count represents the total size of the lineage tree rather than the length of one individual trajectory.


In [ ]:
component_records = []

for geff_path in train_geff:
    sample_id = geff_path.stem
    group = get_group(sample_id)

    store = zarr.open(str(geff_path), mode="r")

    node_ids = store["nodes/ids"][:]
    node_times_array = store["nodes/props/t/values"][:]
    edge_ids = store["edges/ids"][:]

    node_times = dict(
        zip(
            node_ids.tolist(),
            node_times_array.tolist()
        )
    )

    graph = nx.DiGraph()
    graph.add_nodes_from(node_ids.tolist())
    graph.add_edges_from(map(tuple, edge_ids.tolist()))

    components = list(
        nx.weakly_connected_components(graph)
    )

    # Give components a reproducible order.
    components = sorted(
        components,
        key=lambda nodes_in_component: (
            min(
                node_times[node_id]
                for node_id in nodes_in_component
            ),
            min(nodes_in_component)
        )
    )

    for component_number, component_nodes in enumerate(
        components,
        start=1
    ):
        component_graph = graph.subgraph(component_nodes)

        component_times = np.array([
            node_times[node_id]
            for node_id in component_nodes
        ])

        in_degrees = np.array([
            degree
            for _, degree in component_graph.in_degree()
        ])

        out_degrees = np.array([
            degree
            for _, degree in component_graph.out_degree()
        ])

        first_t = int(component_times.min())
        last_t = int(component_times.max())

        n_division_nodes = int(
            (out_degrees >= 2).sum()
        )

        component_records.append({
            "sample_id": sample_id,
            "group": group,
            "is_test_movie": sample_id in test_sample_ids,
            "component_id": component_number,
            "n_nodes": component_graph.number_of_nodes(),
            "n_edges": component_graph.number_of_edges(),
            "first_t": first_t,
            "last_t": last_t,
            "duration_timepoints": last_t - first_t + 1,
            "n_observed_timepoints": (
                np.unique(component_times).size
            ),
            "n_roots": int((in_degrees == 0).sum()),
            "n_leaves": int((out_degrees == 0).sum()),
            "n_division_nodes": n_division_nodes,
            "is_branched": n_division_nodes > 0,
            "is_simple_path": n_division_nodes == 0
        })

component_summary = pd.DataFrame(component_records)

### Preview of the Component-Level Table

Each row in this table represents one weak component from one movie. A component can either be a simple directed path or a branching lineage tree.

The `component_id` was created for this analysis and is only unique within its own movie. I also ordered the components by their first annotated timepoint rather than by their size.


In [ ]:
print("Shape:", component_summary.shape)
display(component_summary.head())

print("\nMissing values:")
display(component_summary.isna().sum())

Validate the Component-Level Table
Component counts, nodes, and edges are summed back to the movie level to verify that the component table reconstructs every original annotation graph.

In [ ]:
component_reconstruction = (
    component_summary
    .groupby("sample_id")
    .agg(
        calculated_components=("component_id", "size"),
        calculated_nodes=("n_nodes", "sum"),
        calculated_edges=("n_edges", "sum")
    )
    .reset_index()
    .merge(
        graph_summary[
            [
                "sample_id",
                "n_weak_components",
                "n_nodes",
                "n_edges"
            ]
        ],
        on="sample_id",
        how="left"
    )
)

print(
    "Correct component counts:",
    (
        component_reconstruction["calculated_components"]
        == component_reconstruction["n_weak_components"]
    ).sum(),
    "out of 199"
)

print(
    "Correct node totals:",
    (
        component_reconstruction["calculated_nodes"]
        == component_reconstruction["n_nodes"]
    ).sum(),
    "out of 199"
)

print(
    "Correct edge totals:",
    (
        component_reconstruction["calculated_edges"]
        == component_reconstruction["n_edges"]
    ).sum(),
    "out of 199"
)

print(
    "Components satisfying edges = nodes - 1:",
    (
        component_summary["n_edges"]
        == component_summary["n_nodes"] - 1
    ).sum(),
    "out of",
    len(component_summary)
)

print(
    "Components satisfying leaves = 1 + divisions:",
    (
        component_summary["n_leaves"]
        == 1 + component_summary["n_division_nodes"]
    ).sum(),
    "out of",
    len(component_summary)
)

### Component-Table Validation

Across the **199 annotation graphs**, there are **4,435 weak components**. Each row in the component-level table represents one separate supplied trajectory or branching lineage tree.

The component-level results match the graph-level table for all **199 movies**. The component counts, node counts, and edge counts all reconstruct the original graph-level totals exactly.

Every component has exactly one fewer edge than nodes, which confirms that each component forms a tree. They also all follow:

`number of leaves = 1 + number of division nodes`

This makes sense because each component has one root and only contains binary divisions.

The five components shown here are all simple paths, with one annotated node at each timepoint. However, branching components definitely exist across the dataset since **87 movie graphs** contain at least one annotated division.

Now that the component table is validated, I can look across the full dataset to see how common branching components actually are, how their sizes and durations compare to simple paths, and whether temporal gaps occur in either type.

## Compare Simple Paths and Branching Components

I'm separating the components into two groups:

* **simple paths:** components with no annotated divisions;
* **branching components:** components with at least one annotated division.

I'll look at branching prevalence at both the component level and the movie level. These measure different things because one movie can contain multiple separate components.

### Prevalence of Branching Components

This table shows how many individual components contain at least one annotated division. This is different from the earlier movie-level result, which looked at how many movies had at least one division.


In [ ]:
component_summary["component_type"] = np.where(
    component_summary["is_branched"],
    "Branching component",
    "Simple path"
)

branching_summary_records = []

for group in group_order:
    group_components = component_summary.loc[
        component_summary["group"] == group
    ]

    branched_components = group_components.loc[
        group_components["is_branched"]
    ]

    branching_summary_records.append({
        "group": group,
        "n_components": len(group_components),
        "simple_paths": (
            (~group_components["is_branched"]).sum()
        ),
        "branching_components": len(branched_components),
        "branching_components_percentage": (
            100
            * len(branched_components)
            / len(group_components)
        ),
        "movies_with_a_branching_component": (
            branched_components["sample_id"].nunique()
        )
    })

branching_summary_by_group = (
    pd.DataFrame(branching_summary_records)
    .set_index("group")
    .round(3)
)

display(branching_summary_by_group) 

### Component Size and Duration by Type

This table compares node counts and temporal durations between simple paths and branching components.

Quartiles are reported because component sizes and durations may have long upper tails. A branching component can contain multiple nodes at the same timepoint, so its node count may exceed its duration.

In [ ]:
def first_quartile(values):
    return values.quantile(0.25)


def third_quartile(values):
    return values.quantile(0.75)


component_size_duration_summary = (
    component_summary
    .groupby(
        ["group", "component_type"],
        observed=True
    )
    .agg(
        n_components=("component_id", "size"),
        n_movies=("sample_id", "nunique"),

        minimum_nodes=("n_nodes", "min"),
        first_quartile_nodes=(
            "n_nodes", first_quartile
        ),
        median_nodes=("n_nodes", "median"),
        third_quartile_nodes=(
            "n_nodes", third_quartile
        ),
        maximum_nodes=("n_nodes", "max"),

        minimum_duration=("duration_timepoints", "min"),
        first_quartile_duration=(
            "duration_timepoints", first_quartile
        ),
        median_duration=(
            "duration_timepoints", "median"
        ),
        third_quartile_duration=(
            "duration_timepoints", third_quartile
        ),
        maximum_duration=(
            "duration_timepoints", "max"
        )
    )
    .round(1)
)

display(component_size_duration_summary)

### Branching Components Are Rare but Generally Long

Most of the supplied components are simple paths. Only **7.0%** of `44b6` components and **3.1%** of `6bba` components contain a division.

This doesn't contradict the earlier finding that divisions occur in a larger percentage of `6bba` movies. The `6bba` graphs have many more components overall, and most of those are simple paths. So divisions occur across more `6bba` movies, but branching components make up a smaller percentage of the much larger number of total components.

Branching components are generally larger and last longer than simple paths. In `44b6`, branching components have a median of **118 nodes** and a duration of **97.5 timepoints**, compared to **48 nodes** and **48 timepoints** for simple paths. In `6bba`, the medians are **44 nodes** and **36 timepoints** for branching components, compared to **20 nodes** and **20 timepoints** for simple paths.

Components also tend to be longer in `44b6` than in `6bba`, regardless of component type. This matches what I saw earlier, where `44b6` annotations are more concentrated in fewer, larger components, while `6bba` annotations are spread across more shorter components.

The number of branching components exactly matches the number of division nodes in both groups: **26 in `44b6`** and **125 in `6bba`**. This means every branching component contains exactly one annotated division. I didn't find any component with a second division farther along either daughter branch. So when a movie has multiple annotated divisions, those divisions are in separate weak components rather than occurring successively within the same supplied lineage tree.

Since the annotations are sparse, this doesn't mean the actual biological lineages only divide once. Additional divisions could simply be missing from the supplied annotations.

## Visualise Component Durations

I'm using empirical cumulative distribution functions (ECDFs) to compare component durations without having to rely on equal group sizes or histogram bins.

At any point on the horizontal axis, the vertical value shows the proportion of components with that duration or shorter. A curve that rises earlier means there are more shorter components, while a curve shifted further to the right represents generally longer components.

I'm showing simple paths and branching components separately because their duration distributions and sample sizes differ.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

component_type_order = [
    "Simple path",
    "Branching component"
]

plot_titles = {
    "Simple path": "Duration of Simple-Path Components",
    "Branching component": "Duration of Branching Components"
}

for ax, component_type in zip(
    axes,
    component_type_order
):
    plot_data = component_summary.loc[
        component_summary["component_type"]
        == component_type
    ]

    sns.ecdfplot(
        data=plot_data,
        x="duration_timepoints",
        hue="group",
        hue_order=group_order,
        palette=group_palette,
        linewidth=2,
        ax=ax
    )

    ax.set_title(plot_titles[component_type])
    ax.set_xlabel("Component duration (timepoints)")
    ax.set_ylabel("Cumulative proportion of components")
    ax.set_xlim(0, 102)
    ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.show()

### Component-Duration Distributions

For simple paths, the two groups cross at around **10–12 timepoints**. `44b6` has a slightly larger proportion of extremely short paths, but after this point the `6bba` curve rises much faster. So, most `6bba` paths are relatively short, while `44b6` has a much wider range of durations.

The large jump in the `44b6` curve at **100** shows that a substantial number of its simple paths span the entire 100-timepoint movie. Only a small portion of `6bba` paths reaches that same duration.

The difference is even more noticeable for branching components. Most `6bba` branching components have durations below around **60 timepoints**, while the `44b6` branching components are mostly much longer.

About half of the `44b6` branching components have a duration of exactly **100 timepoints**, which creates the large final jump in the curve. The bigger steps are also partly because there are only **26 branching components** in total.

A duration of 100 means that a component has annotations at both timepoint **0 and timepoint 99**. It doesn't necessarily mean that every timepoint in between has an annotation, so I'll check temporal continuity separately.



## Build an Edge-Level Summary Table

Each row in the edge-level table represents one supplied directed connection between two annotated nodes.

I'm adding the source and target node information to each edge so I can calculate things like temporal gaps and physical movement. This table doesn't create any new connections; it only describes the edges that are already provided in the GEFF files.


In [ ]:
edge_frames = []

for geff_path in train_geff:
    sample_id = geff_path.stem
    group = get_group(sample_id)

    store = zarr.open(str(geff_path), mode="r")

    node_table = pd.DataFrame({
        "node_id": store["nodes/ids"][:],
        "t": store["nodes/props/t/values"][:],
        "z": store["nodes/props/z/values"][:],
        "y": store["nodes/props/y/values"][:],
        "x": store["nodes/props/x/values"][:]
    })

    edge_table = pd.DataFrame(
        store["edges/ids"][:],
        columns=["source_id", "target_id"]
    )

    source_table = node_table.rename(columns={
        "node_id": "source_id",
        "t": "source_t",
        "z": "source_z",
        "y": "source_y",
        "x": "source_x"
    })

    target_table = node_table.rename(columns={
        "node_id": "target_id",
        "t": "target_t",
        "z": "target_z",
        "y": "target_y",
        "x": "target_x"
    })

    edge_table = (
        edge_table
        .merge(
            source_table,
            on="source_id",
            how="left",
            validate="many_to_one"
        )
        .merge(
            target_table,
            on="target_id",
            how="left",
            validate="many_to_one"
        )
    )

    edge_table.insert(0, "sample_id", sample_id)
    edge_table.insert(1, "group", group)
    edge_table.insert(
        2,
        "is_test_movie",
        sample_id in test_sample_ids
    )

    edge_frames.append(edge_table)

edge_summary = pd.concat(
    edge_frames,
    ignore_index=True
)

edge_summary["delta_t"] = (
    edge_summary["target_t"]
    - edge_summary["source_t"]
)

## Preview of the Edge-Level Table

Each row contains one supplied edge, followed by the time and spatial coordinates of its source and target nodes. delta_t is the number of time points between the two connected annotations

In [ ]:
print("Shape:", edge_summary.shape)
display(edge_summary.head())

print("\nMissing values:")
display(edge_summary.isna().sum())

print(
    "\nEdge total agrees with graph table:",
    len(edge_summary)
    == graph_summary["n_edges"].sum()
)


## Examine Temporal Gaps Across All Edges

For each directed edge, I calculate the time difference as:

`delta_t = target timepoint − source timepoint`

A value of **1** means the edge connects consecutive timepoints. A value greater than **1** means one or more timepoints were skipped. A value of **0** would mean the two nodes are in the same frame, while a negative value would mean the edge goes backward in time.

Looking at the edge-level gaps gives me a better way to check temporal continuity than just looking at component duration. A component can span a long period of time while still having gaps between some of its annotations.


In [ ]:
temporal_edge_summary = (
    edge_summary
    .groupby("group")
    .agg(
        n_edges=("delta_t", "size"),
        forward_edges=(
            "delta_t",
            lambda values: (values > 0).sum()
        ),
        consecutive_edges=(
            "delta_t",
            lambda values: (values == 1).sum()
        ),
        temporal_gap_edges=(
            "delta_t",
            lambda values: (values > 1).sum()
        ),
        same_time_edges=(
            "delta_t",
            lambda values: (values == 0).sum()
        ),
        backward_edges=(
            "delta_t",
            lambda values: (values < 0).sum()
        ),
        median_delta_t=("delta_t", "median"),
        maximum_delta_t=("delta_t", "max")
    )
)

temporal_edge_summary["temporal_gap_percentage"] = (
    100
    * temporal_edge_summary["temporal_gap_edges"]
    / temporal_edge_summary["n_edges"]
)

display(temporal_edge_summary.round(3))

print("\nOverall delta_t distribution:")
display(
    edge_summary["delta_t"]
    .value_counts()
    .sort_index()
    .rename_axis("delta_t")
    .to_frame("n_edges")
)

### All Supplied Edges Connect Consecutive Timepoints

The edge-level table contains all **128,883 supplied graph edges**, with no missing source or target information. The total number of edges also matches the graph-level table exactly.

Every edge moves forward by exactly **one timepoint**. There are no same-frame edges, backward edges, or edges that skip frames in either dataset group.

So, there aren't any temporal gaps within the supplied annotation components. A component can start or end partway through the movie, but once it appears in the annotations, each connection goes from one frame directly to the next.

This also means that every component has at least one annotated node at every timepoint between its first and last annotation. For a simple path, the **node count equals the duration**. A branching component can have more nodes than its duration because both daughter branches can be present at the same timepoint.

Based on the ground-truth structure, it makes sense for an initial tracking baseline to focus on links between consecutive frames. I wouldn't introduce longer predicted links without validating them first since they aren't part of the supplied edge structure.

## Measure Physical Movement Across All Edges

I'm calculating physical movement using the difference between the source and target coordinates.

Before calculating the Euclidean distance, I convert the voxel differences into **micrometres** because the Z-axis spacing is about four times larger than the X and Y spacing.

I'm separating the edges into two groups:

* **continuation edges:** the source has one outgoing edge, connecting two consecutive observations of the same biological cell;
* **division edges:** the source has two outgoing edges, connecting the dividing cell to its two annotated daughter cells.

Each division node therefore contributes **two division edges**, one for each daughter.


In [ ]:
edge_summary["source_out_degree"] = (
    edge_summary
    .groupby(["sample_id", "source_id"])["target_id"]
    .transform("size")
)

edge_summary["edge_type"] = np.where(
    edge_summary["source_out_degree"] == 2,
    "Division edge",
    "Continuation edge"
)

edge_summary["delta_z_voxels"] = (
    edge_summary["target_z"]
    - edge_summary["source_z"]
)

edge_summary["delta_y_voxels"] = (
    edge_summary["target_y"]
    - edge_summary["source_y"]
)

edge_summary["delta_x_voxels"] = (
    edge_summary["target_x"]
    - edge_summary["source_x"]
)

edge_summary["delta_z_um"] = (
    edge_summary["delta_z_voxels"] * 1.625
)

edge_summary["delta_y_um"] = (
    edge_summary["delta_y_voxels"] * 0.40625
)

edge_summary["delta_x_um"] = (
    edge_summary["delta_x_voxels"] * 0.40625
)

edge_summary["distance_um"] = np.sqrt(
    edge_summary["delta_z_um"] ** 2
    + edge_summary["delta_y_um"] ** 2
    + edge_summary["delta_x_um"] ** 2
)

edge_summary["speed_um_per_second"] = (
    edge_summary["distance_um"]
    / edge_summary["delta_t"]
)

## Physical-Displacement Summary

Displacement distributions are summarised separately by dataset group and edge type. Quantiles are more informative than the maximum alone because a small number of unusual edges may form a long upper tail.

In [ ]:
def ninety_fifth_percentile(values):
    return values.quantile(0.95)


def ninety_ninth_percentile(values):
    return values.quantile(0.99)


movement_summary = (
    edge_summary
    .groupby(
        ["group", "edge_type"],
        observed=True
    )
    .agg(
        n_edges=("distance_um", "size"),
        minimum_distance=("distance_um", "min"),
        first_quartile_distance=(
            "distance_um", first_quartile
        ),
        median_distance=("distance_um", "median"),
        mean_distance=("distance_um", "mean"),
        third_quartile_distance=(
            "distance_um", third_quartile
        ),
        percentile_95_distance=(
            "distance_um", ninety_fifth_percentile
        ),
        percentile_99_distance=(
            "distance_um", ninety_ninth_percentile
        ),
        maximum_distance=("distance_um", "max")
    )
    .round(3)
)

display(movement_summary)

### Movement Differs Between Continuations and Divisions

Normal continuation movement is relatively small. The median displacement is about **1.72 µm** in `44b6` and **1.82 µm** in `6bba`.

Division edges are noticeably longer. Their median displacement is about **4.61 µm** in `44b6` and **5.93 µm** in `6bba`. The interquartile ranges for continuation and division edges don't overlap within either group.

This makes sense because a division edge is connecting the parent centroid to one of the two daughter centroids, so the apparent movement is larger than normal frame-to-frame movement. Because of this, a tracking method may need different spatial expectations for continuation and division links.

Movement also tends to be somewhat larger in `6bba`, especially toward the higher end of the distributions. For continuation edges, the 95th percentile is about **5.47 µm** in `6bba`, compared to **3.92 µm** in `44b6`.

Some continuation edges have zero displacement. This just means the integer centroid coordinates stayed the same between two consecutive frames. The cell could be nearly stationary, or the actual movement could be smaller than the coordinate resolution.

Each division node creates two daughter edges. The **52 `44b6` division edges** come from **26 division nodes**, while the **250 `6bba` division edges** come from **125 division nodes**.

The maximum `6bba` continuation displacement is **60.76 µm**, which is much higher than its 99th percentile of **8.46 µm**. I'll need to look at this edge individually before deciding whether it's real movement, an unusual annotation, or some other edge case.

Also, the **7 µm evaluation tolerance** is for matching predicted cell positions to the ground-truth positions. It isn't a maximum movement distance, so I shouldn't use it directly as a linking threshold.



## Inspect the Longest Supplied Edges

I'm looking at the longest edges individually because the maximum displacement can be heavily affected by just a few unusual observations.

I'm keeping the source and target coordinates so I can later check these extreme movements against the actual microscopy images and see what's going on.


In [ ]:
longest_edge_columns = [
    "sample_id",
    "group",
    "edge_type",
    "source_id",
    "target_id",
    "source_t",
    "target_t",
    "source_z",
    "source_y",
    "source_x",
    "target_z",
    "target_y",
    "target_x",
    "delta_z_um",
    "delta_y_um",
    "delta_x_um",
    "distance_um"
]

longest_edges = (
    edge_summary
    .nlargest(15, "distance_um")
    [longest_edge_columns]
    .reset_index(drop=True)
)

display(longest_edges.round(3))

In [ ]:
def percentage_above(values, threshold):
    return 100 * (values > threshold).mean()


long_edge_summary = (
    edge_summary
    .groupby(
        ["group", "edge_type"],
        observed=True
    )
    .agg(
        n_edges=("distance_um", "size"),
        percentage_above_10_um=(
            "distance_um",
            lambda values: percentage_above(values, 10)
        ),
        percentage_above_15_um=(
            "distance_um",
            lambda values: percentage_above(values, 15)
        ),
        percentage_above_20_um=(
            "distance_um",
            lambda values: percentage_above(values, 20)
        )
    )
    .round(3)
)

display(long_edge_summary)

### Longest Edges Cluster by Movie and Time

The **eight longest supplied edges** all come from `6bba_f20478e9` and happen around the transitions from **timepoint 64 → 65** and **65 → 66**.

Most of these large movements are mainly coming from the **Z-axis**. Some nodes move around **50–60 µm** in one direction, and then other edges show similarly large movements in the opposite direction in the next transition.

Since these extreme movements are all concentrated in one movie and around two consecutive transitions, this looks more like a **frame-level or annotation-level issue** than normal cell movement. It could be caused by an abrupt shift between image volumes, unusual tracking associations, or potentially real movement. The edge table alone isn't enough to tell which one it is, so I'll need to look at those specific timepoints in the actual images.

The next three longest edges also come from the same movie, `6bba_e5e44988`, during the **14 → 15** transition. These movements are mainly caused by large changes along the **Y-axis**.

Overall, very long edges are rare. Only about **0.016%** of `6bba` continuation edges are longer than **20 µm**, and none of the `44b6` continuation edges reach that distance. No division edge is longer than **15 µm** in either group.

Division edges are more likely than continuation edges to be above **10 µm**, but the most extreme movements are still all continuation edges.

Because of this, I wouldn't choose a tracking search radius based on the maximum displacement of **60.76 µm**. That would allow way too many unrelated cells to become candidates during normal frames. The search radius should instead be based on the typical movement distribution and tuned through validation, while treating these unusual transitions separately.



## Investigate the Extreme Movement Cluster

I'm looking more closely at `6bba_f20478e9` by calculating the movement for each consecutive-frame transition. This should show whether the extreme distances are mainly limited to **timepoints 64–66** or if the unusual movement continues for a longer part of the movie.

For source timepoint **64**, the plotted distances represent edges going from **timepoint 64 to 65**.


In [ ]:
movement_focus_id = "6bba_f20478e9"

focus_movement_edges = edge_summary.loc[
    edge_summary["sample_id"] == movement_focus_id
].copy()

focus_transition_summary = (
    focus_movement_edges
    .groupby("source_t")
    .agg(
        n_edges=("distance_um", "size"),
        median_distance=("distance_um", "median"),
        percentile_95_distance=(
            "distance_um",
            ninety_fifth_percentile
        ),
        maximum_distance=("distance_um", "max")
    )
    .reset_index()
)

print("Transitions with a maximum distance above 15 µm:")

display(
    focus_transition_summary.loc[
        focus_transition_summary["maximum_distance"] > 15
        ].round(3)
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    focus_transition_summary["source_t"],
    focus_transition_summary["median_distance"],
    marker="o",
    markersize=3,
    linewidth=1.5,
    label="Median"
)

ax.plot(
    focus_transition_summary["source_t"],
    focus_transition_summary["percentile_95_distance"],
    marker="o",
    markersize=3,
    linewidth=1.5,
    label="95th percentile"
)

ax.plot(
    focus_transition_summary["source_t"],
    focus_transition_summary["maximum_distance"],
    marker="o",
    markersize=3,
    linewidth=1.5,
    label="Maximum"
)

ax.axvspan(
    64,
    66,
    color="red",
    alpha=0.12,
    label="Extreme transitions"
)

ax.set_title(
    "Edge Displacement Through Time in "
    "6bba_f20478e9"
)
ax.set_xlabel(
    "Source timepoint of consecutive-frame edge"
)
ax.set_ylabel("Physical displacement (µm)")
ax.legend()

plt.tight_layout()
plt.show()

### Extreme Movement Is Localised Around Timepoint 65

The extreme movement is basically limited to the transitions **into and out of timepoint 65**.

From **64 → 65**, all four supplied edges are unusually long. The median displacement is **51.63 µm**, with the largest being **60.76 µm**.

The same thing happens from **65 → 66**, where the median displacement is **57.74 µm** and the maximum is **60.17 µm**. Outside of these two transitions, the maximum displacement is generally below around **12 µm**.

Since the median, 95th percentile, and maximum all increase at the same time, this doesn't look like just one incorrectly linked edge. Every supplied trajectory that passes through timepoint 65 shows a large displacement.

The movement into timepoint 65 is also mostly reversed immediately afterward. This makes it look like **timepoint 65 itself is spatially out of place compared to the frames around it**.

The edge data alone can't tell me exactly why this happens. It could be that the microscopy volume shifted, the coordinate system changed, or there is something unusual about the supplied annotations.





## Inspect the Microscopy Around Timepoint 65

I'm going to look at the actual microscopy images around timepoint 65 to see what could be causing the extreme movement.

I'll compare **maximum-intensity XZ and YZ projections** for timepoints **64, 65, and 66** using the same intensity scale, with the annotated nodes overlaid in red.

I'm mainly checking:

* whether the fluorescent structures shift along **Z** at timepoint 65;
* whether the annotations are still lined up with the bright structures;
* whether the image shifts back toward its previous position at timepoint 66;
* whether timepoint 65 looks visually different or potentially corrupted.

If the **image and annotations shift together**, then the large movement could be caused by the specimen or imaging volume shifting. If only the annotations move while the image stays in place, then an annotation or tracking issue would be more likely.


In [ ]:
movement_focus_times = [64, 65, 66]

focus_image_array = zarr.open(
    str(train_root / f"{movement_focus_id}.zarr"),
    mode="r"
)["0"]

focus_geff_store = zarr.open(
    str(train_root / f"{movement_focus_id}.geff"),
    mode="r"
)

focus_nodes = pd.DataFrame({
    "node_id": focus_geff_store["nodes/ids"][:],
    "t": focus_geff_store["nodes/props/t/values"][:],
    "z": focus_geff_store["nodes/props/z/values"][:],
    "y": focus_geff_store["nodes/props/y/values"][:],
    "x": focus_geff_store["nodes/props/x/values"][:]
})

focus_projections = {}

for timepoint in movement_focus_times:
    frame = focus_image_array[timepoint]

    focus_projections[timepoint] = {
        "xz": frame.max(axis=1),
        "yz": frame.max(axis=2)
    }

all_projection_values = np.concatenate([
    projection.ravel()
    for timepoint_data in focus_projections.values()
    for projection in timepoint_data.values()
])

projection_vmin = np.percentile(
    all_projection_values,
    1
)

projection_vmax = np.percentile(
    all_projection_values,
    99.5
)

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9),
    constrained_layout=True
)

x_extent = [0, 256 * 0.40625]
y_extent = [0, 256 * 0.40625]
z_extent = [0, 64 * 1.625]

for column, timepoint in enumerate(
    movement_focus_times
):
    nodes_at_timepoint = focus_nodes.loc[
        focus_nodes["t"] == timepoint
    ]

    xz_image = axes[0, column].imshow(
        focus_projections[timepoint]["xz"],
        cmap="gray",
        origin="lower",
        extent=x_extent + z_extent,
        aspect="auto",
        vmin=projection_vmin,
        vmax=projection_vmax
    )

    axes[0, column].scatter(
        nodes_at_timepoint["x"] * 0.40625,
        nodes_at_timepoint["z"] * 1.625,
        facecolors="none",
        edgecolors="red",
        linewidths=1.5,
        s=70
    )

    axes[0, column].set_title(
        f"XZ projection, t = {timepoint}"
    )
    axes[0, column].set_xlabel("X position (µm)")
    axes[0, column].set_ylabel("Z position (µm)")

    axes[1, column].imshow(
        focus_projections[timepoint]["yz"],
        cmap="gray",
        origin="lower",
        extent=y_extent + z_extent,
        aspect="auto",
        vmin=projection_vmin,
        vmax=projection_vmax
    )

    axes[1, column].scatter(
        nodes_at_timepoint["y"] * 0.40625,
        nodes_at_timepoint["z"] * 1.625,
        facecolors="none",
        edgecolors="red",
        linewidths=1.5,
        s=70
    )

    axes[1, column].set_title(
        f"YZ projection, t = {timepoint}"
    )
    axes[1, column].set_xlabel("Y position (µm)")
    axes[1, column].set_ylabel("Z position (µm)")

fig.colorbar(
    xz_image,
    ax=axes,
    shrink=0.8,
    label="Fluorescence intensity"
)

fig.suptitle(
    "Microscopy and Annotations Around the Extreme Movement Event",
    fontsize=14
)

plt.show()

### Image Evidence for a Global Spatial Shift

The microscopy images make it pretty clear that the extreme edge movements are connected to a **global shift in the image position at timepoint 65**.

At timepoints **64 and 66**, the fluorescent structures fill most of the Z-range. At **timepoint 65**, a large part of the lower volume is empty, while the fluorescent region has shifted toward higher Z and is partly cut off by the top boundary.

The annotated nodes move along with the visible fluorescent structures and still line up with them. Because of this, the large movements don't look like four individual cells suddenly moving huge distances.

Instead, the image itself appears to shift at timepoint 65 and then move back toward its previous position at timepoint 66. From these images alone, I can't tell whether this was caused by the specimen moving, the microscope stage shifting, or a change in the coordinate system during acquisition.

The maximum-intensity projections do collapse one spatial dimension, so they can't prove that this was a perfectly rigid 3D shift. However, the same general Z-direction shift is visible in both the **XZ and YZ views**, which supports the idea of a global spatial change.



### Modelling Implication: Global Motion Compensation

The shift around timepoint 65 suggests that some of the movement I'm seeing may be coming from the **entire image volume moving**, rather than from individual cells.

One possible approach for a future tracking pipeline would be to estimate the global 3D translation between consecutive frames using image registration. Then, candidate cell links could be compared using both:

* the raw displacement between detected centroids;
* the residual displacement after accounting for the estimated global movement.

This could help the tracker handle sudden shifts without needing an unnecessarily large search radius during normal frames.

I wouldn't automatically apply global registration to every frame, though. The tissue can deform, cells can enter or leave the field of view, and correcting for global movement could potentially remove real biological movement. Because of that, the estimated frame shift may be more useful as an **additional feature for linking** rather than something that always has to be subtracted from the coordinates.





## Visualise Typical Edge Displacements

I'm using empirical cumulative distribution functions to compare physical displacement between the two dataset groups.

Continuation and division edges are shown separately because division edges have noticeably larger typical movements.

I'm restricting the x-axis to **0–15 µm**. This includes every supplied division edge and more than **99.9% of continuation edges**, while keeping the rare global-shift events from making the normal movement patterns harder to see.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

edge_type_order = [
    "Continuation edge",
    "Division edge"
]

edge_type_titles = {
    "Continuation edge":
        "Continuation-Edge Displacement",

    "Division edge":
        "Displacement Along Division Edges"
}

for ax, edge_type in zip(
    axes,
    edge_type_order
):
    plot_data = edge_summary.loc[
        edge_summary["edge_type"] == edge_type
    ]

    sns.ecdfplot(
        data=plot_data,
        x="distance_um",
        hue="group",
        hue_order=group_order,
        palette=group_palette,
        linewidth=2,
        ax=ax
    )

    ax.set_title(edge_type_titles[edge_type])
    ax.set_xlabel("Physical displacement (µm)")
    ax.set_ylabel("Cumulative proportion of edges")
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.show()

### Edge-Displacement Distributions

The `6bba` displacement distributions are generally shifted toward larger values for both types of edges.

For continuation edges, the difference is pretty small around the middle of the distribution. The median displacement is **1.82 µm** for `6bba` compared to **1.72 µm** for `44b6`. The difference becomes more noticeable toward the higher end, which also matches what I saw earlier with the 95th percentiles.

The difference is clearer for division edges. The `6bba` curve is shifted to the right across most of the distribution, with a median displacement of **5.93 µm** compared to **4.61 µm** for `44b6`.

The larger steps in the `44b6` division curve are mostly because there are only **52 division edges**, along with the fact that the coordinates are based on discrete voxel positions.

There is still a lot of overlap between the distributions, so I can't determine the correct link based only on the dataset group or whether an edge is a continuation or division. However, these movement patterns could still be useful as features in a probabilistic tracking model or when choosing different candidate distances for different edge types or groups.



## Compare Test-Movie Movement with Group Distributions

I'm summarising movement within each movie first so that every movie contributes the same amount to the comparison, regardless of how many supplied edges it has.

For each test movie, I'll look at its **median and 95th-percentile movement** and compare those values with the movies in the same dataset group and with the same edge type.


In [ ]:
movie_movement_summary = (
    edge_summary
    .groupby(
        [
            "sample_id",
            "group",
            "is_test_movie",
            "edge_type"
        ],
        observed=True
    )
    .agg(
        n_edges=("distance_um", "size"),
        median_distance=("distance_um", "median"),
        percentile_95_distance=(
            "distance_um",
            ninety_fifth_percentile
        ),
        maximum_distance=("distance_um", "max")
    )
    .reset_index()
)

movie_movement_summary[
    "median_distance_group_percentile"
] = (
    movie_movement_summary
    .groupby(["group", "edge_type"])[
        "median_distance"
    ]
    .rank(pct=True)
    * 100
)

movie_movement_summary[
    "percentile_95_group_percentile"
] = (
    movie_movement_summary
    .groupby(["group", "edge_type"])[
        "percentile_95_distance"
    ]
    .rank(pct=True)
    * 100
)

test_movement_summary = (
    movie_movement_summary.loc[
        movie_movement_summary["is_test_movie"]
    ]
    .sort_values(["group", "sample_id", "edge_type"])
)

display(
    test_movement_summary[
        [
            "sample_id",
            "group",
            "edge_type",
            "n_edges",
            "median_distance",
            "median_distance_group_percentile",
            "percentile_95_distance",
            "percentile_95_group_percentile",
            "maximum_distance"
        ]
    ].round(3)
)

### Movement Characteristics of the Test Movies

The two `44b6` test movies have relatively large continuation movements compared to the rest of their group.

`44b6_0113de3b` stands out the most. Its median displacement is around the **90th percentile** of the `44b6` movies, while its 95th percentile is around the **87th percentile**. So, the movie I used for the initial detailed analysis isn't really typical of `44b6` when it comes to movement.

`44b6_0b24845f` is also on the higher side, with both its median and 95th-percentile displacement around the **70th percentile** of the group.

The two `6bba` test movies are much more different from each other. `6bba_05b6850b` has one of the smallest continuation-movement distributions in its group. Its median is around the **2nd percentile**, and its 95th percentile is around the **10th percentile**.

`6bba_05db0fb1` is much closer to typical `6bba` movement, with its median around the **38th percentile** and its 95th percentile around the **54th percentile**.

Only `6bba_05db0fb1` has supplied division edges. Its **six division edges** come from three annotated divisions, and their median displacement is around the middle of the corresponding `6bba` distribution.

There are only six division-edge observations for this movie, so its 95th percentile and maximum aren't as reliable as the continuation-edge results.



## Measure Directional Persistence Along Cell Tracks

So far, I've looked at how far a cell moves between frames, but distance alone doesn't tell me whether the cell keeps moving in the same direction.

I'm measuring directional persistence using pairs of consecutive continuation edges:

* first movement: **A → B**
* second movement: **B → C**

The angle between the two physical movement vectors shows how much the direction changes:

* **0°** means the cell keeps moving in the same direction;
* **90°** means it makes a right-angle turn;
* **180°** means it completely reverses direction.

I'm only using continuation edges so that I don't follow a trajectory through a cell division. I'm also excluding pairs where one movement has zero distance because its direction can't be defined.

This could be useful for tracking because the recent direction a cell was moving in may help distinguish its actual next position from other nearby cells.


In [ ]:
direction_edges = edge_summary.copy()

continuation_edges = direction_edges.loc[
    direction_edges["edge_type"].eq("Continuation edge"),
    [
        "sample_id",
        "group",
        "is_test_movie",
        "source_id",
        "target_id",
        "source_t",
        "target_t",
        "delta_z_um",
        "delta_y_um",
        "delta_x_um",
        "distance_um",
    ],
].copy()

consecutive_pairs = continuation_edges.merge(
    continuation_edges,
    left_on=["sample_id", "target_id"],
    right_on=["sample_id", "source_id"],
    suffixes=("_first", "_second"),
    validate="one_to_one",
)

# Keep only the first copy of movie-level information.
consecutive_pairs = consecutive_pairs.rename(
    columns={
        "group_first": "group",
        "is_test_movie_first": "is_test_movie",
        "source_id_first": "first_node_id",
        "target_id_first": "middle_node_id",
        "target_id_second": "last_node_id",
        "source_t_first": "first_t",
        "target_t_first": "middle_t",
        "target_t_second": "last_t",
    }
)

# Direction is undefined if either movement has zero length.
zero_distance_pair = (
    consecutive_pairs["distance_um_first"].eq(0)
    | consecutive_pairs["distance_um_second"].eq(0)
)

print("Consecutive continuation-edge pairs:", len(consecutive_pairs))
print("Pairs containing a zero-distance edge:", zero_distance_pair.sum())
print("Pairs available for directional analysis:", (~zero_distance_pair).sum())

display(
    consecutive_pairs.loc[
        ~zero_distance_pair,
        [
            "sample_id",
            "group",
            "first_node_id",
            "middle_node_id",
            "last_node_id",
            "first_t",
            "middle_t",
            "last_t",
            "distance_um_first",
            "distance_um_second",
        ],
    ].head()
)

### Calculate Turning Angles

For two consecutive physical movement vectors, **v₁** and **v₂**, I calculate the turning angle using their dot product:

$$
\cos(\theta)
=
\frac{\mathbf{v}_1 \cdot \mathbf{v}_2}
{\lVert\mathbf{v}_1\rVert \lVert\mathbf{v}_2\rVert}
$$


The movement vectors are converted to **micrometres** rather than using the raw voxel coordinates. This is important because one Z voxel represents about four times the physical distance of one X or Y voxel.

I've already removed all pairs where the movement distance is zero since their direction can't be defined. However, very small non-zero movements can still give unstable angles because even a small change in coordinates can create a large change in direction. I'll look at how sensitive the results are to movement length later.


In [ ]:
directional_pairs = consecutive_pairs.loc[
    ~zero_distance_pair
].copy()

first_vectors = directional_pairs[
    [
        "delta_z_um_first",
        "delta_y_um_first",
        "delta_x_um_first",
    ]
].to_numpy(dtype=float)

second_vectors = directional_pairs[
    [
        "delta_z_um_second",
        "delta_y_um_second",
        "delta_x_um_second",
    ]
].to_numpy(dtype=float)

dot_products = (
    first_vectors * second_vectors
).sum(axis=1)

first_lengths = np.linalg.norm(
    first_vectors,
    axis=1
)

second_lengths = np.linalg.norm(
    second_vectors,
    axis=1
)

cosine_turning_angle = (
    dot_products
    / (first_lengths * second_lengths)
)

# Protect against tiny floating-point errors such as 1.0000000002.
cosine_turning_angle = np.clip(
    cosine_turning_angle,
    -1,
    1,
)

directional_pairs["turning_angle_degrees"] = np.degrees(
    np.arccos(cosine_turning_angle)
)

angle_checks = pd.Series(
    {
        "number of directional pairs": len(directional_pairs),
        "all angles are finite": np.isfinite(
            directional_pairs["turning_angle_degrees"]
        ).all(),
        "minimum angle is at least 0 degrees": (
            directional_pairs["turning_angle_degrees"].min() >= 0
        ),
        "maximum angle is at most 180 degrees": (
            directional_pairs["turning_angle_degrees"].max() <= 180
        ),
        "first vector lengths match stored distances": np.allclose(
            first_lengths,
            directional_pairs["distance_um_first"],
        ),
        "second vector lengths match stored distances": np.allclose(
            second_lengths,
            directional_pairs["distance_um_second"],
        ),
    },
    name="value",
)

display(angle_checks.to_frame())

display(
    directional_pairs[
        [
            "sample_id",
            "group",
            "first_t",
            "middle_t",
            "last_t",
            "distance_um_first",
            "distance_um_second",
            "turning_angle_degrees",
        ]
    ].head().round(3)
)

### Summarise Turning Angles by Dataset Group

Turning-angle distributions were summarized across all eligible movement pairs in the dataset. Smaller turning angles indicate that cells tend to maintain their direction of movement, while larger angles indicate greater changes in direction. Specifically, a higher percentage of movement pairs with angles below 30° suggests more persistent, approximately forward movement, whereas a higher percentage above 90° suggests more frequent reversals. The median turning angle represents the typical change in direction across movement pairs.

These results are descriptive and summarize movement at the pair level. Because movement pairs are nested within individual movies and connected components, and neighboring pairs may share an edge, the pairs cannot be considered fully independent observations for formal statistical significance testing.


In [ ]:
def percentage_below_30(values):
    return 100 * (values < 30).mean()


def percentage_above_90(values):
    return 100 * (values > 90).mean()


turning_angle_summary = (
    directional_pairs
    .groupby("group")
    .agg(
        n_directional_pairs=(
            "turning_angle_degrees",
            "size",
        ),
        minimum_angle=(
            "turning_angle_degrees",
            "min",
        ),
        first_quartile_angle=(
            "turning_angle_degrees",
            first_quartile,
        ),
        median_angle=(
             "turning_angle_degrees",
            "median",
        ),
        mean_angle=(
            "turning_angle_degrees",
            "mean",
        ),
        third_quartile_angle=(
            "turning_angle_degrees",
            third_quartile,
        ),
        maximum_angle=(
            "turning_angle_degrees",
            "max",
        ),
        percentage_below_30_degrees=(
            "turning_angle_degrees",
            percentage_below_30,
        ),
        percentage_above_90_degrees=(
            "turning_angle_degrees",
            percentage_above_90,
        ),
    )
    .round(3)
)

display(turning_angle_summary)

### Initial Directional-Persistence Findings

At the movement-pair level, `6bba` shows greater directional persistence than `44b6`. The median turning angle is approximately 54.0° for `6bba`, compared with 69.9° for `44b6`. In addition, about 27.6% of movement pairs in `6bba` have turning angles below 30°, compared with 17.9% in `44b6`, suggesting that `6bba` more often continues moving in approximately the same direction.

In contrast, about 29.1% of `44b6` movement pairs have turning angles above 90°, compared with 20.2% in `6bba`. An angle above 90° indicates that the second movement is directed more opposite to the direction of the first movement, representing a greater tendency toward reversal.

Both groups have broad turning-angle distributions ranging from 0° to 180°, indicating that movement is not consistently in one direction. The 30° and 90° cutoffs are used as descriptive reference points for comparing the movement patterns and should not be interpreted as established biological thresholds.

These findings are based on individual movement pairs rather than independent biological replicates. Because movement pairs are nested within movies and connected components, the observed difference could partly reflect variation between movies or components. Very short movements may also affect the results because small changes in integer-coordinate positions can produce relatively large changes in the calculated direction.


### Visualise the Turning-Angle Distributions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.ecdfplot(
    data=directional_pairs,
    x="turning_angle_degrees",
    hue="group",
    hue_order=group_order,
    palette=group_palette,
    linewidth=2,
    ax=ax,
)

ax.axvline(
    30,
    color="grey",
    linestyle="--",
    linewidth=1,
    label="30° reference",
)

ax.axvline(
    90,
    color="black",
    linestyle="--",
    linewidth=1,
    label="90° reference",
)

ax.set_title("Turning Angles Between Consecutive Continuation Movements")
ax.set_xlabel("Turning angle (degrees)")
ax.set_ylabel("Cumulative proportion of movement pairs")
ax.set_xlim(0, 180)
ax.set_ylim(0, 1.02)

from matplotlib.lines import Line2D

legend_handles = [
    Line2D(
        [0],
        [0],
        color=group_palette["44b6"],
        linewidth=2,
        label="44b6",
    ),
    Line2D(
        [0],
        [0],
        color=group_palette["6bba"],
        linewidth=2,
        label="6bba",
    ),
    Line2D(
        [0],
        [0],
        color="grey",
        linestyle="--",
        linewidth=1,
        label="30° reference",
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        linewidth=1,
        label="90° reference",
    ),
]

ax.legend(
    handles=legend_handles,
    title="Dataset group",
    loc="lower right",
)

plt.tight_layout()
plt.show()

The 6bba ECDF lies above the 44b6 ECDF across almost the entire range, showing a greater proportion of small turning angles. Thus, a typical annotated movement pair in 6bba shows greater directional persistence. A separate movie-level comparison is needed to determine whether this is also true of a typical movie.

### Compare Directional Persistence Between Typical Movies

The previous ECDF gives each eligible movement pair equal weight, meaning that movies with more annotated trajectories contribute more movement pairs and therefore have a greater influence on the overall distribution.

To compare movement patterns across individual movies, the median turning angle is calculated separately for each movie. These movie-level medians are then compared between `6bba` and `44b6`, so that each movie contributes one observation regardless of how many trajectories or movement pairs it contains.

A lower movie-level median turning angle indicates greater directional persistence, meaning that cells tend to continue moving in a more consistent direction. This movie-level comparison provides a more balanced view of the differences between the two groups.


In [ ]:
movie_direction_summary = (
    directional_pairs
    .groupby(
        [
            "sample_id",
            "group",
            "is_test_movie",
        ]
    )
    .agg(
        n_directional_pairs=(
            "turning_angle_degrees",
            "size",
        ),
        median_turning_angle=(
            "turning_angle_degrees",
            "median",
        ),
        percentage_below_30_degrees=(
            "turning_angle_degrees",
            percentage_below_30,
        ),
        percentage_above_90_degrees=(
            "turning_angle_degrees",
            percentage_above_90,
        ),
    )
    .reset_index()
)

# Include all 199 movies so that any movie without a valid
# directional pair is visible rather than silently omitted.
movie_direction_summary = (
    graph_summary[
        [
            "sample_id",
            "group",
            "is_test_movie",
        ]
    ]
    .merge(
        movie_direction_summary,
        on=[
            "sample_id",
            "group",
            "is_test_movie",
        ],
        how="left",
        validate="one_to_one",
    )
)

typical_movie_direction_summary = (
    movie_direction_summary
    .groupby("group")
    .agg(
        n_movies=("sample_id", "size"),
        movies_with_directional_pairs=(
            "median_turning_angle",
            "count",
        ),
        first_quartile_movie_median=(
            "median_turning_angle",
            first_quartile,
        ),
        median_movie_median=(
            "median_turning_angle",
            "median",
        ),
        mean_movie_median=(
            "median_turning_angle",
            "mean",
        ),
        third_quartile_movie_median=(
            "median_turning_angle",
            third_quartile,
        ),
        median_percentage_below_30=(
            "percentage_below_30_degrees",
            "median",
        ),
        median_percentage_above_90=(
            "percentage_above_90_degrees",
            "median",
        ),
    )
    .round(3)
)

display(typical_movie_direction_summary)

print(
    "\nMovies without a valid directional pair:",
    movie_direction_summary[
        "median_turning_angle"
    ].isna().sum(),
)

### Movie-Level Directional-Persistence Findings

All 199 movies contain at least one valid pair of consecutive movements. When each movie is treated as one observation, the median turning angle for a typical `6bba` movie is approximately 55.3°, compared with 62.5° for a typical `44b6` movie. The `6bba` movies also tend to have a higher proportion of movement pairs with angles below 30° and a lower proportion with angles above 90°.

The difference between the two groups is smaller at the movie level than when all movement pairs are pooled together. This suggests that movies containing larger numbers of annotated movement pairs contributed more strongly to the separation seen in the pair-level analysis. Even with each movie weighted equally, however, the results still show a pattern of greater directional persistence in `6bba`.

There is substantial overlap in the interquartile ranges of the two groups, showing that turning behaviour varies considerably from movie to movie. Therefore, group membership alone does not explain the movement behaviour observed in an individual movie.


### Visualise Movie-Level Directional Persistence

In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=movie_direction_summary,
    x="group",
    y="median_turning_angle",
    order=group_order,
    palette=group_palette,
    showfliers=False,
    ax=ax,
)

sns.stripplot(
    data=movie_direction_summary,
    x="group",
    y="median_turning_angle",
    order=group_order,
    color="black",
    alpha=0.45,
    jitter=0.2,
    size=4,
    ax=ax,
)

# Plot the two test movies in each group with small fixed offsets
# so that overlapping values cannot hide a marker.
test_direction_data = movie_direction_summary.loc[
    movie_direction_summary["is_test_movie"]
]

for group_position, group in enumerate(group_order):
    group_test_data = test_direction_data.loc[
        test_direction_data["group"] == group
    ]

    x_positions = np.linspace(
        group_position - 0.06,
        group_position + 0.06,
        len(group_test_data),
    )

    ax.scatter(
        x_positions,
        group_test_data["median_turning_angle"],
        color="red",
        edgecolor="black",
        marker="D",
        s=100,
        zorder=5,
    )

ax.set_title("Median Turning Angle Within Each Movie")
ax.set_xlabel("Dataset group")
ax.set_ylabel("Movie median turning angle (degrees)")

test_handle = Line2D(
    [0],
    [0],
    marker="D",
    color="red",
    markeredgecolor="black",
    linestyle="none",
    markersize=9,
    label="Test movie",
)

ax.legend(handles=[test_handle])

plt.tight_layout()
plt.show()

### Test-Movie Positions in the Movie-Level Distributions

Although `6bba` has a lower typical turning angle overall, the two test movies in each group do not follow this overall pattern.

Both `44b6` test movies have relatively low median turning angles compared with other movies in the `44b6` group. In contrast, both `6bba` test movies have relatively high median turning angles within their group. This means that the test movies do not necessarily reflect the typical directional-persistence pattern observed across their respective groups.

The boxplots also show substantial overlap between the two groups, highlighting the amount of variation between individual movies. Therefore, group-level differences alone do not fully describe the directional movement observed in the test movies.


In [ ]:
movie_direction_summary[
    "median_angle_group_percentile"
] = (
    movie_direction_summary
    .groupby("group")["median_turning_angle"]
    .rank(pct=True)
    * 100
)

test_direction_summary = (
    movie_direction_summary.loc[
        movie_direction_summary["is_test_movie"]
    ]
    .sort_values(["group", "sample_id"])
)

display(
    test_direction_summary[
        [
            "sample_id",
            "group",
            "n_directional_pairs",
            "median_turning_angle",
            "median_angle_group_percentile",
            "percentage_below_30_degrees",
            "percentage_above_90_degrees",
        ]
    ].round(3)
)

### Test-Movie Directional Persistence

The test movies do not follow the overall directional-persistence pattern observed across the full dataset.

Both `44b6` test movies have relatively small median turning angles compared with other movies in their group. `44b6_0113de3b` falls at approximately the 7th percentile of the `44b6` movies, while `44b6_0b24845f` falls at approximately the 24th percentile. This indicates that both test movies show relatively strong directional persistence within their group.

In contrast, both `6bba` test movies have relatively large median turning angles. `6bba_05b6850b` is around the 97th percentile, with a median turning angle of 90°, while `6bba_05db0fb1` is around the 70th percentile. These values indicate more frequent changes in direction compared with most other `6bba` movies.

The `44b6` estimates are based on only 45–48 movement pairs, making their median estimates less stable than those from the more densely annotated `6bba` test movies. Overall, the test movies show substantial variation within each group, suggesting that group membership alone does not fully capture the directional movement characteristics observed in these samples.


### Check Sensitivity to Short Movements

Turning angles can be less reliable when one or both movements are very short. Since the cell centroids are recorded using integer voxel coordinates, even a small change in position can cause a relatively large change in the calculated direction.

To check whether this affects the results, the analysis is repeated using different minimum distance requirements, where both movements in a pair must be longer than the specified distance. If the difference between `6bba` and `44b6` stays similar after removing very short movements, this would suggest that the observed pattern is not mainly caused by coordinate rounding or discretisation.


In [ ]:
directional_pairs["minimum_step_distance_um"] = (
    directional_pairs[
        [
            "distance_um_first",
            "distance_um_second",
        ]
    ]
    .min(axis=1)
)

minimum_distance_thresholds = [0, 0.5, 1.0, 2.0]

sensitivity_rows = []

for threshold in minimum_distance_thresholds:
    threshold_data = directional_pairs.loc[
        directional_pairs["minimum_step_distance_um"]
        >= threshold
    ]

    for group in group_order:
        group_data = threshold_data.loc[
            threshold_data["group"] == group,
            "turning_angle_degrees",
        ]

        sensitivity_rows.append(
            {
                "minimum_step_distance_um": threshold,
                "group": group,
                "n_directional_pairs": len(group_data),
                "median_turning_angle": group_data.median(),
                "percentage_below_30_degrees":
                    percentage_below_30(group_data),
                "percentage_above_90_degrees":
                    percentage_above_90(group_data),
            }
        )

turning_angle_sensitivity = pd.DataFrame(
    sensitivity_rows
)

display(
    turning_angle_sensitivity
    .set_index(
        [
            "minimum_step_distance_um",
            "group",
        ]
    )
    .round(3)
)

### Short-Movement Sensitivity Findings

The greater directional persistence observed in `6bba` remains across all of the minimum-distance thresholds tested. This suggests that the difference between `6bba` and `44b6` is not simply caused by very short movements.

For each threshold, both movements within a pair are required to be at least that distance long. Therefore, increasing the threshold progressively removes movement pairs where either the first or second movement is shorter than the specified minimum. The thresholds can be interpreted as follows:

* **No minimum-distance filter:** all eligible movement pairs are included.
* **5-voxel threshold:** pairs are excluded if either movement is shorter than 5 voxels.
* **10-voxel threshold:** pairs are excluded if either movement is shorter than 10 voxels.
* **20-voxel threshold:** pairs are excluded if either movement is shorter than 20 voxels.
* **30-voxel threshold:** pairs are excluded if either movement is shorter than 30 voxels.

For both groups, the median turning angle decreases as shorter movements are removed. At the same time, the proportion of movement pairs below 30° increases, while the proportion above 90° decreases. This shows that excluding very short movements generally results in a more consistent directional pattern.

This pattern is consistent with very short movements having less stable directions, since small changes in centroid position can have a larger effect when the movement distance is small, and coordinates are recorded at integer-voxel resolution. It may also suggest that larger-scale movements tend to follow more coherent directions.


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    sharex=True,
)

sns.lineplot(
    data=turning_angle_sensitivity,
    x="minimum_step_distance_um",
    y="median_turning_angle",
    hue="group",
    hue_order=group_order,
    palette=group_palette,
    marker="o",
    linewidth=2,
    ax=axes[0],
)

axes[0].set_title("Median Turning Angle")
axes[0].set_xlabel("Minimum length of both movements (µm)")
axes[0].set_ylabel("Median turning angle (degrees)")

sns.lineplot(
    data=turning_angle_sensitivity,
    x="minimum_step_distance_um",
    y="percentage_above_90_degrees",
    hue="group",
    hue_order=group_order,
    palette=group_palette,
    marker="o",
    linewidth=2,
    ax=axes[1],
)

axes[1].set_title("Movement Pairs Turning More Than 90°")
axes[1].set_xlabel("Minimum length of both movements (µm)")
axes[1].set_ylabel("Percentage of movement pairs")

threshold_labels = [
    "All non-zero",
    "≥ 0.5",
    "≥ 1.0",
    "≥ 2.0",
]

for ax in axes:
    ax.set_xticks(
        minimum_distance_thresholds,
        labels=threshold_labels,
    )

# Keep only one shared legend.
axes[0].legend(title="Dataset group")
axes[1].get_legend().remove()

plt.tight_layout()
plt.show()

### Directional-Persistence Summary

* Across all consecutive-movement pairs, `6bba` has smaller turning angles than `44b6`, indicating greater directional persistence at the pair level.
* The same overall pattern remains when each movie is given equal weight, although there is substantial overlap between the movie-level distributions.
* The test movies do not follow the overall group pattern. Both `44b6` test movies have relatively small turning angles, while both `6bba` test movies have comparatively large turning angles.
* Removing shorter movements lowers the median turning angle in both groups, but the difference between `6bba` and `44b6` remains across all minimum-distance thresholds tested.
* Overall, recent movement direction could be useful as a feature for linking consecutive cell positions. However, its reliability depends on movement length, and apparent changes in direction may also be influenced by global image motion.


## Examine Spatial Crowding and Link Ambiguity

Tracking becomes more difficult when several possible target cells are close to the same source cell in the next frame, making the correct link more ambiguous.

This analysis measures distances between supplied annotations within each movie and time point. Since the dataset is sparsely labelled, these distances reflect crowding among annotated cells rather than overall cell density. Differences between `6bba` and `44b6` may also reflect differences in annotation coverage.

A node table is created from the source and target endpoints in the edge table. No isolated nodes were found, so every supplied node belongs to at least one edge.


### Build a Dataset-Level Node Table

In [ ]:
source_node_summary = (
    edge_summary[
        [
            "sample_id",
            "group",
            "is_test_movie",
            "source_id",
            "source_t",
            "source_z",
            "source_y",
            "source_x",
        ]
    ]
    .rename(
        columns={
            "source_id": "node_id",
            "source_t": "t",
            "source_z": "z",
            "source_y": "y",
            "source_x": "x",
        }
    )
)

target_node_summary = (
    edge_summary[
        [
            "sample_id",
            "group",
            "is_test_movie",
            "target_id",
            "target_t",
            "target_z",
            "target_y",
            "target_x",
        ]
    ]
    .rename(
        columns={
            "target_id": "node_id",
            "target_t": "t",
            "target_z": "z",
            "target_y": "y",
            "target_x": "x",
        }
    )
)

node_occurrences = pd.concat(
    [
        source_node_summary,
        target_node_summary,
    ],
    ignore_index=True,
)

# Check whether repeated appearances of the same node
# always have identical coordinates.
coordinate_conflicts = (
    node_occurrences
    .groupby(["sample_id", "node_id"])[
        ["t", "z", "y", "x"]
    ]
    .nunique()
    .gt(1)
    .any(axis=1)
    .sum()
)

node_summary = (
    node_occurrences
    .drop_duplicates(["sample_id", "node_id"])
    .sort_values(["sample_id", "t", "node_id"])
    .reset_index(drop=True)
)

node_table_checks = pd.Series(
    {
        "number of nodes": len(node_summary),
        "movies represented": node_summary["sample_id"].nunique(),
        "duplicate movie-node IDs": node_summary.duplicated(
            ["sample_id", "node_id"]
        ).sum(),
        "coordinate conflicts": coordinate_conflicts,
        "missing values": node_summary.isna().sum().sum(),
        "node total agrees with graph table": (
            len(node_summary)
            == graph_summary["n_nodes"].sum()
        ),
    },
    name="value",
)

display(node_table_checks.to_frame())
display(node_summary.head())

### Count Supplied Nodes Within Each Frame

A frame with no supplied nodes provides no tracking information, while a frame with only one node has no competing target. Nearest-neighbour crowding can only be measured when at least two nodes are present.

All 100 time points are included for every movie, including frames with no annotations. Since every movie has the same number of frames, each movie contributes equally to the frame-level summary.


In [ ]:
observed_frame_counts = (
    node_summary
    .groupby(
        [
            "sample_id",
            "group",
            "is_test_movie",
            "t",
        ]
    )
    .size()
    .rename("n_annotated_nodes")
    .reset_index()
)

# Construct all 100 frames for every movie so that frames
# containing no supplied nodes are not silently omitted.
all_movie_frames = (
    graph_summary[
        [
            "sample_id",
            "group",
            "is_test_movie",
        ]
    ]
    .merge(
        pd.DataFrame({"t": np.arange(100)}),
        how="cross",
    )
)

frame_node_counts = (
    all_movie_frames
    .merge(
        observed_frame_counts,
        on=[
            "sample_id",
            "group",
            "is_test_movie",
            "t",
        ],
        how="left",
        validate="one_to_one",
    )
)

frame_node_counts["n_annotated_nodes"] = (
    frame_node_counts["n_annotated_nodes"]
    .fillna(0)
    .astype(int)
)

frame_count_summary = (
    frame_node_counts
    .groupby("group")
    .agg(
        n_frames=("t", "size"),
        mean_nodes_per_frame=(
            "n_annotated_nodes",
            "mean",
        ),
        median_nodes_per_frame=(
            "n_annotated_nodes",
            "median",
        ),
        maximum_nodes_in_one_frame=(
            "n_annotated_nodes",
            "max",
        ),
        percentage_empty_frames=(
            "n_annotated_nodes",
            lambda values: 100 * values.eq(0).mean(),
        ),
        percentage_one_node_frames=(
            "n_annotated_nodes",
            lambda values: 100 * values.eq(1).mean(),
        ),
        percentage_frames_with_two_or_more=(
            "n_annotated_nodes",
            lambda values: 100 * values.ge(2).mean(),
        ),
    )
    .round(3)
)

display(frame_count_summary)

print(
    "\nTotal movie-frame rows:",
    len(frame_node_counts),
)

### Supplied Frame-Level Annotation Density

The supplied annotations are much denser in `6bba` frames. A typical `6bba` frame contains about eight annotated nodes, compared with two in `44b6`.

Of the **total 199 `6bba` movies × 100 frames per movie**, 96.5% of frames contain at least two annotations and can therefore be used to calculate a nearest-neighbour distance. For `44b6`, 69.1% of the **total `44b6` movie × frame observations** contain at least two annotations.

This does not necessarily mean that the biological tissue is denser in `6bba`. Because annotation coverage differs substantially between the groups, these results mainly describe the density of the supplied annotations.


### Calculate Within-Frame Nearest-Neighbour Distances

For each supplied node, the nearest other supplied node in the same movie and timepoint is identified using physical coordinates. The Euclidean distance between two nodes is calculated as:

$$
d=\sqrt{(x_2-x_1)^2+(y_2-y_1)^2+(z_2-z_1)^2}
$$

The smallest distance for each node is used as its nearest-neighbour distance.

A smaller distance means another annotated cell is closer, which may make detection and linking more ambiguous. However, unannotated cells are not included in this calculation.

Frames with only one supplied node have no nearest-neighbour distance.


In [ ]:
from scipy.spatial import cKDTree

node_crowding_frames = []

for (_, _), frame_nodes in node_summary.groupby(
    ["sample_id", "t"],
    sort=False,
):
    frame_nodes = frame_nodes.copy()

    physical_coordinates = np.column_stack(
        [
            frame_nodes["z"].to_numpy() * 1.625,
            frame_nodes["y"].to_numpy() * 0.40625,
            frame_nodes["x"].to_numpy() * 0.40625,
        ]
    )

    if len(frame_nodes) >= 2:
        tree = cKDTree(physical_coordinates)

        neighbour_distances, neighbour_indices = tree.query(
            physical_coordinates,
            k=2,
        )

        frame_nodes["nearest_neighbor_distance_um"] = (
            neighbour_distances[:, 1]
        )

        frame_nodes["nearest_neighbor_node_id"] = (
            frame_nodes.iloc[
                neighbour_indices[:, 1]
            ]["node_id"].to_numpy()
        )
    else:
        frame_nodes["nearest_neighbor_distance_um"] = np.nan
        frame_nodes["nearest_neighbor_node_id"] = np.nan

    node_crowding_frames.append(frame_nodes)

node_crowding = pd.concat(
    node_crowding_frames,
    ignore_index=True,
)

node_crowding["nearest_neighbor_node_id"] = (
    node_crowding["nearest_neighbor_node_id"]
    .astype("Int64")
)

nodes_in_multi_node_frames = (
    frame_node_counts.loc[
        frame_node_counts["n_annotated_nodes"] >= 2,
        "n_annotated_nodes",
    ]
    .sum()
)

nearest_neighbor_checks = pd.Series(
    {
        "number of nodes": len(node_crowding),
        "movies represented": node_crowding[
            "sample_id"
        ].nunique(),
        "nodes with a nearest neighbour": node_crowding[
            "nearest_neighbor_distance_um"
        ].notna().sum(),
        "expected nodes with a nearest neighbour":
            nodes_in_multi_node_frames,
        "all distances are non-negative": node_crowding[
            "nearest_neighbor_distance_um"
        ].dropna().ge(0).all(),
    },
    name="value",
)

display(nearest_neighbor_checks.to_frame())

display(
    node_crowding.loc[
        node_crowding["nearest_neighbor_distance_um"].notna(),
        [
            "sample_id",
            "group",
            "node_id",
            "t",
            "z",
            "y",
            "x",
            "nearest_neighbor_node_id",
            "nearest_neighbor_distance_um",
        ],
    ].head().round(3)
)

### Summarise Supplied-Annotation Crowding

Nearest-neighbour distances are summarized for nodes that have at least one other supplied annotation in the same frame.

Smaller distances indicate greater crowding among the supplied labels. Since `6bba` has more annotations per frame, it may have smaller distances simply because more labelled cells are present, even if the actual biological density is similar.

These are descriptive node-level measurements, and nodes from the same movie or frame are not independent observations.


In [ ]:
crowding_summary = (
    node_crowding
    .groupby("group")
    .agg(
        n_nodes_with_neighbor=(
            "nearest_neighbor_distance_um",
            "count",
        ),
        minimum_distance=(
            "nearest_neighbor_distance_um",
            "min",
        ),
        first_quartile_distance=(
            "nearest_neighbor_distance_um",
            first_quartile,
        ),
        median_distance=(
            "nearest_neighbor_distance_um",
            "median",
        ),
        mean_distance=(
            "nearest_neighbor_distance_um",
            "mean",
        ),
        third_quartile_distance=(
            "nearest_neighbor_distance_um",
            third_quartile,
        ),
        percentile_95_distance=(
            "nearest_neighbor_distance_um",
            ninety_fifth_percentile,
        ),
        maximum_distance=(
            "nearest_neighbor_distance_um",
            "max",
        ),
        percentage_within_5_um=(
            "nearest_neighbor_distance_um",
            lambda values: 100 * values.le(5).mean(),
        ),
        percentage_within_10_um=(
            "nearest_neighbor_distance_um",
            lambda values: 100 * values.le(10).mean(),
        ),
    )
    .round(3)
)

display(crowding_summary)

### Initial Supplied-Crowding Findings

The higher number of supplied annotations in `6bba` does not necessarily result in smaller nearest-neighbour distances.

`44b6` has a slightly smaller median nearest-neighbour distance than `6bba` (22.76 µm vs. 25.29 µm). Supplied neighbours within 10 µm are also more common in `44b6`, although they are uncommon in both groups.

At the same time, `44b6` has a longer upper tail, with a 95th percentile of 60.28 µm compared with 50.31 µm in `6bba`. This means `44b6` contains both more closely spaced and more widely separated supplied annotations.

The interquartile ranges overlap substantially. Because the dataset is sparsely labelled, these distances only describe the spatial arrangement of the supplied annotations and do not represent complete biological cell density. Differences may also reflect differences in annotation coverage between the groups.


### Visualise Nearest-Neighbour Distances

In [ ]:
crowding_plot_data = node_crowding.loc[
    node_crowding["nearest_neighbor_distance_um"].notna()
]

fig, ax = plt.subplots(figsize=(10, 6))

sns.ecdfplot(
    data=crowding_plot_data,
    x="nearest_neighbor_distance_um",
    hue="group",
    hue_order=group_order,
    palette=group_palette,
    linewidth=2,
    ax=ax,
)

# Descriptive close-neighbour reference distances.
ax.axvline(
    5,
    color="grey",
    linestyle="--",
    linewidth=1,
)

ax.axvline(
    10,
    color="black",
    linestyle="--",
    linewidth=1,
)

ax.set_title(
    "Nearest Supplied Annotation Within the Same Frame"
)
ax.set_xlabel("Nearest-neighbour distance (µm)")
ax.set_ylabel("Cumulative proportion of annotated nodes")
ax.set_xlim(0, 80)
ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.show()

### Nearest-Neighbour Distribution Findings

The nearest-neighbour ECDFs cross at about 40 µm. This means that at around 40 µm, the two groups have approximately the same proportion of supplied annotations with a nearest neighbour at or below that distance. Therefore, neither group consistently has smaller nearest-neighbour distances across the full range.

Below 40 µm, `44b6` has a greater proportion of relatively close supplied neighbours. Above 40 µm, more `44b6` annotations remain in the upper tail, indicating more highly isolated annotations.

Overall, `44b6` has a more spread-out nearest-neighbour distribution, while `6bba` is more concentrated around the middle range. Very close supplied neighbours below 10 µm are uncommon in both groups.


### Compare Crowding Between Typical Movies

Nearest-neighbour distances are summarized separately within each movie and then compared across movies, so each movie contributes one observation.

Here, supplied annotations refer to the cell positions that are provided in the dataset. The movie-level median represents the typical spacing between these annotated cell positions, while the 95th percentile shows the spacing of the more isolated annotated cells.


In [ ]:
valid_node_crowding = node_crowding.loc[
    node_crowding["nearest_neighbor_distance_um"].notna()
]

movie_crowding_summary = (
    valid_node_crowding
    .groupby(
        [
            "sample_id",
            "group",
            "is_test_movie",
        ]
    )
    .agg(
        n_nodes_with_neighbor=(
            "nearest_neighbor_distance_um",
            "size",
        ),
        median_nearest_neighbor_distance=(
            "nearest_neighbor_distance_um",
            "median",
        ),
        percentile_95_nearest_neighbor_distance=(
            "nearest_neighbor_distance_um",
            ninety_fifth_percentile,
        ),
        percentage_within_10_um=(
            "nearest_neighbor_distance_um",
            lambda values: 100 * values.le(10).mean(),
        ),
    )
    .reset_index()
)

movie_crowding_summary = (
    graph_summary[
        [
            "sample_id",
            "group",
            "is_test_movie",
        ]
    ]
    .merge(
        movie_crowding_summary,
        on=[
            "sample_id",
            "group",
            "is_test_movie",
        ],
        how="left",
        validate="one_to_one",
    )
)

typical_movie_crowding_summary = (
    movie_crowding_summary
    .groupby("group")
    .agg(
        n_movies=("sample_id", "size"),
        movies_with_neighbor_distances=(
            "median_nearest_neighbor_distance",
            "count",
        ),
        first_quartile_movie_median=(
            "median_nearest_neighbor_distance",
            first_quartile,
        ),
        median_movie_median=(
            "median_nearest_neighbor_distance",
            "median",
        ),
        mean_movie_median=(
            "median_nearest_neighbor_distance",
            "mean",
        ),
        third_quartile_movie_median=(
            "median_nearest_neighbor_distance",
            third_quartile,
        ),
        median_movie_95th_percentile=(
            "percentile_95_nearest_neighbor_distance",
            "median",
        ),
        median_percentage_within_10_um=(
            "percentage_within_10_um",
            "median",
        ),
    )
    .round(3)
)

display(typical_movie_crowding_summary)

print(
    "\nMovies without a nearest-neighbour distance:",
    movie_crowding_summary[
        "median_nearest_neighbor_distance"
    ].isna().sum(),
)

### Movie-Level Supplied-Crowding Findings

When each movie is weighted equally, the typical nearest-neighbour distances are very similar: about 25.00 µm in `44b6` and 26.00 µm in `6bba`.

The main difference is in how much the distances vary between movies. `44b6` has a wider interquartile range (17.61–34.11 µm) than `6bba` (22.35–30.59 µm), showing more variation in supplied-label spacing across `44b6` movies.

This suggests that the higher number of close `44b6` neighbours seen earlier is concentrated in certain movies rather than being typical of the group as a whole.

For each movie, the percentage represents the **proportion of nodes with a measurable nearest-neighbour distance that are within 10 µm of another supplied annotation**. The median of these movie-level percentages is 0% in `44b6` and 0.25% in `6bba`, so very close annotations are uncommon in a typical movie.

Seven movies have no nearest-neighbour measurements because they do not contain frames with at least two supplied annotations. These movies need to be accounted for when interpreting the movie-level results.


### Identify Movies Without Measurable Within-Frame Crowding

In [ ]:
movie_frame_count_summary = (
    frame_node_counts
    .groupby(
        [
            "sample_id",
            "group",
            "is_test_movie",
        ]
    )
    .agg(
        total_annotated_nodes=(
            "n_annotated_nodes",
            "sum",
        ),
        maximum_nodes_in_one_frame=(
            "n_annotated_nodes",
            "max",
        ),
        frames_with_two_or_more=(
            "n_annotated_nodes",
            lambda values: values.ge(2).sum(),
        ),
    )
    .reset_index()
)

movies_without_crowding = (
    movie_crowding_summary.loc[
        movie_crowding_summary[
            "median_nearest_neighbor_distance"
        ].isna(),
        [
            "sample_id",
            "group",
            "is_test_movie",
        ],
    ]
    .merge(
        movie_frame_count_summary,
        on=[
            "sample_id",
            "group",
            "is_test_movie",
        ],
        how="left",
        validate="one_to_one",
    )
)

display(movies_without_crowding)

### Movies with Undefined Nearest-Neighbour Distances

All seven movies without a nearest-neighbour measurement belong to `44b6`. In these movies, each frame contains at most one supplied annotation, so no within-frame nearest-neighbour distance can be calculated from the available annotations.

This includes the test movie `44b6_0113de3b`. Its undefined crowding measurement does **not** indicate that nearby biological cells are absent. It only indicates that the sparse supplied annotations do not contain a second annotated cell in the same frame to measure against.


### Compare Test-Movie Supplied-Annotation Crowding

In [ ]:
movie_crowding_summary[
    "median_distance_group_percentile"
] = (
    movie_crowding_summary
    .groupby("group")[
        "median_nearest_neighbor_distance"
    ]
    .rank(pct=True)
    * 100
)

test_crowding_summary = (
    movie_crowding_summary.loc[
        movie_crowding_summary["is_test_movie"]
    ]
    .merge(
        movie_frame_count_summary[
            [
                "sample_id",
                "maximum_nodes_in_one_frame",
                "frames_with_two_or_more",
            ]
        ],
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(["group", "sample_id"])
)

display(
    test_crowding_summary[
        [
            "sample_id",
            "group",
            "maximum_nodes_in_one_frame",
            "frames_with_two_or_more",
            "n_nodes_with_neighbor",
            "median_nearest_neighbor_distance",
            "median_distance_group_percentile",
            "percentile_95_nearest_neighbor_distance",
            "percentage_within_10_um",
        ]
    ].round(3)
)

### Test-Movie Supplied-Crowding Findings

The test movies span contrasting supplied-annotation crowding conditions.

`44b6_0113de3b` has no measurable within-frame crowding because it never contains more than one supplied node per frame. The other `44b6` test movie, `44b6_0b24845f`, is the most isolated **valid** movie in its group, although its estimate is based on only 22 nodes.

The two `6bba` test movies occupy opposite parts of their group distribution. `6bba_05b6850b` is relatively isolated at approximately the 88th percentile, whereas `6bba_05db0fb1` is relatively crowded at approximately the 12th percentile.

These differences show that the test movies do not all represent the same annotation-density or spatial-crowding conditions. A validation strategy should therefore include movies with different annotation densities and spatial arrangements rather than relying only on dataset group.


### Visualise Movie-Level Supplied Crowding

In [ ]:
valid_movie_crowding = movie_crowding_summary.loc[
    movie_crowding_summary[
        "median_nearest_neighbor_distance"
    ].notna()
]

fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=valid_movie_crowding,
    x="group",
    y="median_nearest_neighbor_distance",
    hue="group",
    order=group_order,
    hue_order=group_order,
    palette=group_palette,
    dodge=False,
    legend=False,
    showfliers=False,
    ax=ax,
)

sns.stripplot(
    data=valid_movie_crowding,
    x="group",
    y="median_nearest_neighbor_distance",
    order=group_order,
    color="black",
    alpha=0.45,
    jitter=0.2,
    size=4,
    ax=ax,
)

valid_test_crowding = test_crowding_summary.loc[
    test_crowding_summary[
        "median_nearest_neighbor_distance"
    ].notna()
]

for group_position, group in enumerate(group_order):
    group_test_data = valid_test_crowding.loc[
        valid_test_crowding["group"] == group
    ]

    if len(group_test_data) == 1:
        offsets = np.array([0.0])
    else:
        offsets = np.linspace(
            -0.06,
            0.06,
            len(group_test_data),
        )

    for offset, (_, row) in zip(
        offsets,
        group_test_data.iterrows(),
    ):
        x_position = group_position + offset

        ax.scatter(
            x_position,
            row["median_nearest_neighbor_distance"],
            color="red",
            edgecolor="black",
            marker="D",
            s=100,
            zorder=5,
        )

        group_median = valid_movie_crowding.loc[
            valid_movie_crowding["group"] == group,
            "median_nearest_neighbor_distance",
        ].median()
        
        vertical_offset = (
            -15
            if row["median_nearest_neighbor_distance"] >= group_median
            else 8
        )
        
        ax.annotate(
            row["sample_id"].split("_", 1)[1],
            (
                x_position,
                row["median_nearest_neighbor_distance"],
            ),
            xytext=(7, vertical_offset),
            textcoords="offset points",
            fontsize=8,
            bbox={
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.75,
                "pad": 1,
            },
        )

ax.set_title(
    "Median Nearest Supplied-Annotation Distance Within Each Movie"
)
ax.set_xlabel("Dataset group")
ax.set_ylabel("Movie median nearest-neighbour distance (µm)")

test_handle = Line2D(
    [0],
    [0],
    marker="D",
    color="red",
    markeredgecolor="black",
    linestyle="none",
    markersize=9,
    label="Test movie with defined distance",
)

ax.legend(handles=[test_handle])

ax.text(
    0.02,
    0.98,
    "44b6_0113de3b: undefined",
    transform=ax.transAxes,
    ha="left",
    va="top",
    color="red",
    fontsize=9,
)

plt.tight_layout()
plt.show()

### Movie-Level Crowding Plot

The two groups have very similar central values, with typical movie-level median nearest-neighbour distances of approximately 25–26 µm. The main difference is variability: `44b6` has a much wider distribution of typical nearest-neighbour distances, while `6bba` is more concentrated.

The group distributions overlap substantially. `44b6_0b24845f` is an extreme isolated case among movies with measurable distances, whereas `44b6_0113de3b` has no defined distance because it contains at most one supplied annotation per frame. The two `6bba` test movies occupy opposite sides of their group distribution, with `05b6850b` relatively isolated and `05db0fb1` relatively crowded.


### Rank True Continuation Targets Among Next-Frame Candidates

For each supplied continuation edge, every supplied node in the following frame of the same movie is treated as a possible target.

The physical distance from the source node to each candidate target is calculated and used to rank the candidates. This lets us determine whether the true supplied target is:

* the nearest candidate;
* tied for the nearest candidate; or
* farther away than at least one incorrect candidate.

Frames containing only one supplied candidate are trivial because the true target must rank first. Results will therefore be separated by the number of available candidates to distinguish easy cases from more ambiguous ones.

This remains an analysis of sparse supplied annotations. A real detection model will produce additional candidate cells that are not represented here. Division edges are excluded initially and will be considered separately.


In [ ]:
continuation_source_edges = (
    edge_summary.loc[
        edge_summary["edge_type"].eq("Continuation edge"),
        [
            "sample_id",
            "group",
            "is_test_movie",
            "source_id",
            "target_id",
            "source_t",
            "target_t",
            "source_z",
            "source_y",
            "source_x",
            "distance_um",
        ],
    ]
    .reset_index()
    .rename(
        columns={
            "index": "edge_row_id",
            "target_id": "true_target_id",
            "distance_um": "true_target_distance_um",
        }
    )
)

candidate_nodes = (
    node_summary[
        [
            "sample_id",
            "node_id",
            "t",
            "z",
            "y",
            "x",
        ]
    ]
    .rename(
        columns={
            "node_id": "candidate_node_id",
            "t": "candidate_t",
            "z": "candidate_z",
            "y": "candidate_y",
            "x": "candidate_x",
        }
    )
)

continuation_candidates = continuation_source_edges.merge(
    candidate_nodes,
    left_on=["sample_id", "target_t"],
    right_on=["sample_id", "candidate_t"],
    how="left",
    validate="many_to_many",
)

continuation_candidates["candidate_delta_z_um"] = (
    continuation_candidates["candidate_z"]
    - continuation_candidates["source_z"]
) * 1.625

continuation_candidates["candidate_delta_y_um"] = (
    continuation_candidates["candidate_y"]
    - continuation_candidates["source_y"]
) * 0.40625

continuation_candidates["candidate_delta_x_um"] = (
    continuation_candidates["candidate_x"]
    - continuation_candidates["source_x"]
) * 0.40625

continuation_candidates["candidate_distance_um"] = np.sqrt(
    continuation_candidates["candidate_delta_z_um"] ** 2
    + continuation_candidates["candidate_delta_y_um"] ** 2
    + continuation_candidates["candidate_delta_x_um"] ** 2
)

continuation_candidates["is_true_target"] = (
    continuation_candidates["candidate_node_id"]
    == continuation_candidates["true_target_id"]
)

candidate_counts = (
    continuation_candidates
    .groupby("edge_row_id")
    .size()
)

true_target_counts = (
    continuation_candidates
    .groupby("edge_row_id")["is_true_target"]
    .sum()
)

true_candidate_rows = continuation_candidates.loc[
    continuation_candidates["is_true_target"]
]

candidate_table_checks = pd.Series(
    {
        "continuation edges": len(continuation_source_edges),
        "candidate rows": len(continuation_candidates),
        "every edge has at least one candidate":
            candidate_counts.ge(1).all(),
        "every edge has exactly one true target":
            true_target_counts.eq(1).all(),
        "true candidate distances match edge distances":
            np.allclose(
                true_candidate_rows["candidate_distance_um"],
                true_candidate_rows["true_target_distance_um"],
            ),
        "minimum candidates for one edge":
            candidate_counts.min(),
        "maximum candidates for one edge":
            candidate_counts.max(),
    },
    name="value",
)

display(candidate_table_checks.to_frame())

# Show one example with multiple candidate targets.
example_edge_row_id = (
    candidate_counts.loc[
        candidate_counts >= 2
    ]
    .index[0]
)

display(
    continuation_candidates.loc[
        continuation_candidates["edge_row_id"]
        == example_edge_row_id,
        [
            "sample_id",
            "source_id",
            "source_t",
            "target_t",
            "candidate_node_id",
            "candidate_distance_um",
            "is_true_target",
        ],
    ]
    .sort_values("candidate_distance_um")
    .round(3)
)

### Classify Continuation-Link Ambiguity

Each true continuation link is placed into one of four categories:

* **only candidate**: no alternative supplied node exists in the next frame;
* **uniquely nearest**: the true target is closer than every alternative candidate;
* **tied nearest**: at least one alternative candidate is at the same distance as the true target;
* **incorrect candidate closer**: at least one alternative candidate is closer than the supplied true target.

For links with at least one alternative candidate, the **distance margin** is calculated as:

**nearest incorrect-candidate distance − true-target distance**

A large positive margin means the true target is clearly closer than the alternatives, while a margin near zero indicates an ambiguous decision. A negative margin means that at least one incorrect candidate is closer than the supplied true target, so nearest-neighbour linking would select the wrong candidate.


In [ ]:
distance_tolerance = 1e-9

continuation_candidates["distance_difference_um"] = (
    continuation_candidates["candidate_distance_um"]
    - continuation_candidates["true_target_distance_um"]
)

continuation_candidates["incorrect_candidate_is_closer"] = (
    ~continuation_candidates["is_true_target"]
    & (
        continuation_candidates["distance_difference_um"]
        < -distance_tolerance
    )
)

continuation_candidates["incorrect_candidate_is_tied"] = (
    ~continuation_candidates["is_true_target"]
    & (
        continuation_candidates["distance_difference_um"]
        .abs()
        <= distance_tolerance
    )
)

continuation_candidates[
    "incorrect_candidate_distance_um"
] = continuation_candidates[
    "candidate_distance_um"
].where(
    ~continuation_candidates["is_true_target"]
)

link_ambiguity = (
    continuation_candidates
    .groupby("edge_row_id")
    .agg(
        sample_id=("sample_id", "first"),
        group=("group", "first"),
        is_test_movie=("is_test_movie", "first"),
        source_id=("source_id", "first"),
        true_target_id=("true_target_id", "first"),
        source_t=("source_t", "first"),
        target_t=("target_t", "first"),
        true_target_distance_um=(
            "true_target_distance_um",
            "first",
        ),
        n_candidates=("candidate_node_id", "size"),
        n_incorrect_candidates_closer=(
            "incorrect_candidate_is_closer",
            "sum",
        ),
        n_incorrect_candidates_tied=(
            "incorrect_candidate_is_tied",
            "sum",
        ),
        nearest_incorrect_candidate_distance_um=(
            "incorrect_candidate_distance_um",
            "min",
        ),
    )
    .reset_index()
)

link_ambiguity["true_target_rank"] = (
    1
    + link_ambiguity["n_incorrect_candidates_closer"]
)

link_ambiguity["distance_margin_um"] = (
    link_ambiguity[
        "nearest_incorrect_candidate_distance_um"
    ]
    - link_ambiguity["true_target_distance_um"]
)

link_ambiguity["ambiguity_status"] = np.select(
    [
        link_ambiguity["n_candidates"].eq(1),
        link_ambiguity[
            "n_incorrect_candidates_closer"
        ].gt(0),
        link_ambiguity[
            "n_incorrect_candidates_tied"
        ].gt(0),
    ],
    [
        "Only candidate",
        "Incorrect candidate closer",
        "Tied nearest",
    ],
    default="Uniquely nearest",
)

ambiguity_checks = pd.Series(
    {
        "number of continuation links": len(link_ambiguity),
        "candidate rows accounted for": (
            link_ambiguity["n_candidates"].sum()
            == len(continuation_candidates)
        ),
        "minimum true-target rank": (
            link_ambiguity["true_target_rank"].min()
        ),
        "maximum true-target rank": (
            link_ambiguity["true_target_rank"].max()
        ),
        "only-candidate margins are missing": (
            link_ambiguity.loc[
                link_ambiguity["ambiguity_status"]
                == "Only candidate",
                "distance_margin_um",
            ]
            .isna()
            .all()
        ),
        "unique-nearest margins are positive": (
            link_ambiguity.loc[
                link_ambiguity["ambiguity_status"]
                == "Uniquely nearest",
                "distance_margin_um",
            ]
            .gt(0)
            .all()
        ),
    },
    name="value",
)

display(ambiguity_checks.to_frame())

display(
    link_ambiguity["ambiguity_status"]
    .value_counts()
    .to_frame("n_links")
)

display(link_ambiguity.head().round(3))

### Most Supplied Continuation Links Are Spatially Unambiguous

Most continuation links are very clear based on distance alone. About 98.61% of the true targets are uniquely the closest supplied annotation in the next frame, and another 1.25% are the only supplied candidate available.

Only 171 links, or about 0.13%, have an incorrect candidate that is closer than the true target. One additional link is tied, and in the most difficult case, the true target ranks sixth instead of first.

Overall, this shows that physical distance alone is enough to identify the correct supplied target in almost every case in this dataset. However, this does **not** mean a real tracking model would have 98–99% accuracy. The supplied annotations are sparse, while a real detector would have to deal with more candidate cells, false positives, and errors in detected cell positions.

The 171 cases where distance alone fails are especially useful because they show where other information, such as recent movement direction, cell appearance, or global image motion, could help make the correct link.

### Compare Continuation-Link Ambiguity Between Groups

In [ ]:
ambiguity_group_summary = (
    link_ambiguity
    .groupby("group")
    .agg(
        n_continuation_links=(
            "edge_row_id",
            "size",
        ),
        mean_candidates=(
            "n_candidates",
            "mean",
        ),
        median_candidates=(
            "n_candidates",
            "median",
        ),
        maximum_candidates=(
            "n_candidates",
            "max",
        ),
        percentage_only_candidate=(
            "ambiguity_status",
            lambda values:
                100 * values.eq("Only candidate").mean(),
        ),
        percentage_uniquely_nearest=(
            "ambiguity_status",
            lambda values:
                100 * values.eq("Uniquely nearest").mean(),
        ),
        percentage_incorrect_candidate_closer=(
            "ambiguity_status",
            lambda values:
                100
                * values.eq(
                    "Incorrect candidate closer"
                ).mean(),
        ),
        percentage_tied_nearest=(
            "ambiguity_status",
            lambda values:
                100 * values.eq("Tied nearest").mean(),
        ),
        maximum_true_target_rank=(
            "true_target_rank",
            "max",
        ),
    )
    .round(3)
)

display(ambiguity_group_summary)

### Group Differences in Supplied Link Ambiguity

`6bba` continuation links generally have more supplied candidates to choose from in the following frame, with a median of 11 candidates compared with only four in `44b6`.

Because `44b6` has fewer supplied candidates, links where the true target is the only available candidate are much more common in that group. When there are multiple candidates, `6bba` is slightly more likely to have an incorrect candidate closer than the true target. Its hardest case is also more ambiguous, with the true target ranking sixth compared with second in `44b6`.

Even with these differences, failures are extremely rare in both groups. When only-candidate and uniquely-nearest cases are combined, physical distance gives a clear and correct choice for about 99.9% of the sparse supplied continuation links.

### Examine Links with Genuine Alternatives

Links with only one supplied candidate are removed from this analysis. This means every remaining source has at least one incorrect candidate that could potentially be chosen instead of the true target.

The **distance margin** shows how much farther away the nearest incorrect candidate is compared with the true target:

- a **large positive margin** means the true target is clearly closer;
- a **small positive margin** means the true target is still closer, but the decision is less clear;
- a **zero margin** means the true target and an incorrect candidate are tied;
- a **negative margin** means an incorrect candidate is closer than the true target.

In [ ]:
nontrivial_link_ambiguity = link_ambiguity.loc[
    link_ambiguity["n_candidates"] >= 2
].copy()


def fifth_percentile(values):
    return values.quantile(0.05)


nontrivial_ambiguity_summary = (
    nontrivial_link_ambiguity
    .groupby("group")
    .agg(
        n_links_with_alternatives=(
            "edge_row_id",
            "size",
        ),
        percentage_uniquely_nearest=(
            "ambiguity_status",
            lambda values:
                100 * values.eq("Uniquely nearest").mean(),
        ),
        percentage_incorrect_candidate_closer=(
            "ambiguity_status",
            lambda values:
                100
                * values.eq(
                    "Incorrect candidate closer"
                ).mean(),
        ),
        percentage_tied_nearest=(
            "ambiguity_status",
            lambda values:
                100 * values.eq("Tied nearest").mean(),
        ),
        percentile_5_distance_margin=(
            "distance_margin_um",
            fifth_percentile,
        ),
        first_quartile_distance_margin=(
            "distance_margin_um",
            first_quartile,
        ),
        median_distance_margin=(
            "distance_margin_um",
            "median",
        ),
        third_quartile_distance_margin=(
            "distance_margin_um",
            third_quartile,
        ),
        percentage_margin_at_most_1_um=(
            "distance_margin_um",
            lambda values: 100 * values.le(1).mean(),
        ),
        percentage_margin_at_most_5_um=(
            "distance_margin_um",
            lambda values: 100 * values.le(5).mean(),
        ),
    )
    .round(3)
)

display(nontrivial_ambiguity_summary)

### True Targets Usually Have a Large Distance Advantage

When there is at least one incorrect alternative to choose from, the true target is still uniquely the closest candidate in more than 99.8% of continuation links in both groups.

Most of these decisions are also not close calls. The median distance advantage over the nearest incorrect candidate is about 20.88 µm in `44b6` and 23.23 µm in `6bba`. Even at the 5th percentile, the true target is still closer by about 6.65 µm in `44b6` and 9.81 µm in `6bba`.

The difference between the groups matters because they show different types of linking difficulty. `44b6` has more cases where the true target is ahead by less than 5 µm. These links are still correct based on distance, but the smaller margin means that localization errors or additional detected cells could more easily change which candidate appears closest.

`6bba`, on the other hand, usually has a larger distance advantage, but it also has a slightly higher rate of cases where an incorrect candidate is actually closer than the true target. In these cases, nearest-neighbour linking would already make the wrong decision even without additional detection errors.

This means one group is not necessarily harder overall. Instead, `44b6` has more fragile correct decisions, while `6bba` has slightly more rare cases where distance alone fails. These difficult cases are especially useful for testing whether movement direction, cell appearance, or global motion correction can improve linking.

### Locate Incorrect and Tied Nearest-Candidate Cases

The cases where distance alone fails or produces a tie are rare, so the next step is to find exactly where they occur. This helps determine whether they are scattered randomly across the dataset or concentrated in specific movies and time points.

If these difficult cases cluster together, there may be a systematic reason behind them. Possible causes include crowded local cell arrangements, unusual movement patterns, nearby divisions, global image shifts, or irregularities in the supplied annotations. Identifying these patterns can help show where distance-based linking struggles and what additional information may be useful for improving the tracker.

In [ ]:
movie_ambiguity_summary = (
    nontrivial_link_ambiguity
    .groupby(
        [
            "sample_id",
            "group",
            "is_test_movie",
        ]
    )
    .agg(
        n_links_with_alternatives=(
            "edge_row_id",
            "size",
        ),
        n_incorrect_candidate_closer=(
            "ambiguity_status",
            lambda values:
                values.eq(
                    "Incorrect candidate closer"
                ).sum(),
        ),
        n_tied_nearest=(
            "ambiguity_status",
            lambda values:
                values.eq("Tied nearest").sum(),
        ),
        minimum_distance_margin=(
            "distance_margin_um",
            "min",
        ),
        maximum_true_target_rank=(
            "true_target_rank",
            "max",
        ),
    )
    .reset_index()
)

movie_ambiguity_summary["n_ambiguous_or_incorrect"] = (
    movie_ambiguity_summary[
        "n_incorrect_candidate_closer"
    ]
    + movie_ambiguity_summary["n_tied_nearest"]
)

movie_ambiguity_summary[
    "ambiguous_or_incorrect_percentage"
] = (
    100
    * movie_ambiguity_summary[
        "n_ambiguous_or_incorrect"
    ]
    / movie_ambiguity_summary[
        "n_links_with_alternatives"
    ]
)

movies_with_ambiguity = (
    movie_ambiguity_summary.loc[
        movie_ambiguity_summary[
            "n_ambiguous_or_incorrect"
        ] > 0
    ]
    .sort_values(
        [
            "n_ambiguous_or_incorrect",
            "ambiguous_or_incorrect_percentage",
        ],
        ascending=False,
    )
)

print(
    "Movies with at least one incorrect or tied nearest candidate:",
    len(movies_with_ambiguity),
)

display(
    movies_with_ambiguity.head(15).round(3)
)

In [ ]:
affected_movie_overview = (
    movies_with_ambiguity
    .groupby("group")
    .agg(
        n_affected_movies=(
            "sample_id",
            "nunique",
        ),
        total_incorrect_candidate_closer=(
            "n_incorrect_candidate_closer",
            "sum",
        ),
        total_tied_nearest=(
            "n_tied_nearest",
            "sum",
        ),
        total_problematic_links=(
            "n_ambiguous_or_incorrect",
            "sum",
        ),
        median_problematic_links_per_affected_movie=(
            "n_ambiguous_or_incorrect",
            "median",
        ),
        maximum_problematic_links_in_one_movie=(
            "n_ambiguous_or_incorrect",
            "max",
        ),
        maximum_problematic_percentage=(
            "ambiguous_or_incorrect_percentage",
            "max",
        ),
    )
    .round(3)
)

top_15_problematic_links = (
    movies_with_ambiguity
    .head(15)["n_ambiguous_or_incorrect"]
    .sum()
)

display(affected_movie_overview)

print(
    "\nProblematic links in the 15 most affected movies:",
    top_15_problematic_links,
)

print(
    "Percentage of all problematic links in those movies:",
    round(
        100
        * top_15_problematic_links
        / movies_with_ambiguity[
            "n_ambiguous_or_incorrect"
        ].sum(),
        3,
    ),
)

### Incorrect Nearest Candidates Cluster in Particular Movies

Incorrect or tied nearest candidates appear in 57 movies, so the problem is not limited to one unusual movie. Six of these movies are from `44b6`, containing 16 problematic links, while 51 are from `6bba`, containing 156.

However, the failures are not evenly spread across those movies. The 15 most affected movies contain 98 of the 172 problematic links, or about 57.0%. Even within these affected movies, the overall percentage of problematic links remains low, with the highest rate in any movie being about 3.03%.

Two movies that were already identified as movement anomalies also appear here. `6bba_f20478e9`, which contains the extreme global Z-shift, has the largest negative distance margin and the most difficult case, where the true target ranks sixth. `6bba_e5e44988`, which contains unusually large Y-axis movements, is also among the movies with the most problematic links.

The fact that these movement-anomaly movies reappear suggests that some distance-based linking failures may be related to larger or unusual frame-to-frame movement rather than crowding alone. This supports testing whether motion history or estimated global frame movement, used alongside raw source-to-candidate distance, can improve the difficult cases.

### Measure Continuation-Link Ambiguity in the Test Movies

In [ ]:
test_link_ambiguity_summary = (
    link_ambiguity.loc[
        link_ambiguity["is_test_movie"]
    ]
    .groupby(
        [
            "sample_id",
            "group",
        ]
    )
    .agg(
        n_continuation_links=(
            "edge_row_id",
            "size",
        ),
        mean_candidates=(
            "n_candidates",
            "mean",
        ),
        median_candidates=(
            "n_candidates",
            "median",
        ),
        links_with_alternatives=(
            "n_candidates",
            lambda values: values.ge(2).sum(),
        ),
        n_only_candidate=(
            "ambiguity_status",
            lambda values:
                values.eq("Only candidate").sum(),
        ),
        n_uniquely_nearest=(
            "ambiguity_status",
            lambda values:
                values.eq("Uniquely nearest").sum(),
        ),
        n_incorrect_candidate_closer=(
            "ambiguity_status",
            lambda values:
                values.eq(
                    "Incorrect candidate closer"
                ).sum(),
        ),
        n_tied_nearest=(
            "ambiguity_status",
            lambda values:
                values.eq("Tied nearest").sum(),
        ),
        minimum_distance_margin=(
            "distance_margin_um",
            "min",
        ),
        maximum_true_target_rank=(
            "true_target_rank",
            "max",
        ),
    )
    .reset_index()
)

test_link_ambiguity_summary[
    "problematic_percentage_among_links_with_alternatives"
] = np.where(
    test_link_ambiguity_summary[
        "links_with_alternatives"
    ] > 0,
    100
    * (
        test_link_ambiguity_summary[
            "n_incorrect_candidate_closer"
        ]
        + test_link_ambiguity_summary[
            "n_tied_nearest"
        ]
    )
    / test_link_ambiguity_summary[
        "links_with_alternatives"
    ],
    np.nan,
)

display(test_link_ambiguity_summary.round(3))

### Test-Movie Continuation Links Are Unambiguous

Using nearest distance, the supplied true target is correctly identified for every continuation link across all four test movies.

All links in `44b6_0113de3b` have only one supplied candidate, so there is no alternative target to choose from. The other three test movies contain genuine alternatives, but the true target is still uniquely the closest candidate in every case.

The closest calls occur in the two `6bba` test movies, where the smallest distance margins are approximately 2.34–3.63 µm. This means that, in their most ambiguous links, the nearest incorrect candidate is only about 2.34–3.63 µm farther away than the true target. These links are therefore less clearly separated than those in `44b6_0b24845f`, although distance alone still makes the correct choice.

Overall, the test movies do not contain any cases where an incorrect supplied candidate is closer than the true target. However, this is still an optimistic result because the candidate set includes only the sparse supplied annotations. A real detection model would have to choose between more biological cells and could also introduce false positives or localization errors.

## Examine Division Geometry and Candidate Ambiguity

Division linking is different from normal continuation linking. For a continuation, the goal is to identify the one node that continues the cell’s trajectory. During a division, one parent cell connects to two daughter cells, so both daughters need to be identified correctly.

The **continuation links provide the main baseline for comparison**. In the previous analysis, more than 99.8% of continuation targets with genuine alternatives were uniquely nearest to their source. Division links can now be compared against this baseline to see whether daughter cells are farther from their parent, less clearly separated from incorrect candidates, or harder to identify using distance alone.

Because divisions are also scored separately in the competition, we will examine:

- the distance from the parent to each daughter, compared with normal continuation movement distances;
- the separation between the two daughters;
- the directions of the daughters relative to the parent;
- and the ranks of both daughters among other supplied next-frame candidates, compared with the near-perfect nearest-candidate performance of continuation links.

These comparisons will show whether division links create a different or more ambiguous spatial problem than normal continuation links and whether distance alone is enough to identify both daughters.


### Build a Division-Event Table

Each supplied division appears as two rows in the edge table because the same parent cell has one outgoing edge to each of its two daughter cells.

These two edges are combined into a single row so that each row represents one complete division event: one parent and its two daughters. This makes it possible to analyze the two daughter cells together rather than treating them as separate movements.

The daughters are labeled **daughter 1** and **daughter 2** based only on their node IDs to keep the table consistent. These labels do not indicate any biological difference, spatial order, or importance between the two daughters.

In [ ]:
division_edges = (
    edge_summary.loc[
        edge_summary["edge_type"].eq("Division edge")
    ]
    .sort_values(
        [
            "sample_id",
            "source_id",
            "target_id",
        ]
    )
    .copy()
)

division_records = []

for (
    sample_id,
    division_node_id,
), event_edges in division_edges.groupby(
    ["sample_id", "source_id"],
    sort=False,
):
    event_edges = event_edges.sort_values(
        "target_id"
    )

    daughter_1 = event_edges.iloc[0]
    daughter_2 = event_edges.iloc[1]

    division_records.append(
        {
            "sample_id": sample_id,
            "group": daughter_1["group"],
            "is_test_movie":
                daughter_1["is_test_movie"],
            "division_node_id":
                division_node_id,
            "division_t":
                daughter_1["source_t"],
            "division_z":
                daughter_1["source_z"],
            "division_y":
                daughter_1["source_y"],
            "division_x":
                daughter_1["source_x"],
            "daughter_1_id":
                daughter_1["target_id"],
            "daughter_1_t":
                daughter_1["target_t"],
            "daughter_1_z":
                daughter_1["target_z"],
            "daughter_1_y":
                daughter_1["target_y"],
            "daughter_1_x":
                daughter_1["target_x"],
            "daughter_1_distance_um":
                daughter_1["distance_um"],
            "daughter_2_id":
                daughter_2["target_id"],
            "daughter_2_t":
                daughter_2["target_t"],
            "daughter_2_z":
                daughter_2["target_z"],
            "daughter_2_y":
                daughter_2["target_y"],
            "daughter_2_x":
                daughter_2["target_x"],
            "daughter_2_distance_um":
                daughter_2["distance_um"],
        }
    )

division_events = pd.DataFrame(
    division_records
)

division_event_checks = pd.Series(
    {
        "number of division edges":
            len(division_edges),
        "number of division events":
            len(division_events),
        "every division has two edges": (
            division_edges
            .groupby(
                [
                    "sample_id",
                    "source_id",
                ]
            )
            .size()
            .eq(2)
            .all()
        ),
        "all daughter IDs are distinct": (
            division_events["daughter_1_id"]
            != division_events["daughter_2_id"]
        ).all(),
        "daughter 1 appears one frame later": (
            division_events["daughter_1_t"]
            - division_events["division_t"]
        ).eq(1).all(),
        "daughter 2 appears one frame later": (
            division_events["daughter_2_t"]
            - division_events["division_t"]
        ).eq(1).all(),
    },
    name="value",
)

display(division_event_checks.to_frame())

display(
    division_events["group"]
    .value_counts()
    .rename_axis("group")
    .to_frame("n_division_events")
)

display(division_events.head())

### Calculate Division Geometry

Four geometric measurements are calculated for each division to describe how the two daughters are positioned relative to their parent:

- **daughter separation**: the physical distance between the two daughter cell centers;
- **opening angle**: the angle between the two directions from the parent to each daughter;
- **daughter-midpoint offset**: the distance from the parent to the midpoint between the two daughters, showing how centered the parent is relative to the daughter pair;
- **relative distance asymmetry**: how different the two parent-to-daughter distances are relative to their total distance.

An opening angle near 0° means both daughters appear in roughly the same direction from the parent, while an angle near 180° means they appear on opposite sides. Relative distance asymmetry ranges from 0 to 1. A value near 0 means the daughters are about equally far from the parent, while a value closer to 1 means one daughter is much farther away than the other.

These measurements help describe what a supplied division normally looks like and can later be compared with unrelated pairs of next-frame candidates. This may help determine whether division geometry provides useful information beyond distance alone. Daughter separation is especially useful because a shared global shift between frames moves both daughters together without changing the distance between them.

In [ ]:
daughter_1_vectors = np.column_stack(
    [
        (
            division_events["daughter_1_z"]
            - division_events["division_z"]
        ) * 1.625,
        (
            division_events["daughter_1_y"]
            - division_events["division_y"]
        ) * 0.40625,
        (
            division_events["daughter_1_x"]
            - division_events["division_x"]
        ) * 0.40625,
    ]
)

daughter_2_vectors = np.column_stack(
    [
        (
            division_events["daughter_2_z"]
            - division_events["division_z"]
        ) * 1.625,
        (
            division_events["daughter_2_y"]
            - division_events["division_y"]
        ) * 0.40625,
        (
            division_events["daughter_2_x"]
            - division_events["division_x"]
        ) * 0.40625,
    ]
)

daughter_1_lengths = np.linalg.norm(
    daughter_1_vectors,
    axis=1,
)

daughter_2_lengths = np.linalg.norm(
    daughter_2_vectors,
    axis=1,
)

daughter_separation_vectors = (
    daughter_2_vectors - daughter_1_vectors
)

division_events["daughter_separation_um"] = (
    np.linalg.norm(
        daughter_separation_vectors,
        axis=1,
    )
)

daughter_vector_dot_products = (
    daughter_1_vectors * daughter_2_vectors
).sum(axis=1)

daughter_vector_cosines = (
    daughter_vector_dot_products
    / (
        daughter_1_lengths
        * daughter_2_lengths
    )
)

daughter_vector_cosines = np.clip(
    daughter_vector_cosines,
    -1,
    1,
)

division_events["opening_angle_degrees"] = np.degrees(
    np.arccos(daughter_vector_cosines)
)

# The midpoint vector is the average of the two
# parent-to-daughter displacement vectors.
daughter_midpoint_vectors = (
    daughter_1_vectors + daughter_2_vectors
) / 2

division_events["daughter_midpoint_offset_um"] = (
    np.linalg.norm(
        daughter_midpoint_vectors,
        axis=1,
    )
)

division_events["relative_distance_asymmetry"] = (
    np.abs(
        daughter_1_lengths
        - daughter_2_lengths
    )
    / (
        daughter_1_lengths
        + daughter_2_lengths
    )
)

division_geometry_checks = pd.Series(
    {
        "number of division events":
            len(division_events),
        "daughter 1 lengths match stored distances":
            np.allclose(
                daughter_1_lengths,
                division_events[
                    "daughter_1_distance_um"
                ],
            ),
        "daughter 2 lengths match stored distances":
            np.allclose(
                daughter_2_lengths,
                division_events[
                    "daughter_2_distance_um"
                ],
            ),
        "all geometric measurements are finite": (
            np.isfinite(
                division_events[
                    [
                        "daughter_separation_um",
                        "opening_angle_degrees",
                        "daughter_midpoint_offset_um",
                        "relative_distance_asymmetry",
                    ]
                ].to_numpy()
            ).all()
        ),
        "opening angles lie between 0 and 180":
            division_events[
                "opening_angle_degrees"
            ].between(0, 180).all(),
        "relative asymmetry lies between 0 and 1":
            division_events[
                "relative_distance_asymmetry"
            ].between(0, 1).all(),
    },
    name="value",
)

display(division_geometry_checks.to_frame())

display(
    division_events[
        [
            "sample_id",
            "group",
            "division_t",
            "daughter_1_distance_um",
            "daughter_2_distance_um",
            "daughter_separation_um",
            "opening_angle_degrees",
            "daughter_midpoint_offset_um",
            "relative_distance_asymmetry",
        ]
    ].head().round(3)
)

### Summarise Division Geometry by Dataset Group

Division geometry is summarized separately for `44b6` and `6bba` to see whether the typical spatial pattern of a division differs between the two groups.

There are 26 supplied divisions in `44b6` compared with 125 in `6bba`. Because the `44b6` sample is much smaller, its medians, percentiles, and percentages are based on fewer division events and may be more affected by individual cases. Any apparent group differences should therefore be interpreted cautiously, especially for `44b6`.

In [ ]:
division_geometry_summary = (
    division_events
    .groupby("group")
    .agg(
        n_division_events=(
            "division_node_id",
            "size",
        ),
        first_quartile_daughter_separation=(
            "daughter_separation_um",
            first_quartile,
        ),
        median_daughter_separation=(
            "daughter_separation_um",
            "median",
        ),
        mean_daughter_separation=(
            "daughter_separation_um",
            "mean",
        ),
        third_quartile_daughter_separation=(
            "daughter_separation_um",
            third_quartile,
        ),
        median_opening_angle=(
            "opening_angle_degrees",
            "median",
        ),
        mean_opening_angle=(
            "opening_angle_degrees",
            "mean",
        ),
        percentage_opening_angle_above_90=(
            "opening_angle_degrees",
            lambda values: 100 * values.gt(90).mean(),
        ),
        percentage_opening_angle_above_150=(
            "opening_angle_degrees",
            lambda values: 100 * values.gt(150).mean(),
        ),
        median_midpoint_offset=(
            "daughter_midpoint_offset_um",
            "median",
        ),
        median_relative_asymmetry=(
            "relative_distance_asymmetry",
            "median",
        ),
        percentage_relative_asymmetry_above_0_5=(
            "relative_distance_asymmetry",
            lambda values: 100 * values.gt(0.5).mean(),
        ),
    )
    .round(3)
)

display(division_geometry_summary)

### Division-Geometry Findings

The two daughter cells are usually clearly separated from each other, with a median separation of about 8.98 µm in `44b6` and 11.47 µm in `6bba`.

Their directional pattern is also very similar between the two groups. The median opening angle is around 135–140°, and about 89% of divisions have an opening angle above 90°. This means the two daughters usually appear in noticeably different directions from the parent rather than both moving in the same direction.

The parent also tends to remain fairly centered between the two daughters. The median distance between the parent and the daughters’ midpoint is only about 2 µm. Relative distance asymmetry is also similar between the groups, with a median of about 0.23, meaning the two daughters are usually at reasonably similar distances from the parent.

Together, these patterns suggest that supplied divisions have a recognizable spatial structure: two separated daughters that usually appear in different directions, with the parent located relatively close to their midpoint. This geometry could therefore provide useful information for distinguishing a true daughter pair from unrelated next-frame candidates.

Strong distance asymmetry above 0.5 is more common in `44b6`, meaning one daughter is much farther from the parent than the other in more of its divisions. However, `44b6` contains only 26 supplied divisions compared with 125 in `6bba`, so this difference should be interpreted cautiously.

### Examine Test-Movie Division Geometry

In [ ]:
division_percentile_variables = [
    "daughter_separation_um",
    "opening_angle_degrees",
    "daughter_midpoint_offset_um",
    "relative_distance_asymmetry",
]

for variable in division_percentile_variables:
    division_events[
        f"{variable}_group_percentile"
    ] = (
        division_events
        .groupby("group")[variable]
        .rank(pct=True)
        * 100
    )

test_division_events = (
    division_events.loc[
        division_events["is_test_movie"]
    ]
    .sort_values(
        [
            "sample_id",
            "division_t",
        ]
    )
)

display(
    test_division_events[
        [
            "sample_id",
            "division_node_id",
            "division_t",
            "daughter_1_distance_um",
            "daughter_2_distance_um",
            "daughter_separation_um",
            "daughter_separation_um_group_percentile",
            "opening_angle_degrees",
            "opening_angle_degrees_group_percentile",
            "daughter_midpoint_offset_um",
            "daughter_midpoint_offset_um_group_percentile",
            "relative_distance_asymmetry",
            "relative_distance_asymmetry_group_percentile",
        ]
    ].round(3)
)

### Test-Movie Division-Geometry Findings

The three supplied divisions in `6bba_05db0fb1` show that division geometry can vary quite a bit even within the same movie.

The division at timepoint 24 is relatively balanced. The daughters are about 12.49 µm apart; both are at similar distances from the parent, and the parent remains close to the midpoint between them.

The divisions at timepoints 52 and 62 look different. Both have smaller daughter separations and larger differences between their parent-to-daughter distances. The timepoint-62 division is especially asymmetric, falling around the 91st percentile of relative distance asymmetry within `6bba`. In other words, it is more asymmetric than about 91% of the supplied `6bba` divisions. At the same time, its very large opening angle means the two daughters still appear in strongly different directions from the parent.

There are only three supplied divisions in this test movie, which is far too few to define what test-time divisions will generally look like. However, they show why a division model cannot depend on one strict geometric pattern. It needs to handle both balanced divisions and cases where one daughter is much farther from the parent than the other.

### Build Candidate Sets for Division Events

For each division parent, every supplied node in the following frame is treated as a possible candidate, just like in the continuation-link analysis.

The difference is that a division has **two correct targets instead of one**. Both known daughters are therefore treated as correct candidates. When evaluating one daughter, the other daughter is not counted as an incorrect alternative because both belong to the same true division event.

The main question is whether the two supplied daughters can be identified from the rest of the candidate set. In particular, we test whether any **non-daughter candidate** is closer to the parent than either of the two true daughters. If so, selecting the two nearest candidates by distance alone would fail to recover the complete daughter pair.

In [ ]:
division_candidate_sources = (
    division_events
    .reset_index()
    .rename(
        columns={
            "index": "division_event_id",
            "daughter_1_t": "candidate_t",
        }
    )
)

division_candidates = division_candidate_sources.merge(
    candidate_nodes,
    on=["sample_id", "candidate_t"],
    how="left",
    validate="many_to_many",
)

division_candidates["candidate_delta_z_um"] = (
    division_candidates["candidate_z"]
    - division_candidates["division_z"]
) * 1.625

division_candidates["candidate_delta_y_um"] = (
    division_candidates["candidate_y"]
    - division_candidates["division_y"]
) * 0.40625

division_candidates["candidate_delta_x_um"] = (
    division_candidates["candidate_x"]
    - division_candidates["division_x"]
) * 0.40625

division_candidates["candidate_distance_um"] = np.sqrt(
    division_candidates["candidate_delta_z_um"] ** 2
    + division_candidates["candidate_delta_y_um"] ** 2
    + division_candidates["candidate_delta_x_um"] ** 2
)

division_candidates["is_daughter_1"] = (
    division_candidates["candidate_node_id"]
    == division_candidates["daughter_1_id"]
)

division_candidates["is_daughter_2"] = (
    division_candidates["candidate_node_id"]
    == division_candidates["daughter_2_id"]
)

division_candidates["is_true_daughter"] = (
    division_candidates["is_daughter_1"]
    | division_candidates["is_daughter_2"]
)

division_candidate_counts = (
    division_candidates
    .groupby("division_event_id")
    .size()
)

true_daughter_counts = (
    division_candidates
    .groupby("division_event_id")[
        "is_true_daughter"
    ]
    .sum()
)

daughter_1_candidate_rows = (
    division_candidates.loc[
        division_candidates["is_daughter_1"]
    ]
    .sort_values("division_event_id")
)

daughter_2_candidate_rows = (
    division_candidates.loc[
        division_candidates["is_daughter_2"]
    ]
    .sort_values("division_event_id")
)

division_candidate_checks = pd.Series(
    {
        "number of division events":
            division_candidates[
                "division_event_id"
            ].nunique(),
        "every event has at least two candidates":
            division_candidate_counts.ge(2).all(),
        "every event contains both daughters":
            true_daughter_counts.eq(2).all(),
        "daughter 1 candidate distances match":
            np.allclose(
                daughter_1_candidate_rows[
                    "candidate_distance_um"
                ],
                daughter_1_candidate_rows[
                    "daughter_1_distance_um"
                ],
            ),
        "daughter 2 candidate distances match":
            np.allclose(
                daughter_2_candidate_rows[
                    "candidate_distance_um"
                ],
                daughter_2_candidate_rows[
                    "daughter_2_distance_um"
                ],
            ),
        "minimum candidates for one division":
            division_candidate_counts.min(),
        "maximum candidates for one division":
            division_candidate_counts.max(),
    },
    name="value",
)

display(division_candidate_checks.to_frame())

example_division_event_id = (
    division_candidate_counts.loc[
        division_candidate_counts > 2
    ]
    .index[0]
)

display(
    division_candidates.loc[
        division_candidates["division_event_id"]
        == example_division_event_id,
        [
            "sample_id",
            "division_node_id",
            "division_t",
            "candidate_node_id",
            "candidate_distance_um",
            "is_daughter_1",
            "is_daughter_2",
        ],
    ]
    .sort_values("candidate_distance_um")
    .round(3)
)

### Assess Whether Both Daughters Are the Best Candidates

A division-linking method has to identify **both** daughter cells correctly, rather than just finding one target. We therefore test whether both supplied daughters are closer to the parent than every non-daughter candidate in the following frame.

The farther of the two daughters is especially important. If a non-daughter is closer than that daughter, simply choosing the two nearest candidates would select the wrong pair.

For each division, the **division-pair margin** is calculated as:

**division-pair margin = nearest non-daughter distance − farther daughter distance**

- A **positive margin** means both true daughters are closer to the parent than every non-daughter candidate.
- A **negative margin** means at least one non-daughter is closer than the farther true daughter, so choosing the two nearest candidates would fail.
- A **margin close to zero** means the correct daughter pair is only slightly better separated from the alternatives, making the decision more sensitive to small errors.
- The margin is **undefined** when the two daughters are the only supplied candidates because there is no incorrect candidate to compare against.

This provides a direct division equivalent to the distance-margin analysis used for continuation links, but now the correct decision requires recovering **both** true targets.

In [ ]:
# Begin with one row per division event.
division_pair_ambiguity = (
    division_candidate_sources[
        [
            "division_event_id",
            "sample_id",
            "group",
            "is_test_movie",
            "division_node_id",
            "division_t",
            "daughter_1_distance_um",
            "daughter_2_distance_um",
        ]
    ]
    .drop_duplicates("division_event_id")
    .copy()
)

division_pair_ambiguity["farther_daughter_distance_um"] = (
    division_pair_ambiguity[
        [
            "daughter_1_distance_um",
            "daughter_2_distance_um",
        ]
    ].max(axis=1)
)

# Count all candidates and identify the nearest non-daughter.
candidate_counts = (
    division_candidates
    .groupby("division_event_id")
    .size()
    .rename("n_candidates")
)

non_daughter_summary = (
    division_candidates.loc[
        ~division_candidates["is_true_daughter"]
    ]
    .groupby("division_event_id")
    .agg(
        n_non_daughter_candidates=(
            "candidate_node_id",
            "size",
        ),
        nearest_non_daughter_distance_um=(
            "candidate_distance_um",
            "min",
        ),
    )
)

division_pair_ambiguity = (
    division_pair_ambiguity
    .merge(
        candidate_counts,
        on="division_event_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        non_daughter_summary,
        on="division_event_id",
        how="left",
        validate="one_to_one",
    )
)

division_pair_ambiguity["n_non_daughter_candidates"] = (
    division_pair_ambiguity[
        "n_non_daughter_candidates"
    ]
    .fillna(0)
    .astype(int)
)

division_pair_ambiguity["division_pair_margin_um"] = (
    division_pair_ambiguity[
        "nearest_non_daughter_distance_um"
    ]
    - division_pair_ambiguity[
        "farther_daughter_distance_um"
    ]
)

# Compare every non-daughter with the farther true daughter.
comparison_rows = division_candidates.merge(
    division_pair_ambiguity[
        [
            "division_event_id",
            "farther_daughter_distance_um",
        ]
    ],
    on="division_event_id",
    how="left",
    validate="many_to_one",
)

comparison_rows["distance_difference_um"] = (
    comparison_rows["candidate_distance_um"]
    - comparison_rows["farther_daughter_distance_um"]
)

tolerance = 1e-9

comparison_rows["non_daughter_closer"] = (
    ~comparison_rows["is_true_daughter"]
    & (
        comparison_rows["distance_difference_um"]
        < -tolerance
    )
)

comparison_rows["non_daughter_tied"] = (
    ~comparison_rows["is_true_daughter"]
    & np.isclose(
        comparison_rows["distance_difference_um"],
        0,
        atol=tolerance,
        rtol=0,
    )
)

problem_counts = (
    comparison_rows
    .groupby("division_event_id")
    .agg(
        n_non_daughters_closer=(
            "non_daughter_closer",
            "sum",
        ),
        n_non_daughters_tied=(
            "non_daughter_tied",
            "sum",
        ),
    )
)

division_pair_ambiguity = division_pair_ambiguity.merge(
    problem_counts,
    on="division_event_id",
    how="left",
    validate="one_to_one",
)

division_pair_ambiguity[
    [
        "n_non_daughters_closer",
        "n_non_daughters_tied",
    ]
] = (
    division_pair_ambiguity[
        [
            "n_non_daughters_closer",
            "n_non_daughters_tied",
        ]
    ]
    .fillna(0)
    .astype(int)
)

division_pair_ambiguity["ambiguity_status"] = np.select(
    [
        division_pair_ambiguity[
            "n_non_daughter_candidates"
        ].eq(0),
        division_pair_ambiguity[
            "n_non_daughters_closer"
        ].gt(0),
        division_pair_ambiguity[
            "n_non_daughters_tied"
        ].gt(0),
    ],
    [
        "Only daughters",
        "Non-daughter closer than farther daughter",
        "Non-daughter tied with farther daughter",
    ],
    default=(
        "Both daughters closer than all non-daughters"
    ),
)

division_pair_checks = pd.Series(
    {
        "number of division events":
            len(division_pair_ambiguity),
        "candidate rows accounted for":
            division_pair_ambiguity[
                "n_candidates"
            ].sum() == len(division_candidates),
        "only-daughter margins are missing":
            division_pair_ambiguity.loc[
                division_pair_ambiguity[
                    "ambiguity_status"
                ].eq("Only daughters"),
                "division_pair_margin_um",
            ].isna().all(),
        "clear pair margins are positive":
            division_pair_ambiguity.loc[
                division_pair_ambiguity[
                    "ambiguity_status"
                ].eq(
                    "Both daughters closer than all non-daughters"
                ),
                "division_pair_margin_um",
            ].gt(0).all(),
        "closer non-daughter margins are negative":
            division_pair_ambiguity.loc[
                division_pair_ambiguity[
                    "ambiguity_status"
                ].eq(
                    "Non-daughter closer than farther daughter"
                ),
                "division_pair_margin_um",
            ].lt(0).all(),
    },
    name="value",
)

display(division_pair_checks.to_frame())

display(
    division_pair_ambiguity[
        "ambiguity_status"
    ]
    .value_counts()
    .rename("n_division_events")
    .to_frame()
)

# Display the divisions with the smallest margins.
display(
    division_pair_ambiguity.loc[
        division_pair_ambiguity[
            "division_pair_margin_um"
        ].notna(),
        [
            "sample_id",
            "group",
            "is_test_movie",
            "division_node_id",
            "division_t",
            "n_candidates",
            "farther_daughter_distance_um",
            "nearest_non_daughter_distance_um",
            "division_pair_margin_um",
            "n_non_daughters_closer",
            "ambiguity_status",
        ],
    ]
    .sort_values("division_pair_margin_um")
    .head(15)
    .round(3)
)

### Division Candidate Findings

Distance alone identifies both supplied daughters extremely well. In 145 of the 151 division events, both daughters are closer to the parent than every non-daughter candidate. Another five divisions contain only the two true daughters in the following frame.

These five divisions are excluded from the **99.3% candidate-comparison calculation** because there is no incorrect candidate for the daughters to compete against. Choosing the two nearest supplied candidates would automatically return the two daughters, regardless of their distances, because they are the only two options. Including these cases would make distance-based linking appear more successful without actually testing whether distance can distinguish the daughters from an incorrect alternative.

This leaves **146 divisions with at least one non-daughter candidate**. Of those, both true daughters are the two closest candidates in 145:

**145 ÷ 146 × 100 ≈ 99.3%**

There is only **one actual failure**. In `6bba_debd7bfa`, a non-daughter candidate is 2.358 µm closer to the parent than the farther true daughter. Simply selecting the two nearest candidates would therefore choose the wrong pair.

There is also one very close call in `6bba_afb141ff`, where the division-pair margin is positive by only 0.018 µm. The correct daughters still rank first and second, but such a small difference could easily be reversed by a minor localization error.

Overall, distance is highly informative for division linking when genuine alternatives exist. However, the few small or negative margins show where motion history, cell appearance, or division geometry could become important.

As with the continuation-link analysis, this is still an optimistic result because the candidate pool contains only sparse supplied annotations rather than every biological cell or every detection produced by a real model.

### Compare Division-Pair Ambiguity Across Dataset Groups

The previous analysis showed whether both true daughters are closer to the parent than every non-daughter candidate. We now compare `44b6` and `6bba` to see whether one group has more clearly separated daughter pairs than the other.

The **division-pair margin** is the main comparison. A larger positive margin means the farther true daughter still has a clear distance advantage over the nearest non-daughter. A margin close to zero means the decision is fragile, while a negative margin means a non-daughter is closer than one of the true daughters and distance alone would select the wrong pair.

The lower end of the margin distribution is especially important because it contains the most difficult division events. The median shows how well separated a typical division is, while the lower percentiles show how close the more ambiguous cases come to failure.

Any percentages based on the division-pair margin include **only divisions with at least one non-daughter candidate**. Divisions where the two daughters are the only supplied candidates are excluded because there is no incorrect alternative to compare against, so a division-pair margin cannot be calculated.

In [ ]:
division_group_analysis = division_pair_ambiguity.copy()

division_group_analysis["has_non_daughter_alternative"] = (
    division_group_analysis[
        "n_non_daughter_candidates"
    ].gt(0)
)

division_group_analysis["both_daughters_closest"] = (
    division_group_analysis[
        "ambiguity_status"
    ].eq(
        "Both daughters closer than all non-daughters"
    )
)

division_group_analysis["problematic_pair"] = (
    division_group_analysis[
        "n_non_daughters_closer"
    ].gt(0)
    | division_group_analysis[
        "n_non_daughters_tied"
    ].gt(0)
)

division_group_analysis["margin_at_most_1_um"] = (
    division_group_analysis[
        "has_non_daughter_alternative"
    ]
    & division_group_analysis[
        "division_pair_margin_um"
    ].le(1)
)

division_group_analysis["margin_at_most_5_um"] = (
    division_group_analysis[
        "has_non_daughter_alternative"
    ]
    & division_group_analysis[
        "division_pair_margin_um"
    ].le(5)
)

division_group_summary = (
    division_group_analysis
    .groupby("group")
    .agg(
        n_division_events=(
            "division_event_id",
            "size",
        ),
        events_with_non_daughter_alternatives=(
            "has_non_daughter_alternative",
            "sum",
        ),
        events_with_only_daughters=(
            "has_non_daughter_alternative",
            lambda values: (~values).sum(),
        ),
        events_with_both_daughters_closest=(
            "both_daughters_closest",
            "sum",
        ),
        problematic_events=(
            "problematic_pair",
            "sum",
        ),
        minimum_margin=(
            "division_pair_margin_um",
            "min",
        ),
        first_quartile_margin=(
            "division_pair_margin_um",
            lambda values: values.quantile(0.25),
        ),
        median_margin=(
            "division_pair_margin_um",
            "median",
        ),
        third_quartile_margin=(
            "division_pair_margin_um",
            lambda values: values.quantile(0.75),
        ),
        events_with_margin_at_most_1_um=(
            "margin_at_most_1_um",
            "sum",
        ),
        events_with_margin_at_most_5_um=(
            "margin_at_most_5_um",
            "sum",
        ),
    )
)

division_group_summary[
    "percentage_both_daughters_closest"
] = (
    100
    * division_group_summary[
        "events_with_both_daughters_closest"
    ]
    / division_group_summary[
        "events_with_non_daughter_alternatives"
    ]
)

division_group_summary[
    "problematic_percentage"
] = (
    100
    * division_group_summary[
        "problematic_events"
    ]
    / division_group_summary[
        "events_with_non_daughter_alternatives"
    ]
)

division_group_summary[
    "percentage_margin_at_most_1_um"
] = (
    100
    * division_group_summary[
        "events_with_margin_at_most_1_um"
    ]
    / division_group_summary[
        "events_with_non_daughter_alternatives"
    ]
)

division_group_summary[
    "percentage_margin_at_most_5_um"
] = (
    100
    * division_group_summary[
        "events_with_margin_at_most_5_um"
    ]
    / division_group_summary[
        "events_with_non_daughter_alternatives"
    ]
)

display(division_group_summary.round(3))

### Division-Pair Ambiguity Findings

Both true daughters are closer than every non-daughter candidate in all **21 assessable `44b6` divisions (100%)** and in **124 of the 125 assessable `6bba` divisions (99.2%)**. Here, “assessable” means that the division contains at least one non-daughter candidate, allowing the daughter pair to actually be compared against an incorrect alternative.

The distance advantage is generally larger in `44b6`. Its median division-pair margin is 31.395 µm, compared with 15.555 µm in `6bba`, and the same pattern appears at both quartiles. This means the true daughter pair is typically more clearly separated from the nearest incorrect candidate in the supplied `44b6` annotations.

The difference is especially noticeable among the most difficult cases. No assessable `44b6` division has a margin at or below 1 µm, while three `6bba` divisions do. These include the single negative-margin event where distance alone would select the wrong second daughter. Using a wider 5 µm threshold, seven `6bba` divisions and one `44b6` division have small margins.

However, this does **not** necessarily mean that `44b6` divisions are biologically easier to link. There are 26 supplied `44b6` divisions in total, but five are excluded from the margin analysis because the two daughters are the only supplied candidates. This leaves only 21 assessable `44b6` divisions compared with all 125 `6bba` divisions. `44b6` also has generally sparser supplied annotations, meaning there are fewer competing candidates that could produce a small margin.

Overall, selecting the two closest candidates provides a strong baseline for division linking in both groups. Division geometry, motion history, and image appearance are likely to be most useful for the small number of events where the distance margin is very small or negative.

### Place Test-Movie Divisions in the Margin Distribution

Only `6bba_05db0fb1` contains supplied division events among the four test movies. To see how difficult these test divisions are compared with the training data, their division-pair margins are compared with the other **assessable `6bba` divisions**.

Here, the percentile shows where each test division’s margin falls relative to the distribution of `6bba` divisions that contain at least one non-daughter candidate. Divisions with only the two true daughters are not included because their margin is undefined.

A **high percentile** means the test division has a larger margin than most other assessable `6bba` divisions, so both daughters are clearly separated from the nearest non-daughter candidate. A **low percentile** means the margin is smaller than most, making the division more ambiguous and potentially more sensitive to localization errors or additional candidate detections.

In [ ]:
division_margin_comparison = (
    division_pair_ambiguity.copy()
)

division_margin_comparison[
    "division_pair_margin_group_percentile"
] = (
    division_margin_comparison
    .groupby("group")[
        "division_pair_margin_um"
    ]
    .rank(
        method="average",
        pct=True,
    )
    * 100
)

test_division_margins = (
    division_margin_comparison.loc[
        division_margin_comparison[
            "is_test_movie"
        ],
        [
            "sample_id",
            "group",
            "division_node_id",
            "division_t",
            "n_candidates",
            "farther_daughter_distance_um",
            "nearest_non_daughter_distance_um",
            "division_pair_margin_um",
            "division_pair_margin_group_percentile",
            "ambiguity_status",
        ],
    ]
    .sort_values(
        [
            "sample_id",
            "division_t",
        ]
    )
)

display(test_division_margins.round(3))

### Test-Movie Division Margins

All three supplied test-movie divisions have both true daughters closer to the parent than every non-daughter candidate.

Their division-pair margins range from 12.936 to 19.501 µm. Two of the divisions fall around the 38th percentile of the assessable `6bba` margin distribution, meaning their margins are larger than about 38% of comparable `6bba` divisions. The third is around the 66th percentile, meaning its margin is larger than about 66% of comparable divisions.

None of the three test divisions are near the extreme low or high end of the distribution. They therefore appear reasonably representative of the supplied `6bba` divisions rather than being unusually easy or unusually difficult based on distance.

Distance alone would correctly select both daughters in all three supplied test-movie divisions. However, this result is based on only three division events from a single test movie and on the sparse supplied candidate set, so it should not be treated as evidence that all test-time divisions will be this clear.

**Practically, this means selecting the two closest candidates is a strong starting point for division linking.** More complex information such as division geometry, motion history, or cell appearance can then be used as additional evidence, especially when the distance margin is small or negative rather than being required for every division.

### Visualise Division-Pair Margins

This plot shows how division-pair margins vary across movies, with the three supplied test-movie divisions highlighted as red diamonds.

The **zero line** represents the failure boundary. A division below this line has a negative margin, meaning a non-daughter candidate is closer to the parent than the farther true daughter. In that case, simply selecting the two nearest candidates would choose the wrong pair.

The **5 µm line** is used as a descriptive reference for small positive margins. Divisions below 5 µm are still correctly resolved by distance if their margin remains positive, but the true daughter pair has relatively little separation from the nearest incorrect alternative. These cases may therefore be more sensitive to localization error or additional detections.

The 5 µm cutoff is not a biological threshold. It is used to highlight the lower-margin cases where additional information, such as motion history, cell appearance, or division geometry, may be most useful.

In [ ]:
label_offsets = {
    24: (10, -12),
    52: (10, 4),
    62: (10, 12),
}

division_margin_plot_data = (
    division_margin_comparison.loc[
        division_margin_comparison[
            "division_pair_margin_um"
        ].notna()
    ]
)

group_order = ["44b6", "6bba"]
group_palette = {
    "44b6": "tab:blue",
    "6bba": "tab:orange",
}

fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=division_margin_plot_data,
    x="group",
    y="division_pair_margin_um",
    order=group_order,
    hue="group",
    hue_order=group_order,
    palette=group_palette,
    dodge=False,
    legend=False,
    ax=ax,
    showfliers=False
)

sns.stripplot(
    data=division_margin_plot_data,
    x="group",
    y="division_pair_margin_um",
    order=group_order,
    color="black",
    alpha=0.45,
    jitter=0.18,
    size=4,
    ax=ax,
)

test_margin_plot_data = (
    division_margin_plot_data.loc[
        division_margin_plot_data[
            "is_test_movie"
        ]
    ]
)

for row in test_margin_plot_data.itertuples():
    x_position = group_order.index(row.group)

    ax.scatter(
        x_position,
        row.division_pair_margin_um,
        marker="D",
        s=110,
        facecolor="red",
        edgecolor="black",
        zorder=5,
    )

    ax.annotate(
        f"t = {row.division_t}",
        (
            x_position,
            row.division_pair_margin_um,
        ),
        xytext=label_offsets.get(
            row.division_t,
            (10, 0),
        ),
        textcoords="offset points",
        va="center",
        fontsize=9,
    )

ax.axhline(
    0,
    color="red",
    linestyle="--",
    linewidth=1,
    label="Zero margin",
)

ax.axhline(
    5,
    color="grey",
    linestyle=":",
    linewidth=1.5,
    label="5 µm margin",
)

# Add one legend entry for the test markers.
ax.scatter(
    [],
    [],
    marker="D",
    s=90,
    facecolor="red",
    edgecolor="black",
    label="Test-movie division",
)

ax.set_title(
    "Separation of True Daughters from Non-Daughter Candidates"
)
ax.set_xlabel("Dataset group")
ax.set_ylabel("Division-pair margin (µm)")
ax.legend()

plt.tight_layout()
plt.show()

### Division-Pair Margin Plot Findings

Division-pair margins are generally larger in `44b6`. The middle 50% of its margins range from about 20.9 to 51.2 µm, compared with 10.6 to 22.7 µm in `6bba`. There is only limited overlap between these ranges, showing that the typical supplied `44b6` division has a larger distance advantage over incorrect candidates.

The more important difference appears at the lower end of the distribution. `6bba` contains several divisions with margins close to zero, as well as the only division with a negative margin. These are the cases where distance-only linking is most vulnerable because a small localization error could change the selected daughter pair, or, in the negative-margin case, distance already selects the wrong pair.

All three test-movie divisions fall within the middle 50% of the `6bba` distribution. The divisions at `t = 24` and `t = 62` have almost identical margins of about 13 µm, while the division at `t = 52` has a larger margin of about 19.5 µm. This supports the earlier finding that the three test divisions are fairly typical rather than unusually easy or difficult based on distance.

However, the larger margins in `44b6` should not automatically be interpreted as meaning its biological divisions are easier to track. `44b6` has much sparser supplied annotations, which creates fewer competing candidates and can naturally produce larger margins. The group difference may therefore reflect the supplied candidate sets as well as any real difference in division behavior.

### Inspect the Distance-Based Division Failure in Microscopy

Only one supplied division would be linked incorrectly if the two nearest candidates were selected based on distance alone. We therefore inspect this event directly in the microscopy image to see whether the true daughters can be distinguished using visual information.

The plots show local maximum-intensity projections centered on the parent cell:

- **cyan circle**: the parent at the division frame;
- **green circles**: the two true daughters one frame later;
- **red cross**: the nearest non-daughter candidate.

All panels show the same physical region and use the same intensity scale so the images can be compared directly.

The main question is whether the true daughters have a more convincing visual or spatial relationship to the parent than the closer non-daughter. If so, image appearance or local division geometry could help correct the rare cases where distance alone selects the wrong pair.

In [ ]:
from matplotlib.lines import Line2D

difficult_division = (
    division_pair_ambiguity.loc[
        division_pair_ambiguity[
            "ambiguity_status"
        ].eq(
            "Non-daughter closer than farther daughter"
        )
    ]
    .iloc[0]
)

difficult_event_id = difficult_division[
    "division_event_id"
]

difficult_event = (
    division_candidate_sources.loc[
        division_candidate_sources[
            "division_event_id"
        ].eq(difficult_event_id)
    ]
    .drop_duplicates("division_event_id")
    .iloc[0]
)

difficult_event_candidates = (
    division_candidates.loc[
        division_candidates[
            "division_event_id"
        ].eq(difficult_event_id)
    ]
)

nearest_non_daughter = (
    difficult_event_candidates.loc[
        ~difficult_event_candidates[
            "is_true_daughter"
        ]
    ]
    .sort_values("candidate_distance_um")
    .iloc[0]
)

event_locations = pd.DataFrame(
    [
        {
            "role": "Parent",
            "node_id": int(
                difficult_event["division_node_id"]
            ),
            "t": int(
                difficult_event["division_t"]
            ),
            "z": int(
                difficult_event["division_z"]
            ),
            "y": int(
                difficult_event["division_y"]
            ),
            "x": int(
                difficult_event["division_x"]
            ),
            "distance_from_parent_um": 0.0,
        },
        {
            "role": "True daughter 1",
            "node_id": int(
                difficult_event["daughter_1_id"]
            ),
            "t": int(
                difficult_event["candidate_t"]
            ),
            "z": int(
                difficult_event["daughter_1_z"]
            ),
            "y": int(
                difficult_event["daughter_1_y"]
            ),
            "x": int(
                difficult_event["daughter_1_x"]
            ),
            "distance_from_parent_um": difficult_event[
                "daughter_1_distance_um"
            ],
        },
        {
            "role": "True daughter 2",
            "node_id": int(
                difficult_event["daughter_2_id"]
            ),
            "t": int(
                difficult_event["daughter_2_t"]
            ),
            "z": int(
                difficult_event["daughter_2_z"]
            ),
            "y": int(
                difficult_event["daughter_2_y"]
            ),
            "x": int(
                difficult_event["daughter_2_x"]
            ),
            "distance_from_parent_um": difficult_event[
                "daughter_2_distance_um"
            ],
        },
        {
            "role": "Nearest non-daughter",
            "node_id": int(
                nearest_non_daughter[
                    "candidate_node_id"
                ]
            ),
            "t": int(
                nearest_non_daughter["candidate_t"]
            ),
            "z": int(
                nearest_non_daughter["candidate_z"]
            ),
            "y": int(
                nearest_non_daughter["candidate_y"]
            ),
            "x": int(
                nearest_non_daughter["candidate_x"]
            ),
            "distance_from_parent_um":
                nearest_non_daughter[
                    "candidate_distance_um"
                ],
        },
    ]
)

display(event_locations.round(3))

In [ ]:
data_root = Path(
    "/kaggle/input/competitions/"
    "biohub-cell-tracking-during-development"
)

sample_id = difficult_event["sample_id"]

movie_store = zarr.open_group(
    str(
        data_root
        / "train"
        / f"{sample_id}.zarr"
    ),
    mode="r",
)

movie_array = movie_store["0"]

z_scale = 1.625
y_scale = 0.40625
x_scale = 0.40625

parent_t = int(difficult_event["division_t"])
daughter_t = parent_t + 1

parent_z = int(difficult_event["division_z"])
parent_y = int(difficult_event["division_y"])
parent_x = int(difficult_event["division_x"])

# Display approximately 40 × 40 × 40 µm around the parent.
physical_radius_um = 20

z_radius = int(
    np.ceil(physical_radius_um / z_scale)
)
y_radius = int(
    np.ceil(physical_radius_um / y_scale)
)
x_radius = int(
    np.ceil(physical_radius_um / x_scale)
)

z_start = max(0, parent_z - z_radius)
z_stop = min(
    movie_array.shape[1],
    parent_z + z_radius + 1,
)

y_start = max(0, parent_y - y_radius)
y_stop = min(
    movie_array.shape[2],
    parent_y + y_radius + 1,
)

x_start = max(0, parent_x - x_radius)
x_stop = min(
    movie_array.shape[3],
    parent_x + x_radius + 1,
)

projections = {}

for t in [parent_t, daughter_t]:
    local_volume = np.asarray(
        movie_array[
            t,
            z_start:z_stop,
            y_start:y_stop,
            x_start:x_stop,
        ]
    )

    projections[(t, "XY")] = local_volume.max(
        axis=0
    )
    projections[(t, "XZ")] = local_volume.max(
        axis=1
    )
    projections[(t, "YZ")] = local_volume.max(
        axis=2
    )

projection_values = np.concatenate(
    [
        projection.ravel()
        for projection in projections.values()
    ]
)

intensity_min = np.percentile(
    projection_values,
    1,
)
intensity_max = np.percentile(
    projection_values,
    99.5,
)

plot_extents = {
    "XY": [
        (x_start - parent_x) * x_scale,
        (x_stop - parent_x) * x_scale,
        (y_start - parent_y) * y_scale,
        (y_stop - parent_y) * y_scale,
    ],
    "XZ": [
        (x_start - parent_x) * x_scale,
        (x_stop - parent_x) * x_scale,
        (z_start - parent_z) * z_scale,
        (z_stop - parent_z) * z_scale,
    ],
    "YZ": [
        (y_start - parent_y) * y_scale,
        (y_stop - parent_y) * y_scale,
        (z_start - parent_z) * z_scale,
        (z_stop - parent_z) * z_scale,
    ],
}

axis_labels = {
    "XY": ("Relative X (µm)", "Relative Y (µm)"),
    "XZ": ("Relative X (µm)", "Relative Z (µm)"),
    "YZ": ("Relative Y (µm)", "Relative Z (µm)"),
}


def projection_coordinates(row, plane):
    relative_z = (
        row["z"] - parent_z
    ) * z_scale

    relative_y = (
        row["y"] - parent_y
    ) * y_scale

    relative_x = (
        row["x"] - parent_x
    ) * x_scale

    if plane == "XY":
        return relative_x, relative_y

    if plane == "XZ":
        return relative_x, relative_z

    return relative_y, relative_z


fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9),
    constrained_layout=True,
)

planes = ["XY", "XZ", "YZ"]
frames = [parent_t, daughter_t]

for row_index, t in enumerate(frames):
    for column_index, plane in enumerate(planes):
        ax = axes[row_index, column_index]

        image = ax.imshow(
            projections[(t, plane)],
            cmap="gray",
            origin="lower",
            extent=plot_extents[plane],
            vmin=intensity_min,
            vmax=intensity_max,
            aspect="equal",
        )

        locations_at_t = event_locations.loc[
            event_locations["t"].eq(t)
        ]

        for _, location in locations_at_t.iterrows():
            plot_x, plot_y = projection_coordinates(
                location,
                plane,
            )

            if location["role"] == "Parent":
                ax.scatter(
                    plot_x,
                    plot_y,
                    marker="o",
                    s=130,
                    facecolors="none",
                    edgecolors="cyan",
                    linewidths=2,
                    zorder=5,
                )

            elif location["role"].startswith(
                "True daughter"
            ):
                ax.scatter(
                    plot_x,
                    plot_y,
                    marker="o",
                    s=130,
                    facecolors="none",
                    edgecolors="lime",
                    linewidths=2,
                    zorder=5,
                )

            else:
                ax.scatter(
                    plot_x,
                    plot_y,
                    marker="x",
                    s=130,
                    color="red",
                    linewidths=2.5,
                    zorder=6,
                )

        frame_label = (
            "Parent frame"
            if t == parent_t
            else "Next frame"
        )

        ax.set_title(
            f"{frame_label}, t = {t}: {plane}"
        )
        ax.set_xlabel(axis_labels[plane][0])
        ax.set_ylabel(axis_labels[plane][1])

fig.suptitle(
    "Microscopy Around the Distance-Based "
    f"Division Failure in {sample_id}",
    fontsize=14,
)

legend_handles = [
    Line2D(
        [],
        [],
        marker="o",
        linestyle="none",
        markerfacecolor="none",
        markeredgecolor="cyan",
        markeredgewidth=2,
        markersize=10,
        label="Parent",
    ),
    Line2D(
        [],
        [],
        marker="o",
        linestyle="none",
        markerfacecolor="none",
        markeredgecolor="lime",
        markeredgewidth=2,
        markersize=10,
        label="True daughter",
    ),
    Line2D(
        [],
        [],
        marker="x",
        linestyle="none",
        color="red",
        markeredgewidth=2,
        markersize=10,
        label="Nearest non-daughter",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="outside upper right",
)

fig.colorbar(
    image,
    ax=axes,
    label="Fluorescence intensity",
    shrink=0.8,
)

plt.show()

### Microscopy Interpretation of the Distance Failure

All three next-frame candidates overlap plausible fluorescent structures in the microscopy images. This means the nearest non-daughter cannot be ruled out based on brightness or distance from the division parent alone.

The second true daughter is farther from the parent mainly because of its movement in the Z direction. It moves five Z voxels, equivalent to 8.125 µm, while the non-daughter moves only three Z voxels. This additional depth movement increases the true daughter’s total 3D distance, which is why the non-daughter ends up closer to the parent.

However, looking across time gives us another clue. At `t = 36`, a distinct cell appears above and to the right of the division parent. At `t = 37`, that cell appears to continue toward the red non-daughter marker. If this is the same cell, then the red candidate is close to the dividing parent simply because another trajectory passes nearby.

The orthogonal views also confirm that the candidate arrangement is genuinely three-dimensional. Because maximum-intensity projections combine structures from different depths into one image, the images alone are not enough to prove which links are correct.

We therefore examine the supplied tracking graph next to determine whether the red candidate is actually connected to the neighboring trajectory. This can show whether trajectory history provides information that distance from the division parent alone misses.

### Examine the Existing Trajectories of the Three Candidates

We next examine the supplied incoming edge for each of the three next-frame candidates. The goal is to test the visual hypothesis from the microscopy images: whether the non-daughter is continuing an existing neighboring trajectory while the two true daughters both originate from the division parent.

This analysis uses the supplied ground-truth tracking graph for EDA, so these incoming links would not be known to a prediction model in advance. However, they show what information a tracking method could potentially recover by considering multiple trajectories together instead of making each link independently.

A global tracking algorithm could estimate these connections jointly while enforcing the rule that each target node can have at most one incoming edge. If the non-daughter is already the best continuation of another nearby cell, assigning it to that trajectory would prevent it from also being incorrectly selected as a daughter of the dividing parent.

In [ ]:
candidate_role_lookup = (
    event_locations.loc[
        ~event_locations["role"].eq("Parent"),
        [
            "role",
            "node_id",
        ],
    ]
    .rename(
        columns={
            "node_id": "target_id",
        }
    )
)

candidate_incoming_edges = (
    edge_summary.loc[
        edge_summary["sample_id"].eq(
            difficult_event["sample_id"]
        )
        & edge_summary["target_id"].isin(
            candidate_role_lookup["target_id"]
        ),
        [
            "sample_id",
            "edge_type",
            "source_id",
            "target_id",
            "source_t",
            "target_t",
            "source_z",
            "source_y",
            "source_x",
            "target_z",
            "target_y",
            "target_x",
            "distance_um",
        ],
    ]
    .merge(
        candidate_role_lookup,
        on="target_id",
        how="left",
        validate="one_to_one",
    )
    [
        [
            "role",
            "edge_type",
            "source_id",
            "target_id",
            "source_t",
            "target_t",
            "source_z",
            "source_y",
            "source_x",
            "target_z",
            "target_y",
            "target_x",
            "distance_um",
        ]
    ]
    .sort_values("role")
)

display(candidate_incoming_edges.round(3))

### Existing Trajectory Resolves the Ambiguity

The supplied graph confirms what we saw in the microscopy images. The nearest non-daughter is actually connected to node `37000836` through a continuation edge with a displacement of only 4.909 µm. The two true daughters are both connected to the division parent `37000853`.

This explains why distance alone fails here. The red non-daughter is close to the division parent because two different cell trajectories happen to pass close to each other, not because it is actually part of the division. If we looked at the division by itself, we would choose the wrong second daughter. Once the surrounding trajectories are considered, the ambiguity becomes much clearer because the red candidate already fits the movement of the neighboring cell.

**The practical takeaway is that division links should not be decided completely independently.** A tracking model could also consider where nearby cells came from, whether a candidate already fits another trajectory, and how consistent each possible link is with previous movement. This would give the model more information in the rare cases where distance from the division parent points to the wrong candidate.

### Division Analysis Summary

The supplied graphs contain 151 division events: 26 in `44b6` and 125 in `6bba`. Every division has exactly two different daughters, and both daughters appear one frame after the parent.

The overall division geometry is fairly consistent across the two groups. Daughters usually move away from the parent in different directions, with median opening angles of about 135°–140°. Median daughter separation is larger in `6bba` at 11.469 µm compared with 8.983 µm in `44b6`. Parent midpoint offsets and typical distance asymmetry are similar between the groups.

Distance alone already provides a strong baseline for finding the correct daughter pair. In 145 events, both true daughters are closer to the parent than every non-daughter candidate. Another five divisions contain only the two daughters, meaning there is no incorrect candidate to distinguish between. Out of the 146 divisions where distance can actually be tested against at least one non-daughter, only **one** would be linked incorrectly by simply choosing the two nearest candidates.

Division-pair margins are generally larger in `44b6`, but this needs to be interpreted carefully because `44b6` also has much sparser supplied annotations and therefore fewer competing candidates. All three supplied test-movie divisions occur in `6bba_05db0fb1`. Their margins fall within the typical `6bba` range, and distance correctly selects both daughters in all three cases.

The single distance-based failure is especially useful because it shows exactly where distance stops being enough. The nearest non-daughter is actually the continuation of another nearby cell trajectory. Looking at the division by itself gives the wrong second daughter, but considering the surrounding trajectories resolves the ambiguity. This shows why tracking nearby cells together can matter even when distance works extremely well overall.

**Modelling implications:**

- Score the two daughter candidates together instead of treating each daughter edge as a completely separate prediction.
- Use parent-to-daughter distance as a strong starting feature since it correctly resolves almost every assessable supplied division.
- Include division geometry such as daughter separation, opening angle, midpoint offset, and distance asymmetry.
- Use previous movement and competition from nearby source cells when distance margins are small or negative.
- Make continuation and division decisions together so that a candidate that clearly continues another trajectory is not also incorrectly assigned as a division daughter.

Overall, distance can handle most of the supplied division events, so the goal is not to replace it with a much more complicated method. The additional features are most useful for the small number of ambiguous cases where nearby trajectories overlap and distance alone gives the wrong answer.

# EDA Conclusions and Modelling Implications

## Data and Image Characteristics

- All 199 movies have the same dimensions: 100 timepoints, 64 Z slices, and 256 × 256 pixels per slice.
- The voxel sizes are not equal in every direction. Z spacing is 1.625 µm compared with 0.40625 µm in X and Y, so calculate spatial distances in physical units rather than treating each voxel as the same distance in every direction.
- Image intensity varies considerably between movies, between `44b6` and `6bba`, and sometimes across time within the same movie.
- Because of this variation, robust normalisation at the movie or timepoint level will probably be needed before using image intensity as a model feature.
- Tissue position and the amount of tissue visible in the field of view can also change over time. Changes in low-intensity values therefore do not automatically mean that fluorescence itself changed.
- The supplied annotations are sparse and uneven. `6bba` generally contains many more annotated cells and candidate links than `44b6`, which needs to be considered when comparing the two groups.

## Supplied Graph Structure

The supplied tracking graphs are highly structured. Every graph is a directed acyclic forest, with no self-loops, isolated nodes, merge nodes, or backward edges.

Every non-root node has exactly one incoming edge, while each node has at most two outgoing edges. All supplied edges also connect consecutive frames.

Individual components are either simple trajectories or contain exactly one division. None of the supplied components contain multiple successive divisions.

These patterns give us useful rules for constructing and checking predicted graphs. For example, predicted tracks should not contain cycles or merges, and a target cell should not receive links from multiple parent cells.

One important limitation is that the supplied annotations contain no temporal gaps. This does **not** mean predicted tracks will always be complete. A detection model can miss a cell for one or more frames, so gap repair and synthetic-gap testing may still be useful even though gaps do not appear in the supplied ground-truth graphs.

## Movement and Direction

Most continuation movements are small, with median displacements of about 1.7–1.8 µm. Division movements are noticeably larger, with median parent-to-daughter distances of about 4.6 µm in `44b6` and 5.9 µm in `6bba`.

A small number of extremely large movements also occur. Some of these appear to be caused by global image or tissue movement rather than a single cell suddenly moving a large distance. This means raw displacement should not always be interpreted as individual cell motion.

Direction also contains useful information. `6bba` continuation trajectories generally have smaller turning angles and therefore more directional persistence than `44b6`, although the movie-level distributions still overlap substantially.

When very short movements are removed, turning angles decrease in both groups. This shows that direction becomes less reliable when a cell barely moves because even a small coordinate difference can produce a large change in estimated direction.

The practical implication is that distance and direction should not be used completely independently. Motion history and estimated global frame movement could help distinguish real cell movement from movement of the entire image or tissue.

## Spatial Competition and Continuation Linking

`6bba` generally contains more possible next-frame candidates for each source cell because its supplied annotations are denser.

Even with this additional competition, distance performs extremely well. The supplied continuation target is uniquely nearest in almost every link. Out of 128,581 continuation links, only 172 have either an incorrect or tied nearest candidate.

These difficult cases are also not spread evenly across the dataset. They cluster in particular movies, including some movies where unusual global movement was already identified. This suggests that at least some distance failures may be connected to unusual frame-to-frame motion rather than crowding alone.

All continuation links in the four supplied test movies are correctly resolved by nearest distance among the supplied candidates.

This makes nearest-neighbor linking a very strong baseline. However, these numbers are still optimistic because the candidate sets contain only the sparse supplied annotations. A real detection model will likely produce additional cells, missed cells, false positives, and localization errors, making the linking problem harder.

## Division Behaviour

The supplied graphs contain 151 division events, and every division produces exactly two daughters in the following frame.

The daughter pairs also have a recognizable spatial pattern. They usually move away from the parent in substantially different directions, with median opening angles of about 135°–140°. Daughter separation is generally larger in `6bba`, while midpoint offsets and typical distance asymmetry are fairly similar between the groups.

Distance again provides a very strong baseline. Among the 146 divisions containing at least one non-daughter alternative, selecting the two nearest candidates identifies both true daughters in 145 cases.

The one failure is especially useful because it shows why global tracking matters. The incorrect candidate is closer to the division parent than one of the true daughters, but it is actually the continuation of another nearby cell trajectory. Looking at the division alone gives the wrong answer, while considering the surrounding trajectories resolves the ambiguity.

All three supplied test-movie divisions are correctly resolved by distance and have margins that are reasonably representative of the `6bba` division distribution.

Division linking should therefore start with distance but should not depend on it alone. Candidate daughters can be evaluated together using their separation, opening angle, midpoint position, distance asymmetry, motion history, image appearance, and competition from nearby trajectories.

## Test-Movie Considerations

The four test movies do not represent exactly the same type of tracking problem. They differ in annotation density, image intensity, movement, directional persistence, and spatial competition.

Some test movies also fall near unusual parts of their group distributions for particular measurements. Because of this, validation should be examined separately by movie and dataset group instead of relying only on one overall average score. A model that performs well on average could still struggle with one specific type of movie.

## Initial Modelling Implications

Based on the EDA, a practical tracking system can be divided into four connected stages:

1. **Cell detection**
   - robustly normalise fluorescence intensity;
   - process spatial information in physical 3D units;
   - account for the much larger Z spacing.

2. **Continuation-link scoring**
   - use physical source-to-candidate distance as the main baseline;
   - add previous movement, turning angle, image similarity, and estimated global frame movement;
   - give less weight to direction when previous displacement is extremely small.

3. **Division scoring**
   - score pairs of daughter candidates together rather than treating each daughter edge independently;
   - include parent-to-daughter distance, daughter separation, opening angle, midpoint offset, and distance asymmetry;
   - use surrounding trajectories when distance produces a small or negative margin.

4. **Global graph construction**
   - allow each node to have at most one incoming edge and two outgoing edges;
   - prevent merges, backward links, and cycles;
   - resolve continuation and division assignments together so competing trajectories cannot claim the same target;
   - allow gap repair when a cell is temporarily missed by the detector.

Overall, the EDA shows that **physical distance provides an extremely strong starting point for both continuation and division linking**, so there is no reason to make the first tracking model unnecessarily complicated. The more advanced features are most useful for the relatively small number of ambiguous cases where distance becomes unreliable.

At the same time, do not treat performance on the supplied annotations as expected competition performance. The supplied candidate sets are sparse, while a complete tracking pipeline will also have to deal with detection errors, false positives, missed cells, localization errors, and divisions. The final model therefore needs to be evaluated using the actual competition metric, where detection quality, linking accuracy, division handling, and node-count errors all affect the final result.

In [ ]:
from pathlib import Path

base = Path("/kaggle/input")

for p in base.rglob("*.zarr"):
    print(p)

In [ ]:
import zarr
from pathlib import Path

zarr_paths = list(Path("/kaggle/input").rglob("*.zarr"))

print("Found:", len(zarr_paths), "zarr files")

for p in zarr_paths[:10]:
    print("\n", p)
    z = zarr.open(str(p), mode="r")
    print(z.tree())

In [ ]:
from pathlib import Path

COMP_DIR = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
TEST_DIR = COMP_DIR / "test"

test_paths = sorted(TEST_DIR.glob("*.zarr"))

print("Test movies found:", len(test_paths))

for p in test_paths:
    print(p.name)

In [ ]:
import numpy as np
import pandas as pd
import zarr

from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment

In [ ]:
# Physical voxel spacing in micrometers
SCALE = np.array([1.625, 0.40625, 0.40625])  # z, y, x

def detect_cells(volume, max_cells=250):
    """
    Simple 3D local-maximum detector.
    Returns integer (z, y, x) coordinates.
    """

    vol = volume.astype(np.float32)

    # robust normalization
    lo, hi = np.percentile(vol, [50, 99.8])

    if hi <= lo:
        return np.empty((0, 3), dtype=int)

    norm = np.clip((vol - lo) / (hi - lo), 0, 1)

    # light smoothing
    smooth = gaussian_filter(norm, sigma=(0.7, 1.0, 1.0))

    # local maxima
    local_max = smooth == maximum_filter(
        smooth,
        size=(3, 5, 5)
    )

    threshold = np.percentile(smooth, 99.7)

    coords = np.argwhere(local_max & (smooth >= threshold))

    if len(coords) == 0:
        return np.empty((0, 3), dtype=int)

    scores = smooth[
        coords[:, 0],
        coords[:, 1],
        coords[:, 2]
    ]

    order = np.argsort(scores)[::-1]

    coords = coords[order[:max_cells]]

    return coords.astype(int)

In [ ]:
def physical_distance_matrix(a, b):
    """
    a: (N,3) z,y,x
    b: (M,3) z,y,x
    """

    if len(a) == 0 or len(b) == 0:
        return np.empty((len(a), len(b)))

    diff = (
        a[:, None, :].astype(float)
        - b[None, :, :].astype(float)
    )

    diff *= SCALE

    return np.sqrt((diff ** 2).sum(axis=2))


def link_frames(prev_coords, next_coords, max_distance_um=10.0):
    """
    Hungarian assignment between consecutive frames.
    """

    if len(prev_coords) == 0 or len(next_coords) == 0:
        return []

    dist = physical_distance_matrix(prev_coords, next_coords)

    rows, cols = linear_sum_assignment(dist)

    links = []

    for r, c in zip(rows, cols):
        if dist[r, c] <= max_distance_um:
            links.append((r, c))

    return links

In [ ]:
def track_movie(zarr_path):
    root = zarr.open(str(zarr_path), mode="r")
    movie = root["0"]

    dataset = zarr_path.stem

    node_rows = []
    edge_rows = []

    next_node_id = 1
    previous_nodes = None
    previous_coords = None

    for t in range(movie.shape[0]):

        volume = np.asarray(movie[t])

        coords = detect_cells(volume)

        current_nodes = []

        for z, y, x in coords:

            node_id = next_node_id
            next_node_id += 1

            current_nodes.append(node_id)

            node_rows.append({
                "dataset": dataset,
                "row_type": "node",
                "node_id": node_id,
                "t": t,
                "z": int(z),
                "y": int(y),
                "x": int(x),
                "source_id": -1,
                "target_id": -1
            })

        if previous_nodes is not None:

            links = link_frames(
                previous_coords,
                coords,
                max_distance_um=10.0
            )

            for prev_idx, curr_idx in links:

                edge_rows.append({
                    "dataset": dataset,
                    "row_type": "edge",
                    "node_id": -1,
                    "t": -1,
                    "z": -1,
                    "y": -1,
                    "x": -1,
                    "source_id": previous_nodes[prev_idx],
                    "target_id": current_nodes[curr_idx]
                })

        previous_nodes = current_nodes
        previous_coords = coords

    return node_rows, edge_rows

In [ ]:
all_rows = []

for path in test_paths:

    print("Processing:", path.name)

    nodes, edges = track_movie(path)

    all_rows.extend(nodes)
    all_rows.extend(edges)

    print(
        "  nodes:", len(nodes),
        "edges:", len(edges)
    )

submission = pd.DataFrame(all_rows)

submission.insert(
    0,
    "id",
    np.arange(len(submission))
)

submission = submission[
    [
        "id",
        "dataset",
        "row_type",
        "node_id",
        "t",
        "z",
        "y",
        "x",
        "source_id",
        "target_id"
    ]
]

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print()
print("Saved submission.csv")
print("Rows:", len(submission))

display(submission.head())

In [ ]:
from pathlib import Path

submission_path = Path(
    "/kaggle/working/submission.csv"
)

print("Exists:", submission_path.exists())
print("Size:", submission_path.stat().st_size if submission_path.exists() else 0)

print()
print(submission.groupby(
    ["dataset", "row_type"]
).size())

In [ ]:
summary = (
    submission[submission["row_type"] == "node"]
    .groupby("dataset")
    .agg(
        total_nodes=("node_id", "count"),
        frames=("t", "nunique")
    )
)

summary["nodes_per_frame"] = (
    summary["total_nodes"] / summary["frames"]
)

display(summary)

In [ ]:
for dataset in submission["dataset"].unique():
    df = submission[
        (submission["dataset"] == dataset) &
        (submission["row_type"] == "node")
    ]

    counts = df.groupby("t").size()

    print("\n", dataset)
    print("min/frame:", counts.min())
    print("median/frame:", counts.median())
    print("max/frame:", counts.max())

In [ ]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

In [ ]:
import os

print(os.path.exists("/kaggle/working/submission.csv"))
print(os.listdir("/kaggle/working"))